In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"

EDITIONS_PATH = Path("data/editions.csv")
OUT_PATH = Path("data/edition_qwen3_embedding.feather")

MAX_SEQ_LENGTH = 512
BATCH_SIZE = 64
NORMALIZE_EMB = True  # recommended if you will use cosine similarity

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
print("model:", MODEL_NAME)

In [ ]:
editions = pd.read_csv(EDITIONS_PATH)
need_cols = {"edition_id", "title", "description"}
missing = need_cols - set(editions.columns)
if missing:
    raise ValueError(f"Missing columns in {EDITIONS_PATH}: {sorted(missing)}")

editions["title"] = editions["title"].fillna("").astype(str)
editions["description"] = editions["description"].fillna("").astype(str)

texts = (editions["title"] + "\n\n" + editions["description"]).str.strip()
texts = texts.mask(texts.eq(""), "<empty>")

edition_ids = editions["edition_id"].astype(np.int64).to_numpy()

print("n_editions:", len(editions))
editions.head(3)[["edition_id", "title", "description"]]

In [ ]:
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
try:
    model.max_seq_length = int(MAX_SEQ_LENGTH)
except Exception as e:
    print("Could not set max_seq_length:", repr(e))

model

In [ ]:
emb = model.encode(
    texts.tolist(),
    batch_size=int(BATCH_SIZE),
    normalize_embeddings=bool(NORMALIZE_EMB),
    convert_to_numpy=True,
    show_progress_bar=True,
)
emb = np.asarray(emb, dtype=np.float32)

print("embeddings:", emb.shape, emb.dtype)
print("first norm:", float(np.linalg.norm(emb[0])))

In [ ]:
emb_df = pd.DataFrame(
    {
        "edition_id": edition_ids,
        "embedding": [row.tolist() for row in emb],
    }
).sort_values("edition_id").reset_index(drop=True)

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
emb_df.to_feather(OUT_PATH, index=False)

print("saved:", OUT_PATH)
print("rows:", len(emb_df))
print("dim:", len(emb_df.loc[0, "embedding"]))
emb_df.head(3)

In [ ]:
# Удалит все пользовательские переменные без запроса подтверждения
%reset -f

In [ ]:
import os
from dataclasses import dataclass, field
from typing import Dict, Any, List, Tuple, Optional

@dataclass
class PipelineConfig:
    """
    Абсолютно полный конфигуратор пайплайна.
    """

    # =========================================================================
    # 1. DATA_PATHS & BASE CONSTANTS
    # =========================================================================
    DATA_PATHS: Dict[str, str] = field(default_factory=lambda: {
        "interactions_path": "data/interactions.csv",
        "users_path": "data/users.csv",
        "editions_path": "data/editions.csv",
        "authors_path": "data/authors.csv",
        "genres_path": "data/genres.csv",
        "book_genres_path": "data/book_genres.csv",
        "candidates_path": "submit/candidates.csv",
        "targets_path": "submit/targets.csv",
    })
    
    TOP_N_GENRES: int = 10
    N_TEXT_COMPONENTS: int = 32
    ALS_FACTORS: int = 20

    # =========================================================================
    # 2. QWEN TEXT FEATURES
    # =========================================================================
    USE_QWEN_TEXT_FEATURES: bool = True
    QWEN_EMBS_FEATHER_PATH: str = "data/edition_qwen3_embedding.feather", # Путь к внешним эмбеддингам
    QWEN_PCA_COMPONENTS: int = 16
    QWEN_PCA_BATCH_SIZE: int = 8192
    QWEN_DOT_BATCH_SIZE: int = 8192

    # =========================================================================
    # 3. LIGHTFM FEATURES
    # =========================================================================
    USE_LIGHTFM_FEATURES: bool = False
    LIGHTFM_LEVELS: Tuple[str, ...] = ("edition", "book", "author", "publisher")
    LIGHTFM_DIMS: int = 64
    LIGHTFM_LEARNING_RATE: float = 0.05
    LIGHTFM_LOSS: str = "warp"
    LIGHTFM_EPOCHS: int = 20
    LIGHTFM_NUM_THREADS: int = 4
    LIGHTFM_RANDOM_STATE: int = 42
    LIGHTFM_GLOBAL_Z_SAMPLE_SIZE: int = 10000
    LIGHTFM_VERBOSE_FIT: bool = True
    LIGHTFM_PRINT_LOGS: bool = True

    LIGHTFM_USE_ITEM_STATS: bool = True
    LIGHTFM_USE_CONTENT_SVD: bool = True
    LIGHTFM_CONTENT_L2_NORMALIZE: bool = True

    LFM_FEATS_INCLUDE_SCORE: bool = True
    LFM_FEATS_INCLUDE_BIASES: bool = True
    LFM_FEATS_INCLUDE_NORMS: bool = True
    LFM_FEATS_INCLUDE_GLOBAL_Z: bool = True

    LFM_FEATS_INCLUDE_EMBEDDINGS: bool = True
    LFM_EMB_LEVELS: Tuple[str, ...] = ("edition",)
    LFM_EMB_COMPONENTS: int = 16
    LFM_EMB_INCLUDE_USER: bool = False
    LFM_EMB_INCLUDE_ITEM: bool = False
    LFM_EMB_INCLUDE_UI_PROD: bool = True

    # =========================================================================
    # 4. IMPLICIT ALS FEATURES
    # =========================================================================
    USE_IMPLICIT_ALS_FEATURES: bool = False
    IMPLICIT_ALS_LEVELS: Tuple[str, ...] = ("edition", "book", "author", "publisher")
    IMPLICIT_ALS_FACTORS_BY_LEVEL: Dict[str, int] = field(default_factory=lambda: {
        "edition": 64,
        "book": 64,
        "author": 64,
        "publisher": 64,
    })
    IMPLICIT_ALS_REG: float = 0.01
    IMPLICIT_ALS_ITERATIONS: int = 25
    IMPLICIT_ALS_RANDOM_STATE: int = 42
    IMPLICIT_ALS_GLOBAL_Z_SAMPLE_SIZE: int = 10000
    IMPLICIT_ALS_GLOBAL_Z_EPS: float = 1e-6

    ALS_FEATS_INCLUDE_SCORE: bool = True
    ALS_FEATS_INCLUDE_NORMS: bool = True
    ALS_FEATS_INCLUDE_GLOBAL_Z: bool = True

    ALS_FEATS_INCLUDE_EMBEDDINGS: bool = True
    ALS_EMB_LEVELS: Tuple[str, ...] = ("edition",)
    ALS_EMB_COMPONENTS: int = 16
    ALS_EMB_INCLUDE_USER: bool = False
    ALS_EMB_INCLUDE_ITEM: bool = False
    ALS_EMB_INCLUDE_UI_PROD: bool = True

    # =========================================================================
    # 5. ITEMKNN FEATURES
    # =========================================================================
    USE_ITEMKNN_FEATURES: bool = False
    ITEMKNN_LEVELS: Tuple[str, ...] = ("edition", "book", "author", "publisher")
    ITEMKNN_K: int = 200
    ITEMKNN_SCORE_NORMALIZE_BY_SUMW: bool = True
    ITEMKNN_EPS: float = 1e-6

    KNN_FEATS_INCLUDE_SCORE: bool = True
    KNN_FEATS_INCLUDE_HITS: bool = True
    KNN_FEATS_INCLUDE_MAX_SIM: bool = True
    KNN_FEATS_INCLUDE_GLOBAL_Z: bool = True

    # =========================================================================
    # 6. BM25-ITEMKNN FEATURES
    # =========================================================================
    USE_BM25KNN_FEATURES: bool = False
    BM25KNN_LEVELS: Tuple[str, ...] = ("edition", "book", "author", "publisher")
    BM25KNN_K: int = 350
    BM25KNN_K1: float = 1.2
    BM25KNN_B: float = 0.75
    BM25KNN_SCORE_NORMALIZE_BY_SUMW: bool = True
    BM25KNN_EPS: float = 1e-6

    BM25_FEATS_INCLUDE_SCORE: bool = True
    BM25_FEATS_INCLUDE_HITS: bool = True
    BM25_FEATS_INCLUDE_MAX_SIM: bool = True
    BM25_FEATS_INCLUDE_GLOBAL_Z: bool = True

    # =========================================================================
    # 7. ITEM2VEC FEATURES
    # =========================================================================
    USE_ITEM2VEC_FEATURES: bool = False
    ITEM2VEC_LEVELS: Tuple[str, ...] = ("edition",)
    ITEM2VEC_WINDOW: int = 5
    ITEM2VEC_NEG_K: float = 5.0
    ITEM2VEC_N_COMPONENTS: int = 64
    ITEM2VEC_MIN_ITEM_COUNT: int = 2
    ITEM2VEC_DEDUP_CONSECUTIVE: bool = True
    ITEM2VEC_CACHE_DIR: str = "cache/item2vec"
    ITEM2VEC_USE_CACHE: bool = True

    I2V_FEATS_INCLUDE_SCORE: bool = True
    I2V_FEATS_INCLUDE_GLOBAL_Z: bool = True
    ITEM2VEC_GLOBAL_Z_SAMPLE_SIZE: int = 10000
    ITEM2VEC_GLOBAL_Z_EPS: float = 1e-6

    # =========================================================================
    # 8. FINAL ENSEMBLING / BLEND
    # =========================================================================
    ENABLE_BLEND: bool = True
    USE_LGBM_BLEND_MODEL: bool = True
    USE_XGB_BLEND_MODEL: bool = True

    ENABLE_VALIDATION: bool = False
    ENABLE_PER_USER_TRAINING_WEIGHTS: bool = True
    PER_USER_TRAINING_WEIGHT_ALPHA: float = 1.0
    PER_USER_TRAINING_WEIGHT_CLIP_MIN: Optional[float] = None
    PER_USER_TRAINING_WEIGHT_CLIP_MAX: Optional[float] = None

    BLEND_NDCG_K: int = 20
    BLEND_PER_USER_ZSCORE: bool = True
    BLEND_USE_MANUAL_WEIGHTS: bool = False
    BLEND_MANUAL_WEIGHTS: Dict[str, float] = field(default_factory=lambda: {
        "catboost": 1.0, "lightgbm": 1.0, "xgboost": 1.0
    })

    BLEND_TUNE_WEIGHTS: bool = True
    BLEND_TUNE_METHOD: str = "grid"
    BLEND_TUNE_GRID_STEP: float = 0.1
    BLEND_TUNE_RANDOM_SAMPLES: int = 0
    BLEND_TUNE_SEED: int = 42

    EXPORT_MODEL_SUBMISSIONS: bool = True
    SUBMISSION_CATBOOST_PATH: str = "submission_catboost.csv"
    SUBMISSION_LIGHTGBM_PATH: str = "submission_lightgbm.csv"
    SUBMISSION_XGBOOST_PATH: str = "submission_xgboost.csv"
    SUBMISSION_BLENDED_PATH: str = "submission_blended.csv"

    SAVE_MODEL_LOGS: bool = True
    MODEL_LOG_DIR: str = "cache/model_logs"
    MODEL_LOG_RUN_ID: str = "latest"
    LGBM_LOG_EVAL_PERIOD: int = 200
    XGB_LOG_EVAL_PERIOD: int = 200

    # =========================================================================
    # 9. MODEL CONFIGS (CatBoost, LightGBM, XGBoost)
    # =========================================================================
    CATBOOST_WARM_TUNE_CONFIG: Dict[str, object] = field(default_factory=lambda: {
        "iterations": 7500, "learning_rate": 0.01, "depth": 10,
        "loss_function": "YetiRank", "eval_metric": "NDCG:top=20",
        "od_type": "Iter", "od_wait": 100, "use_best_model": True,
        "random_seed": 42, "verbose": 100
    })
    CATBOOST_WARM_FINAL_CONFIG: Dict[str, object] = field(default_factory=lambda: {
        "iterations": 7500, "learning_rate": 0.015, "depth": 10,
        "loss_function": "YetiRank", "random_seed": 42, "verbose": 100
    })

    LGBM_BLEND_N_ESTIMATORS: int = 8000
    LGBM_BLEND_LEARNING_RATE: float = 0.05
    LGBM_BLEND_NUM_LEAVES: int = 127
    LGBM_BLEND_MIN_CHILD_SAMPLES: int = 20
    LGBM_BLEND_SUBSAMPLE: float = 0.8
    LGBM_BLEND_COLSAMPLE: float = 0.8
    LGBM_BLEND_REG_LAMBDA: float = 1.0
    LGBM_BLEND_EARLY_STOP_ROUNDS: int = 100
    LGBM_BLEND_N_JOBS: int = 8

    LGBM_BLEND_TUNE_CONFIG: Dict[str, object] = field(default_factory=lambda: {
        "objective": "lambdarank", "n_estimators": 8000,
        "learning_rate": 0.05, "num_leaves": 127,
        "min_child_samples": 20, "subsample": 0.8,
        "colsample_bytree": 0.8, "reg_lambda": 1.0,
        "random_state": 42, "n_jobs": 8, "verbosity": -1,
        "__early_stopping_rounds": 100, "__eval_metric": "ndcg", "__eval_at": [20]
    })
    LGBM_BLEND_FINAL_CONFIG: Dict[str, object] = field(default_factory=lambda: {
        "objective": "lambdarank", "n_estimators": 2000,
        "learning_rate": 0.05, "num_leaves": 127,
        "min_child_samples": 20, "subsample": 0.8,
        "colsample_bytree": 0.8, "reg_lambda": 1.0,
        "random_state": 42, "n_jobs": 8, "verbosity": -1
    })

    XGB_BLEND_N_ESTIMATORS: int = 8000
    XGB_BLEND_LEARNING_RATE: float = 0.05
    XGB_BLEND_MAX_DEPTH: int = 8
    XGB_BLEND_SUBSAMPLE: float = 0.8
    XGB_BLEND_COLSAMPLE: float = 0.8
    XGB_BLEND_REG_LAMBDA: float = 1.0
    XGB_BLEND_REG_ALPHA: float = 0.0
    XGB_BLEND_MIN_CHILD_WEIGHT: float = 1.0
    XGB_BLEND_EARLY_STOP_ROUNDS: int = 200
    XGB_BLEND_N_JOBS: int = 8
    XGB_BLEND_TREE_METHOD: str = "hist"

    XGB_BLEND_TUNE_CONFIG: Dict[str, object] = field(default_factory=lambda: {
        "objective": "rank:ndcg", "n_estimators": 8000,
        "learning_rate": 0.05, "max_depth": 8,
        "subsample": 0.8, "colsample_bytree": 0.8,
        "reg_lambda": 1.0, "reg_alpha": 0.0,
        "min_child_weight": 1.0, "random_state": 42,
        "n_jobs": 8, "tree_method": "hist",
        "__early_stopping_rounds": 200, "__eval_metric": "ndcg@20"
    })
    XGB_BLEND_FINAL_CONFIG: Dict[str, object] = field(default_factory=lambda: {
        "objective": "rank:ndcg", "n_estimators": 2000,
        "learning_rate": 0.05, "max_depth": 8,
        "subsample": 0.8, "colsample_bytree": 0.8,
        "reg_lambda": 1.0, "reg_alpha": 0.0,
        "min_child_weight": 1.0, "random_state": 42,
        "n_jobs": 8, "tree_method": "hist"
    })

    # =========================================================================
    # 10. SYSTEM & INTERACTION/LABEL SETTINGS
    # =========================================================================
    AUTO_DISABLE_MISSING_OPTIONAL_DEPS: bool = True
    STRICT_OPTIONAL_DEPS: bool = False
    CATBOOST_TASK_TYPE: str = "AUTO"
    SAVE_MODELS: bool = True
    MODELS_DIR: str = "models"

    USE_WISHLISTS: bool = True
    READ_LABEL: int = 3
    WISHLIST_LABEL: int = 1

    ENABLE_RATING_AWARE_INTERACTION_WEIGHTS: bool = True
    RATING_AWARE_APPLY_TO_CF: bool = True
    RATING_AWARE_APPLY_TO_TEXT_USER_EMB: bool = True
    RATING_AWARE_APPLY_TO_QWEN_USER_EMB: bool = True
    RATING_AWARE_MAX_RATING: float = 10.0
    RATING_AWARE_GAMMA: float = 1.0
    RATING_AWARE_LOW_RATING_THRESHOLD: float = 3.0
    RATING_AWARE_LOW_RATING_ACTION: str = "downweight"
    RATING_AWARE_LOW_RATING_MULT: float = 0.2

    ALS_READ_WEIGHT: float = 3.0
    ALS_WISHLIST_WEIGHT: float = 1.0

    DROP_ZERO_IMPORTANCE_COLS: List[str] = field(default_factory=lambda: [
        "log1p_book_popularity_30d", "log1p_publisher_popularity_30d",
        "log1p_book_wishlist_count_30d", "edition_popularity_30d",
        "same_language_as_user_top", "seen_book_read_cnt",
        "seen_genre_read_cnt_recent", "same_primary_genre_as_recent_top",
        "seen_book_read", "seen_edition_read_cnt", "seen_edition_read_last_days",
        "user_top_language_id", "seen_edition_read", "seen_edition_wish",
        "seen_edition_wish_cnt", "seen_edition_wish_last_days",
        "seen_book_wish", "seen_book_wish_cnt", "seen_book_wish_last_days", "age_ok"
    ])

    # =========================================================================
    # 11. FEATURE SELECTION
    # =========================================================================
    FEATURE_SELECTION_CONFIG: Dict[str, object] = field(default_factory=lambda: {
        "enabled": False,
        "num_features_to_select": 220,
        "steps": 1,
        "algorithm": "ByShapValues",
        "selection_params": {
            "iterations": 1000, "learning_rate": 0.05, "depth": 10,
            "loss_function": "YetiRank", "task_type": "CPU",
            "random_seed": 42, "allow_writing_files": False,
        },
        "use_cache": True,
        "cache_dir": "cache/feature_selection",
        "drop_unselected_columns": True,
        "train_final_model": False,
        "verbose": False,
        "logging_level": "Silent",
    })

    # =========================================================================
    # 12. CANDIDATE RELATIVE FEATURES & SLATE NORMALIZATION
    # =========================================================================
    ENABLE_CANDIDATE_RELATIVE_FEATURES: bool = True
    CAND_REL_INCLUDE_POPULARITY: bool = True
    CAND_REL_ENABLE_ENTITY_RANKS: bool = True
    CAND_REL_INCLUDE_ENTITY_POPULARITY: bool = True
    CAND_REL_INCLUDE_ENTITY_POPULARITY_30D: bool = False
    CAND_REL_ENTITY_PREFIXES: Tuple[str, ...] = ("book", "author", "publisher", "genre")
    CAND_REL_INCLUDE_TEXT_SVD: bool = True
    CAND_REL_INCLUDE_QWEN: bool = True
    
    CAND_REL_INCLUDE_READ_SHARE: bool = True
    CAND_REL_INCLUDE_WISHLIST_SHARE: bool = False
    CAND_REL_INCLUDE_ENTITY_READ_SHARE: bool = True
    CAND_REL_INCLUDE_ENTITY_WISHLIST_SHARE: bool = False
    CAND_REL_INCLUDE_ALS_SCORE_RANKS: bool = False
    CAND_REL_INCLUDE_LFM_SCORE_RANKS: bool = False
    CAND_REL_INCLUDE_KNN_SCORE_RANKS: bool = False

    ENABLE_GLOBAL_CANDIDATE_PRIOR_FEATURES: bool = False
    CANDIDATE_PRIOR_COUNT_UNIQUE_USERS: bool = True
    CANDIDATE_PRIOR_INCLUDE_LOG1P: bool = True

    ENABLE_CANDIDATE_SLATE_NORM_FEATURES: bool = True
    CAND_SLATE_NORM_INCLUDE_Z: bool = True
    CAND_SLATE_NORM_INCLUDE_DIFF: bool = False
    CAND_SLATE_NORM_INCLUDE_ABS_DIFF: bool = True
    CAND_SLATE_NORM_EPS: float = 1e-6
    CAND_SLATE_NORM_Z_CLIP: float = 8.0

    CAND_SLATE_NORM_INCLUDE_POPULARITY: bool = True
    CAND_SLATE_NORM_POPULARITY_COL: str = "log1p_edition_popularity"
    CAND_SLATE_NORM_INCLUDE_POPULARITY_30D: bool = True
    CAND_SLATE_NORM_POPULARITY_30D_COL: str = "log1p_edition_popularity_30d"
    CAND_SLATE_NORM_INCLUDE_RECENCY: bool = True
    CAND_SLATE_NORM_RECENCY_COL: str = "edition_days_since_last_event"
    CAND_SLATE_NORM_INCLUDE_TEXT_SIM: bool = True
    CAND_SLATE_NORM_TEXT_SIM_COL: str = "text_cosine_similarity"
    CAND_SLATE_NORM_INCLUDE_QWEN_SIM: bool = True
    CAND_SLATE_NORM_QWEN_SIM_COL: str = "qwen_cos_user_edition"
    CAND_SLATE_NORM_INCLUDE_BAYES_RATING: bool = True
    CAND_SLATE_NORM_BAYES_RATING_COL: str = "edition_bayes_avg_rating"

    CAND_SLATE_NORM_INCLUDE_ALS_SCORES: bool = True
    CAND_SLATE_NORM_ALS_LEVELS: Tuple[str, ...] = ("edition",)
    CAND_SLATE_NORM_INCLUDE_LFM_SCORES: bool = True
    CAND_SLATE_NORM_LFM_LEVELS: Tuple[str, ...] = ("edition",)
    CAND_SLATE_NORM_INCLUDE_KNN_SCORES: bool = True
    CAND_SLATE_NORM_KNN_LEVELS: Tuple[str, ...] = ("edition",)

    # =========================================================================
    # 13. NEGATIVE SAMPLING CONFIG
    # =========================================================================
    NEGATIVE_SAMPLING_CONFIG: Dict[str, object] = field(default_factory=lambda: {
        "cand_als_worst_dissimilar": 0,
        "cand_als_worst_similar": 0,
        "cand_als_best_similar": 0,
        "cand_random_unseen": 0,
        "cand_popular_unseen": 0,
        "cand_popular_30d_unseen": 0,
        "cand_genre_dissimilar": 0,
        "cand_genre_similar": 0,
        "cand_all_unseen": 0,
        "cand_all_nonpos": 200,
        "cand_qwen_dissimilar_books": 0,
        "cand_qwen_best_among_worst_books": 0,
        "als_hard_outside_candidates": 0,
        "als_popular_high_outside_candidates": 0,
        "als_hard_other_users_candidates": 0,
        "__cand_als_pool_top_n": 150,
        "__cand_qwen_dissimilar_top_n": 100,
        "__cand_exclude_cold_start": True,
        "__cand_popular_30d_lookback_days": 30,
        "__hard_outside_top_n": 800,
        "__hard_outside_oversample": 5,
        "__hard_outside_batch_users": 64,
        "__hard_outside_exclude_same_book": False,
        "__hard_outside_exclude_same_author": False,
        "__hard_outside_exclude_same_publisher": False,
        "__hard_outside_exclude_same_language": False,
        "__popals_global_top_n": 20000,
        "__popals_top_n": 800,
        "__popals_oversample": 5,
        "__popals_batch_users": 64,
        "__othercand_top_n": 800,
        "__othercand_oversample": 5,
        "__othercand_batch_users": 64,
        "__othercand_global_max_items": 0,
        "__als_sampling_fit_until_end_ts": True,
    })

    # =========================================================================
    # 14. TARGETS & MMR
    # =========================================================================
    TRAIN_TARGET_LAST_MONTH_ONLY: bool = True
    TRAIN_WINDOW_DAYS: int = 30
    GENRE_THRESHOLD: float = 0.05
    MMR_LAMBDA: float = 0.0



    # В конец PipelineConfig добавь:
    RATING_PRIOR_COUNT: float = 10.0
    POSTPROCESS_UNIQUE_BOOKS: bool = True
    POSTPROCESS_EXCLUDE_READ_BOOKS: bool = False
    POSTPROCESS_EXCLUDE_READ_BOOKS_ALLOW_READ_FALLBACK: bool = False

In [ ]:
import os
import pandas as pd
import numpy as np
from typing import List

class DataHandler:
    """
    Класс для загрузки и первичной обработки сырых данных соревновательного датасета.
    Основан на функциях _resolve_path и prepare_datasets из оригинального пайплайна.
    """
    
    def __init__(self, config):
        """
        Инициализируется только объектом конфигурации.
        Атрибуты-датафреймы будут заполнены после вызова load_data().
        """
        self.cfg = config
        
        # Заглушки для будущих датафреймов (сохраняем оригинальные нейминги)
        self.interactions = None
        self.users = None
        self.editions = None
        self.authors = None
        self.genres = None
        self.book_genres = None
        self.candidates = None
        self.targets = None

    def _resolve_path(self, p: str) -> str:
        """
        [Оригинальная логика _resolve_path из BLENDED_CHANCE.py]
        Try to resolve a relative path robustly:
        1) as given
        2) basename in CWD
        3) basename in /mnt/data (this notebook environment)
        4) `participants/` prefixes (this repo layout)
        """
        if os.path.exists(p):
            return p

        base = os.path.basename(p)
        candidates: List[str] = []

        # Same dir (basename)
        candidates.append(base)

        # Repo layout: participants/{data,submit}/...
        candidates.append(os.path.join("participants", "data", base))
        candidates.append(os.path.join("participants", "submit", base))

        # If the original path already contains a top-level folder, try to prefix it
        candidates.append(os.path.join("participants", p))

        # Common "data/..." and "submit/..." paths used in competition templates.
        if p.startswith("data" + os.sep) or p.startswith("data/"):
            candidates.append(os.path.join("participants", p))
        if p.startswith("submit" + os.sep) or p.startswith("submit/"):
            candidates.append(os.path.join("participants", p))

        # Notebook environments sometimes mount data in /mnt/data
        candidates.append(os.path.join("/mnt/data", base))

        for cand in candidates:
            if cand and os.path.exists(cand):
                return cand
                
        return p

    def load_data(self) -> None:
        """
        [Оригинальная логика prepare_datasets из BLENDED_CHANCE.py]
        Load CSV files and perform static joins (genres per edition).
        Вместо возврата словаря, сохраняет датафреймы как атрибуты класса (self.*).
        """
        print("[DataHandler] Loading raw datasets...")
        
        # Чтение файлов с использованием путей из конфигуратора
        self.interactions = pd.read_csv(self._resolve_path(self.cfg.DATA_PATHS["interactions_path"]))
        self.users = pd.read_csv(self._resolve_path(self.cfg.DATA_PATHS["users_path"]))
        self.editions = pd.read_csv(self._resolve_path(self.cfg.DATA_PATHS["editions_path"]))
        self.authors = pd.read_csv(self._resolve_path(self.cfg.DATA_PATHS["authors_path"]))
        self.genres = pd.read_csv(self._resolve_path(self.cfg.DATA_PATHS["genres_path"]))
        self.book_genres = pd.read_csv(self._resolve_path(self.cfg.DATA_PATHS["book_genres_path"]))
        self.candidates = pd.read_csv(self._resolve_path(self.cfg.DATA_PATHS["candidates_path"]))
        self.targets = pd.read_csv(self._resolve_path(self.cfg.DATA_PATHS["targets_path"]))

        # === ОРИГИНАЛЬНАЯ ЛОГИКА ПРЕОБРАЗОВАНИЯ ТИПОВ ===
        self.interactions["event_ts"] = pd.to_datetime(self.interactions["event_ts"])

        # Кастуем возраст к int32 с филлом -1, а пол к int8
        self.users["age"] = self.users["age"].fillna(-1).astype(np.int32)
        self.users["gender"] = self.users["gender"].fillna(0).astype(np.int8)

        # === ОРИГИНАЛЬНАЯ ЛОГИКА СТАТИЧЕСКИХ ДЖОЙНОВ ===
        # Attach author names (not used as a feature but useful for inspection).
        self.editions = self.editions.merge(self.authors, on="author_id", how="left")

        # Build book_id -> list[genre_name], then edition -> genres
        bg = self.book_genres.merge(self.genres, on="genre_id", how="left")
        book_to_genres = bg.groupby("book_id", sort=False)["genre_name"].apply(list).to_dict()
        
        # Применяем маппинг, если жанров нет - оставляем пустой список (ровно как в оригинале)
        self.editions["genres"] = self.editions["book_id"].map(book_to_genres).apply(
            lambda x: x if isinstance(x, list) else []
        )
        
        print(f"[DataHandler] Data loaded successfully. "
              f"Interactions: {len(self.interactions)}, Users: {len(self.users)}, "
              f"Editions: {len(self.editions)}")

In [ ]:
import os
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Optional
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, IncrementalPCA
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import coo_matrix

@dataclass
class TextModel:
    vectorizer: TfidfVectorizer
    svd: TruncatedSVD
    edition_id_to_row: Dict[int, int]
    item_emb: np.ndarray          # float32 (n_items, d)
    item_emb_norm: np.ndarray     # float32 (n_items, d)

@dataclass
class QwenTextStore:
    edition_id_to_row: Dict[int, int]
    edition_emb_norm: np.ndarray
    edition_pca: np.ndarray
    book_id_to_row: Dict[int, int]
    book_emb_norm: np.ndarray
    book_pca: np.ndarray
    author_emb_norm: np.ndarray
    author_pca: np.ndarray
    pca_mean: np.ndarray
    pca_components: np.ndarray
    
    @property
    def pca_dim(self) -> int:
        return int(self.pca_components.shape[0])

    def pca_transform(self, X: np.ndarray) -> np.ndarray:
        if X.size == 0:
            return np.empty((0, self.pca_dim), dtype=np.float32)
        X = X.astype(np.float32, copy=False)
        return (X - self.pca_mean[None, :]) @ self.pca_components.T

def _l2_normalize_rows_inplace(X: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X /= np.maximum(norms, eps)
    return X

def _compute_pca_projection_batched(X: np.ndarray, mean: np.ndarray, components: np.ndarray, batch_size: int) -> np.ndarray:
    n = int(X.shape[0])
    k = int(components.shape[0])
    out = np.empty((n, k), dtype=np.float32)
    for start in range(0, n, int(batch_size)):
        end = min(n, start + int(batch_size))
        xb = X[start:end].astype(np.float32, copy=False)
        out[start:end] = (xb - mean[None, :]) @ components.T
    return out

In [ ]:
class ConstantFeatureGenerator:
    """
    Отвечает за генерацию статических признаков (констант), которые не зависят от времени 
    и истории пользователя. Обучается один раз на старте пайплайна.
    """
    def __init__(self, cfg):
        self.cfg = cfg
        self.author_le = LabelEncoder()
        self.genre_le = LabelEncoder()
        self.text_model: Optional[TextModel] = None
        self.qwen_store: Optional[QwenTextStore] = None
        self.static_item_features: Optional[pd.DataFrame] = None

    def fit(self, data_handler):
        """
        Обучает текстовые модели, LabelEncoders и подготавливает датафрейм со статикой.
        """
        editions = data_handler.editions
        
        # 1. Обучаем глобальные LabelEncoders для авторов и жанров
        print("[ConstantFeatureGenerator] Fitting LabelEncoders...")
        author_raw = editions["author_id"].fillna(-1).astype(np.int64).astype(str)
        self.author_le.fit(author_raw.unique())
        
        all_genres = set(["unkgenre"])
        for gl in editions["genres"].dropna():
            if isinstance(gl, list):
                all_genres.update(gl)
        self.genre_le.fit(list(all_genres))
        
        # 2. Обучаем SVD для текстов (title + description)
        print("[ConstantFeatureGenerator] Fitting SVD Text Model...")
        self.text_model = self._fit_text_model(editions)
        
        # 3. Готовим пространство Qwen (включая агрегацию до книг и авторов)
        if getattr(self.cfg, "USE_QWEN_TEXT_FEATURES", False):
            print("[ConstantFeatureGenerator] Building Qwen Store...")
            self.qwen_store = self._build_qwen_store(editions)
            
        # 4. Собираем единую таблицу статических признаков
        print("[ConstantFeatureGenerator] Extracting static item features...")
        self.static_item_features = self._build_static_item_features(editions)
        print("[ConstantFeatureGenerator] Fitting Done.")
        return self

    def transform(self, pairs_df: pd.DataFrame) -> pd.DataFrame:
        """
        Приклеивает (merge) готовую статику к переданному датафрейму кандидатов.
        Векторные расстояния (SVD cos, Qwen cos) здесь НЕ вычисляются, так как они 
        требуют агрегированных профилей юзеров (что зависит от cutoff_ts).
        """
        if self.static_item_features is None:
            raise ValueError("ConstantFeatureGenerator не обучен! Вызовите .fit() сначала.")
            
        return pairs_df.merge(self.static_item_features, on="edition_id", how="left")

    # ================= Внутренние методы логики ================= #

    def _fit_text_model(self, editions: pd.DataFrame) -> TextModel:
        texts = (editions["title"].fillna("") + " " + editions["description"].fillna("")).astype(str).tolist()
        vectorizer = TfidfVectorizer(max_features=20000)
        X = vectorizer.fit_transform(texts)
        
        max_comp = max(1, min(X.shape[0] - 1, X.shape[1] - 1))
        n_comp = int(min(getattr(self.cfg, "N_TEXT_COMPONENTS", 32), max_comp))
        svd = TruncatedSVD(n_components=n_comp, random_state=42)
        Z = svd.fit_transform(X).astype(np.float32)
        
        norms = np.linalg.norm(Z, axis=1, keepdims=True)
        Z_norm = Z / np.maximum(norms, 1e-8)
        
        edition_ids = editions["edition_id"].astype(np.int64).to_numpy()
        edition_id_to_row = {int(eid): i for i, eid in enumerate(edition_ids.tolist())}
        
        return TextModel(
            vectorizer=vectorizer,
            svd=svd,
            edition_id_to_row=edition_id_to_row,
            item_emb=Z,
            item_emb_norm=Z_norm,
        )

    def _build_qwen_store(self, editions: pd.DataFrame) -> Optional[QwenTextStore]:
        feather_path =  "data/edition_qwen3_embedding.feather" 
        pca_components = getattr(self.cfg, "QWEN_PCA_COMPONENTS", 16)
        pca_batch_size = getattr(self.cfg, "QWEN_PCA_BATCH_SIZE", 8192)
        cache_dir = "cache"
        
        if not os.path.exists(feather_path):
            print(f"[WARN] Qwen embeddings file not found: {feather_path}")
            return None

        os.makedirs(cache_dir, exist_ok=True)
        embs_df = pd.read_feather(feather_path)
        
        emb_cols = [c for c in embs_df.columns if str(c).startswith("emb_")]
        if not emb_cols:
            return None

        edition_ids = editions["edition_id"].astype(np.int64).to_numpy()
        embs_df = embs_df.set_index("edition_id")
        
        # Строгое выравнивание по таблице editions
        E = embs_df.loc[edition_ids, emb_cols].to_numpy(dtype=np.float32, copy=True)
        del embs_df
        _l2_normalize_rows_inplace(E)
        edition_id_to_row = {int(eid): int(i) for i, eid in enumerate(edition_ids.tolist())}

        # Кеширование и расчет PCA
        cache_path = os.path.join(cache_dir, f"qwen_pca{int(pca_components)}.npz")
        mean, comps, edition_pca = None, None, None

        if os.path.exists(cache_path):
            try:
                npz = np.load(cache_path, allow_pickle=False)
                if int(npz["k"]) == pca_components and np.all(npz["edition_ids"] == edition_ids):
                    mean = npz["mean"].astype(np.float32, copy=False)
                    comps = npz["components"].astype(np.float32, copy=False)
                    edition_pca = npz["edition_pca"].astype(np.float32, copy=False)
            except Exception:
                pass

        if mean is None or comps is None or edition_pca is None:
            print(f"Fitting Qwen PCA({pca_components}) on edition embeddings...")
            pca = IncrementalPCA(n_components=pca_components, batch_size=pca_batch_size)
            for start in range(0, E.shape[0], pca_batch_size):
                end = min(E.shape[0], start + pca_batch_size)
                pca.partial_fit(E[start:end])
            mean = pca.mean_.astype(np.float32, copy=False)
            comps = pca.components_.astype(np.float32, copy=False)
            edition_pca = _compute_pca_projection_batched(E, mean, comps, pca_batch_size)
            try:
                np.savez_compressed(
                    cache_path, k=np.array([pca_components], dtype=np.int32),
                    edition_ids=edition_ids, mean=mean, components=comps, edition_pca=edition_pca,
                )
            except Exception:
                pass

        # Агрегация до уровня Book
        book_codes, book_ids = pd.factorize(editions["book_id"].fillna(-1).astype(np.int64), sort=False)
        n_books, n_ed = int(len(book_ids)), int(E.shape[0])
        B = coo_matrix((np.ones(n_ed, dtype=np.float32), (book_codes.astype(np.int32), np.arange(n_ed, dtype=np.int32))), shape=(n_books, n_ed)).tocsr()
        book_sum = (B @ E).astype(np.float32, copy=False)
        book_cnt = np.asarray(B.sum(axis=1)).ravel().astype(np.float32, copy=False)
        book_emb = book_sum / np.maximum(book_cnt[:, None], 1.0)
        _l2_normalize_rows_inplace(book_emb)
        book_emb_norm = book_emb.astype(np.float32, copy=False)
        book_pca = (book_emb_norm - mean[None, :]) @ comps.T
        book_id_to_row = {int(bid): int(i) for i, bid in enumerate(book_ids.tolist())}

        # Агрегация до уровня Author (с использованием author_le)
        author_raw = editions["author_id"].fillna(-1).astype(np.int64).astype(str).to_numpy()
        author_codes = self.author_le.transform(author_raw).astype(np.int32, copy=False)
        n_authors = int(len(self.author_le.classes_))
        A = coo_matrix((np.ones(n_ed, dtype=np.float32), (author_codes, np.arange(n_ed, dtype=np.int32))), shape=(n_authors, n_ed)).tocsr()
        author_sum = (A @ E).astype(np.float32, copy=False)
        author_cnt = np.asarray(A.sum(axis=1)).ravel().astype(np.float32, copy=False)
        author_emb = author_sum / np.maximum(author_cnt[:, None], 1.0)
        _l2_normalize_rows_inplace(author_emb)
        author_emb_norm = author_emb.astype(np.float32, copy=False)
        author_pca = (author_emb_norm - mean[None, :]) @ comps.T

        return QwenTextStore(
            edition_id_to_row=edition_id_to_row, edition_emb_norm=E, edition_pca=edition_pca,
            book_id_to_row=book_id_to_row, book_emb_norm=book_emb_norm, book_pca=book_pca,
            author_emb_norm=author_emb_norm, author_pca=author_pca,
            pca_mean=mean, pca_components=comps,
        )

    def _build_static_item_features(self, editions: pd.DataFrame) -> pd.DataFrame:
        cols = ["edition_id", "book_id", "author_id", "publisher_id", "language_id", "publication_year"]
        df = editions[cols].copy()
        df["age_restriction"] = editions["age_restriction"] if "age_restriction" in editions.columns else 0

        # Фичи качества текста
        if getattr(self.cfg, "ENABLE_TEXT_QUALITY_FEATURES", False):
            title = editions["title"].fillna("").astype(str) if "title" in editions.columns else pd.Series([""] * len(editions))
            desc = editions["description"].fillna("").astype(str) if "description" in editions.columns else pd.Series([""] * len(editions))
            
            df["title_len"] = title.str.len().astype(np.int32)
            df["desc_len"] = desc.str.len().astype(np.int32)
            df["text_len"] = (df["title_len"] + df["desc_len"]).astype(np.int32)
            df["has_title"] = (df["title_len"] > 0).astype(np.int8)
            df["has_desc"] = (df["desc_len"] > 0).astype(np.int8)
            df["title_word_cnt"] = title.str.count(r"\S+").astype(np.int32)
            df["desc_word_cnt"] = desc.str.count(r"\S+").astype(np.int32)
            df["text_word_cnt"] = (df["title_word_cnt"] + df["desc_word_cnt"]).astype(np.int32)
            df["desc_title_len_ratio"] = (df["desc_len"] / (df["title_len"] + 1.0)).astype(np.float32)
            df["log1p_text_len"] = np.log1p(df["text_len"].astype(np.float32)).astype(np.float32)

        # Жанры
        df["item_num_genres"] = editions["genres"].apply(lambda x: len(x) if isinstance(x, list) else 0).astype(np.int16)
        df["primary_genre"] = editions["genres"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else "unkgenre")
        df["primary_genre_id"] = self.genre_le.transform(df["primary_genre"].astype(str)).astype(np.int32)

        # Структура книги (только статика)
        if getattr(self.cfg, "ENABLE_BOOK_EDITION_STRUCTURE_FEATURES", False):
            meta = editions[["edition_id", "book_id", "publication_year"]].copy()
            meta["book_id"] = pd.to_numeric(meta["book_id"], errors="coerce").fillna(-1).astype(np.int64)
            meta["publication_year"] = pd.to_numeric(meta["publication_year"], errors="coerce").fillna(-1).astype(np.int32)

            b = meta[meta["book_id"] >= 0].groupby("book_id", as_index=False).agg(
                book_num_editions=("edition_id", "size"),
                book_pub_year_min=("publication_year", "min"),
                book_pub_year_max=("publication_year", "max"),
            )
            if not b.empty:
                b["book_pub_year_span"] = (b["book_pub_year_max"] - b["book_pub_year_min"]).astype(np.int32)
                df = df.merge(b, on="book_id", how="left")
            else:
                df["book_num_editions"] = 0
                df["book_pub_year_min"] = -1
                df["book_pub_year_max"] = -1
                df["book_pub_year_span"] = 0
                
            df["book_num_editions"] = df["book_num_editions"].fillna(0).astype(np.int16)
            df["log1p_book_num_editions"] = np.log1p(df["book_num_editions"].astype(np.float32)).astype(np.float32)
            df["book_pub_year_min"] = df["book_pub_year_min"].fillna(-1).astype(np.int32)
            df["book_pub_year_max"] = df["book_pub_year_max"].fillna(-1).astype(np.int32)
            df["book_pub_year_span"] = df["book_pub_year_span"].fillna(0).astype(np.int32)

            yr = meta[meta["book_id"] >= 0].copy()
            if not yr.empty:
                yr["edition_pub_year_rank_in_book_desc"] = yr.groupby("book_id")["publication_year"].rank(method="dense", ascending=False).fillna(0).astype(np.int16)
                yr["edition_pub_year_rank_in_book_asc"] = yr.groupby("book_id")["publication_year"].rank(method="dense", ascending=True).fillna(0).astype(np.int16)
                df = df.merge(yr[["edition_id", "edition_pub_year_rank_in_book_desc", "edition_pub_year_rank_in_book_asc"]], on="edition_id", how="left")
            else:
                df["edition_pub_year_rank_in_book_desc"] = 0
                df["edition_pub_year_rank_in_book_asc"] = 0

            df["edition_pub_year_rank_in_book_desc"] = df["edition_pub_year_rank_in_book_desc"].fillna(0).astype(np.int16)
            df["edition_pub_year_rank_in_book_asc"] = df["edition_pub_year_rank_in_book_asc"].fillna(0).astype(np.int16)

            pub_year = pd.to_numeric(df["publication_year"], errors="coerce").fillna(-1).astype(np.int32)
            df["is_latest_edition_in_book"] = ((pub_year >= 0) & (df["book_pub_year_max"].to_numpy(dtype=np.int32) >= 0) & (pub_year == df["book_pub_year_max"])).astype(np.int8)

            span = df["book_pub_year_span"].to_numpy(dtype=np.float32)
            miny = df["book_pub_year_min"].to_numpy(dtype=np.float32)
            py = pub_year.to_numpy(dtype=np.float32)
            pos = np.divide((py - miny), span, out=np.zeros_like(py), where=span > 0)
            df["edition_pub_year_position_in_book"] = np.clip(pos, 0.0, 1.0).astype(np.float32)

        return df

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
import implicit

class ALSFeatureGenerator:
    """
    Генератор признаков на основе матричной факторизации (Implicit ALS).
    Обучает отдельные модели для каждого уровня (edition, book, author, publisher).
    """
    def __init__(self, cfg):
        self.cfg = cfg
        self.models = {}
        self.user_maps = {}
        self.item_maps = {}
        self.global_z_stats = {}

    

    def update(self, full_history_df: pd.DataFrame, data_handler):
        print("[ALSFeatureGenerator] Re-fitting on full history for inference...")
        return self.fit(full_history_df, data_handler)

    def fit(self, history_df: pd.DataFrame, data_handler) -> 'ALSFeatureGenerator':
        if not getattr(self.cfg, "USE_IMPLICIT_ALS_FEATURES", False):
            return self

        levels = getattr(self.cfg, "IMPLICIT_ALS_LEVELS", ("edition",))
        editions = data_handler.editions[["edition_id", "book_id", "author_id", "publisher_id"]]
        df = history_df.merge(editions, on="edition_id", how="left")
        
        df["_weight"] = self._compute_interaction_weights(df)
        df = df[df["_weight"] > 0].copy()

        for level in levels:
            level_col = f"{level}_id" if level != "edition" else "edition_id"
            if level_col not in df.columns: continue
                
            level_df = df.groupby(["user_id", level_col], as_index=False)["_weight"].max()
            users, items = level_df["user_id"].unique(), level_df[level_col].unique()
            u_map = {u: i for i, u in enumerate(users)}
            i_map = {item: i for i, item in enumerate(items)}
            
            row = level_df["user_id"].map(u_map).astype(np.int32).to_numpy()
            col = level_df[level_col].map(i_map).astype(np.int32).to_numpy()
            data = level_df["_weight"].astype(np.float32).to_numpy()
            
            sparse_item_user = coo_matrix((data, (col, row)), shape=(len(items), len(users))).tocsr()
            
            factors = self.cfg.IMPLICIT_ALS_FACTORS_BY_LEVEL.get(level, 64)
            model = implicit.als.AlternatingLeastSquares(
                factors=factors, regularization=self.cfg.IMPLICIT_ALS_REG,
                iterations=self.cfg.IMPLICIT_ALS_ITERATIONS, random_state=self.cfg.IMPLICIT_ALS_RANDOM_STATE,
                calculate_training_loss=False
            )
            model.fit(sparse_item_user, show_progress=False)
            
            self.models[level] = model
            self.user_maps[level] = u_map
            self.item_maps[level] = i_map
            
            if getattr(self.cfg, "ALS_FEATS_INCLUDE_GLOBAL_Z", False):
                self._precalc_global_z_stats(level, model, len(users), len(items), u_map)

        return self

    def transform(self, pairs_df: pd.DataFrame, data_handler) -> pd.DataFrame:
        if not self.models:
            return pairs_df

        print("[ALSFeatureGenerator] Transforming features...")
        res = pairs_df.copy()
        
        # ЗАЩИТА ОТ КОЛЛИЗИИ
        missing_cols = [c for c in ["book_id", "author_id", "publisher_id"] if c not in res.columns]
        if missing_cols:
            editions = data_handler.editions[["edition_id"] + missing_cols]
            res = res.merge(editions, on="edition_id", how="left")

        for level, model in self.models.items():
            level_col = f"{level}_id" if level != "edition" else "edition_id"
            if level_col not in res.columns: continue

            u_map, i_map = self.user_maps[level], self.item_maps[level]
            u_idx = res["user_id"].map(u_map).fillna(-1).astype(np.int32).to_numpy()
            i_idx = res[level_col].map(i_map).fillna(-1).astype(np.int32).to_numpy()
            valid_mask = (u_idx >= 0) & (i_idx >= 0)
            
            if getattr(self.cfg, "ALS_FEATS_INCLUDE_SCORE", True):
                scores = np.zeros(len(res), dtype=np.float32)
                if valid_mask.any():
                    scores[valid_mask] = np.sum(model.user_factors[u_idx[valid_mask]] * model.item_factors[i_idx[valid_mask]], axis=1)
                res[f"als_{level}_score"] = scores

            if getattr(self.cfg, "ALS_FEATS_INCLUDE_GLOBAL_Z", False) and level in self.global_z_stats:
                z_stats = self.global_z_stats[level]
                res = res.merge(z_stats, on="user_id", how="left")
                score_col, mean_col, std_col = f"als_{level}_score", f"als_{level}_mean", f"als_{level}_std"
                eps = getattr(self.cfg, "IMPLICIT_ALS_GLOBAL_Z_EPS", 1e-6)
                res[f"als_{level}_global_z"] = (res[score_col] - res[mean_col].fillna(0)) / res[std_col].fillna(1.0).replace(0, eps)
                res = res.drop(columns=[mean_col, std_col])

            # 4. Разворачиваем векторы (эмбеддинги) в фичи
            if getattr(self.cfg, "ALS_FEATS_INCLUDE_EMBEDDINGS", True):
                u_f = np.zeros((len(res), model.factors), dtype=np.float32)
                i_f = np.zeros((len(res), model.factors), dtype=np.float32)
                if valid_mask.any():
                    u_f[valid_mask] = model.user_factors[u_idx[valid_mask]]
                    i_f[valid_mask] = model.item_factors[i_idx[valid_mask]]
                
                u_df = pd.DataFrame(u_f, columns=[f"als_{level}_user_dim_{k}" for k in range(model.factors)])
                i_df = pd.DataFrame(i_f, columns=[f"als_{level}_item_dim_{k}" for k in range(model.factors)])
                res = pd.concat([res, u_df, i_df], axis=1)

        if missing_cols:
            res = res.drop(columns=missing_cols, errors="ignore")
            
        return res

    def _compute_interaction_weights(self, df: pd.DataFrame) -> np.ndarray:
        w = np.where(df["event_type"] == getattr(self.cfg, "READ_LABEL", 3), 
                     getattr(self.cfg, "ALS_READ_WEIGHT", 3.0), 
                     getattr(self.cfg, "ALS_WISHLIST_WEIGHT", 1.0)).astype(np.float32)
        if getattr(self.cfg, "ENABLE_RATING_AWARE_INTERACTION_WEIGHTS", False) and getattr(self.cfg, "RATING_AWARE_APPLY_TO_CF", True):
            ratings = df["rating"].fillna(0.0).to_numpy()
            mask_rated = ratings > 0
            w[mask_rated] = w[mask_rated] * ((ratings[mask_rated] / getattr(self.cfg, "RATING_AWARE_MAX_RATING", 10.0)) ** getattr(self.cfg, "RATING_AWARE_GAMMA", 1.0))
            low_mask = mask_rated & (ratings[mask_rated] <= getattr(self.cfg, "RATING_AWARE_LOW_RATING_THRESHOLD", 3.0))
            full_low_mask = np.zeros_like(mask_rated, dtype=bool)
            full_low_mask[mask_rated] = low_mask
            if getattr(self.cfg, "RATING_AWARE_LOW_RATING_ACTION", "downweight") == "downweight":
                w[full_low_mask] *= getattr(self.cfg, "RATING_AWARE_LOW_RATING_MULT", 0.2)
        return w

    def _precalc_global_z_stats(self, level: str, model, n_users: int, n_items: int, u_map: dict):
        sample_size = min(getattr(self.cfg, "IMPLICIT_ALS_GLOBAL_Z_SAMPLE_SIZE", 10000), n_items)
        np.random.seed(42)
        rand_item_idx = np.random.choice(n_items, size=sample_size, replace=False)
        scores_matrix = model.user_factors @ model.item_factors[rand_item_idx].T
        inv_u_map = {idx: uid for uid, idx in u_map.items()}
        self.global_z_stats[level] = pd.DataFrame({
            "user_id": [inv_u_map[i] for i in range(n_users)],
            f"als_{level}_mean": np.mean(scores_matrix, axis=1),
            f"als_{level}_std": np.std(scores_matrix, axis=1)
        })

In [ ]:
import numpy as np
import pandas as pd

class DynamicFeatureGenerator:
    """
    Генерирует динамические признаки: счетчики, тренды, популярность, байесовские рейтинги.
    Строго привязан к cutoff_ts (моменту времени, на который мы "замораживаем" историю).
    """
    def __init__(self, cfg):
        self.cfg = cfg
        self.user_stats = None
        self.edition_stats = None
        self.book_stats = None
        self.author_stats = None
        self.publisher_stats = None
        self.global_rating_mean = 0.0
        self.rating_prior_weight = 5.0
        self.genre_stats = None

    def fit(self, history_df: pd.DataFrame, data_handler, cutoff_ts: pd.Timestamp):
        self.last_fit_cutoff = cutoff_ts # Запоминаем дату, относительно которой считаем
        print(f"[DynamicFeatureGenerator] Fitting dynamic stats up to {cutoff_ts}...")
        print(f"[DynamicFeatureGenerator] Fitting dynamic stats up to {cutoff_ts}...")
        self.rating_prior_weight = getattr(self.cfg, "RATING_PRIOR_COUNT", 10.0) # ИСПРАВЛЕНИЕ БАЙЕСА
        
        editions = data_handler.editions[["edition_id", "book_id", "author_id", "publisher_id"]]
        df = history_df.merge(editions, on="edition_id", how="left")
        
        df["days_to_cutoff"] = (cutoff_ts - df["event_ts"]).dt.total_seconds() / (24 * 3600)
        df["days_to_cutoff"] = np.clip(df["days_to_cutoff"], 0, None)
        
        # ИСПРАВЛЕНИЕ: Добавляем флаги для правильного подсчета read/wishlist
        df["is_read"] = (df["event_type"] == 2).astype(np.int8)
        df["is_wish"] = (df["event_type"] == 1).astype(np.int8)
        
        mask_7d = df["days_to_cutoff"] <= 7
        mask_14d = df["days_to_cutoff"] <= 14
        mask_30d = df["days_to_cutoff"] <= 30
        
        valid_ratings = df[df["rating"] > 0]["rating"]
        self.global_rating_mean = valid_ratings.mean() if len(valid_ratings) > 0 else 5.0

        self.edition_stats = self._build_entity_stats(df, "edition_id", mask_7d, mask_14d, mask_30d)
        self.edition_stats = self._add_rating_stats(self.edition_stats, df, "edition_id")

        self.book_stats = self._build_entity_stats(df, "book_id", mask_7d, mask_14d, mask_30d)
        self.book_stats = self._add_rating_stats(self.book_stats, df, "book_id")

        self.author_stats = self._build_entity_stats(df, "author_id", mask_7d, mask_14d, mask_30d)
        self.author_stats = self._add_rating_stats(self.author_stats, df, "author_id")

        self.publisher_stats = self._build_entity_stats(df, "publisher_id", mask_7d, mask_14d, mask_30d)
        self.user_stats = self._build_user_stats(df, mask_30d)

        print("[DynamicFeatureGenerator] Fitting Done.")
        return self

    def _build_entity_stats(self, df: pd.DataFrame, entity_col: str, m7: pd.Series, m14: pd.Series, m30: pd.Series) -> pd.DataFrame:
        prefix = entity_col.replace("_id", "")
        
        # ИСПРАВЛЕНИЕ: Правильная агрегация как в оригинале
        agg_dict = {
            f"{prefix}_popularity": ("event_type", "size"),
            f"{prefix}_read_count": ("is_read", "sum"),
        }
        if getattr(self.cfg, "USE_WISHLISTS", True):
            agg_dict[f"{prefix}_wishlist_count"] = ("is_wish", "sum")
            
        stats = df.groupby(entity_col).agg(**agg_dict).reset_index()
        
        # Recency
        last_ev = df.groupby(entity_col)["days_to_cutoff"].min().rename(f"{prefix}_days_since_last_event").reset_index()
        stats = stats.merge(last_ev, on=entity_col, how="left")
        
        # Тренды (30d, 14d, 7d)
        for suff, mask in [("30d", m30), ("14d", m14), ("7d", m7)]:
            agg_w = {
                f"{prefix}_popularity_{suff}": ("event_type", "size"),
                f"{prefix}_read_count_{suff}": ("is_read", "sum"),
            }
            if getattr(self.cfg, "USE_WISHLISTS", True):
                agg_w[f"{prefix}_wishlist_count_{suff}"] = ("is_wish", "sum")
            stats_w = df[mask].groupby(entity_col).agg(**agg_w).reset_index()
            stats = stats.merge(stats_w, on=entity_col, how="left")

        stats = stats.fillna(0)
        
        # Логарифмы
        stats[f"log1p_{prefix}_popularity"] = np.log1p(stats[f"{prefix}_popularity"]).astype(np.float32)
        stats[f"log1p_{prefix}_read_count"] = np.log1p(stats[f"{prefix}_read_count"]).astype(np.float32)
        stats[f"log1p_{prefix}_popularity_30d"] = np.log1p(stats[f"{prefix}_popularity_30d"]).astype(np.float32)
        stats[f"log1p_{prefix}_read_count_30d"] = np.log1p(stats[f"{prefix}_read_count_30d"]).astype(np.float32)

        return stats

    def _add_rating_stats(self, stats_df: pd.DataFrame, df: pd.DataFrame, entity_col: str) -> pd.DataFrame:
        prefix = entity_col.replace("_id", "")
        # ИСПРАВЛЕНИЕ: Берем только READ_LABEL (2) для рейтингов
        rated = df[(df["event_type"] == 2) & (df["rating"] > 0)]
        if rated.empty:
            return stats_df
            
        g = rated.groupby(entity_col)["rating"]
        r_stats = g.agg(rating_mean="mean", rating_std="std", rating_count="count").rename(columns=lambda x: f"{prefix}_{x}")
        r_stats[f"{prefix}_rating_std"] = r_stats[f"{prefix}_rating_std"].fillna(0)
        
        v = r_stats[f"{prefix}_rating_count"]
        R = r_stats[f"{prefix}_rating_mean"]
        m = self.rating_prior_weight
        C = self.global_rating_mean
        
        r_stats[f"{prefix}_bayes_avg_rating"] = ((v * R + m * C) / (v + m)).astype(np.float32)
        # Добавляем rating_conf как в оригинале
        r_stats[f"{prefix}_rating_conf"] = (v / (v + m)).astype(np.float32)
        
        stats_df = stats_df.merge(r_stats.reset_index(), on=entity_col, how="left")
        stats_df[f"{prefix}_bayes_avg_rating"] = stats_df[f"{prefix}_bayes_avg_rating"].fillna(np.float32(C))
        stats_df[f"{prefix}_rating_count"] = stats_df[f"{prefix}_rating_count"].fillna(0.0)
        stats_df[f"{prefix}_rating_conf"] = stats_df[f"{prefix}_rating_conf"].fillna(0.0)
        return stats_df

    def transform(self, pairs_df: pd.DataFrame, data_handler) -> pd.DataFrame:
        if self.edition_stats is None:
            raise ValueError("DynamicFeatureGenerator не обучен! Вызовите .fit() сначала.")
            
        print("[DynamicFeatureGenerator] Transforming (merging stats)...")
        res = pairs_df.copy()
        
        # ЗАЩИТА ОТ КОЛЛИЗИИ: подтягиваем метаданные, только если их еще нет
        missing_cols = [c for c in ["book_id", "author_id", "publisher_id"] if c not in res.columns]
        if missing_cols:
            editions = data_handler.editions[["edition_id"] + missing_cols]
            res = res.merge(editions, on="edition_id", how="left")
        
        res = res.merge(self.user_stats, on="user_id", how="left")
        res = res.merge(self.edition_stats, on="edition_id", how="left")
        res = res.merge(self.book_stats, on="book_id", how="left")
        res = res.merge(self.author_stats, on="author_id", how="left")
        res = res.merge(self.publisher_stats, on="publisher_id", how="left")
        
        fill_dict = {}
        for col in res.columns:
            if col.endswith("_count") or "popularity" in col or col.endswith("_cnt"):
                fill_dict[col] = 0.0
            elif "days_since" in col:
                fill_dict[col] = 9999.0
            elif "rating" in col and "bayes" in col:
                fill_dict[col] = np.float32(self.global_rating_mean)
                
        res = res.fillna(fill_dict)
        
        # Удаляем только те временные ключи, которые мы добавили сами
        if missing_cols:
            res = res.drop(columns=missing_cols, errors="ignore")
        
        return res

    

    

    def _build_user_stats(self, df: pd.DataFrame, m30: pd.Series) -> pd.DataFrame:
        g = df.groupby("user_id")
        u_stats = pd.DataFrame(index=g.indices.keys())
        u_stats.index.name = "user_id"
        
        u_stats["user_activity_total"] = g.size()
        u_stats["user_activity_30d"] = df[m30].groupby("user_id").size()
        u_stats["user_days_since_last_event"] = g["days_to_cutoff"].min()
        
        u_stats = u_stats.fillna(0)
        u_stats["log1p_user_activity"] = np.log1p(u_stats["user_activity_total"]).astype(np.float32)
        
        rated = df[df["rating"] > 0]
        r_g = rated.groupby("user_id")["rating"]
        
        u_ratings = r_g.agg(user_mean_rating="mean", user_std_rating="std", user_rating_count="count")
        u_ratings["user_std_rating"] = u_ratings["user_std_rating"].fillna(1e-3)
        
        u_stats = u_stats.merge(u_ratings, on="user_id", how="left")
        
        u_stats["user_mean_rating"] = u_stats["user_mean_rating"].fillna(np.float32(self.global_rating_mean))
        u_stats["user_std_rating"] = u_stats["user_std_rating"].fillna(1.0)
        u_stats["user_rating_count"] = u_stats["user_rating_count"].fillna(0.0)
        # --- БЛОК ДОБАВЛЕНИЯ ТОП-ЯЗЫКА И ИЗДАТЕЛЯ (ИСПРАВЛЕННЫЙ) ---
        if "language_id" in df.columns:
            top_lang = df[df["event_type"] == 2].groupby("user_id")["language_id"].agg(lambda x: x.mode()[0] if not x.mode().empty else -1)
            u_stats["user_top_language_id"] = top_lang
        
        if "publisher_id" in df.columns:
            top_pub = df[df["event_type"] == 2].groupby("user_id")["publisher_id"].agg(lambda x: x.mode()[0] if not x.mode().empty else -1)
            u_stats["user_top_publisher_id"] = top_pub
            
        # Безопасный филл
        if "user_top_language_id" not in u_stats.columns:
            u_stats["user_top_language_id"] = -1
        u_stats["user_top_language_id"] = u_stats["user_top_language_id"].fillna(-1).astype(np.int32)
        
        if "user_top_publisher_id" not in u_stats.columns:
            u_stats["user_top_publisher_id"] = -1
        u_stats["user_top_publisher_id"] = u_stats["user_top_publisher_id"].fillna(-1).astype(np.int32)
        
        return u_stats.reset_index()
        return u_stats.reset_index()

In [ ]:
import numpy as np
import pandas as pd
from typing import Dict, List, Set, Tuple

class CandidateGenerator:
    """
    Генерирует датасеты пар (user_id, edition_id) для обучения и инференса.
    Реализует поиск позитивных примеров (ground truth) и сложный негативный семплинг.
    """
    def __init__(self, cfg):
        self.cfg = cfg

    def generate_train(self, interactions: pd.DataFrame, candidates_df: pd.DataFrame, 
                       cutoff_ts: pd.Timestamp, als_generator=None) -> pd.DataFrame:
        print(f"[CandidateGenerator] Generating train dataset for cutoff {cutoff_ts}...")
        
        # 1. Точные границы (включая последнюю микросекунду, как в оригинале)
        target_end_ts = interactions["event_ts"].max()
        
        # 2. Формируем позитивы (READ=3, WISHLIST=1)
        allowed_pos_types = [2] + ([1] if getattr(self.cfg, "USE_WISHLISTS", True) else [])
        target_mask = (interactions["event_ts"] >= cutoff_ts) & (interactions["event_ts"] <= target_end_ts)
        target_df = interactions[target_mask].copy()
        target_df = target_df[target_df["event_type"].isin(allowed_pos_types)]
        
        # Фильтруем только тех юзеров, что есть в файле кандидатов
        candidate_users = set(candidates_df["user_id"].unique())
        target_df = target_df[target_df["user_id"].isin(candidate_users)]
        
        # Группировка: READ побеждает WISHLIST
        positives = target_df.groupby(["user_id", "edition_id"], as_index=False)["event_type"].max()
        
        label_map = {2: float(self.cfg.READ_LABEL), 1: float(self.cfg.WISHLIST_LABEL)}
        positives["target"] = positives["event_type"].map(label_map)
        
        # Создаем сет кортежей для быстрой фильтрации негативов
        pos_pairs_set = set(zip(positives["user_id"], positives["edition_id"]))
        
        # 3. Генерация негативов (cand_all_nonpos)
        # В ОРИГИНАЛЕ: Негативы берутся ТОЛЬКО для тех, у кого есть позитивы в окне!
        active_train_users = positives["user_id"].unique()
        
        ns_cfg = self.cfg.NEGATIVE_SAMPLING_CONFIG
        max_n = ns_cfg.get("cand_all_nonpos", 200)
        
        # Берем кандидатов только для "активных" юзеров
        neg_candidates = candidates_df[candidates_df["user_id"].isin(active_train_users)].copy()
        
        # ИСПРАВЛЕНИЕ ОШИБКИ: Используем списковое включение, чтобы избежать проблем с индексами
        # Это гарантирует правильный булевый массив той же длины, что и neg_candidates
        is_pos_mask = [
            (u, e) in pos_pairs_set 
            for u, e in zip(neg_candidates["user_id"], neg_candidates["edition_id"])
        ]
        neg_candidates["is_pos"] = is_pos_mask
        
        # Теперь ~ работает корректно, так как в колонке чистый bool
        negatives = neg_candidates[~neg_candidates["is_pos"]].copy()
        
        # Ограничение N на юзера (head(200))
        negatives = negatives.groupby("user_id").head(max_n).reset_index(drop=True)
        negatives["target"] = 0.0
        
        # 4. Сборка финального train_df
        train_df = pd.concat([
            positives[["user_id", "edition_id", "target"]], 
            negatives[["user_id", "edition_id", "target"]]
        ], ignore_index=True)
        
        # Финальная дедупликация (позитивы в приоритете)
        train_df = train_df.sort_values("target", ascending=False).drop_duplicates(
            subset=["user_id", "edition_id"], keep="first"
        )
        
        print(f"[CandidateGenerator] Train dataset built: {len(train_df)} pairs "
              f"({len(positives)} positives from {len(active_train_users)} users).")
        
        return train_df

    def generate_test(self, candidates_df: pd.DataFrame) -> pd.DataFrame:
        """
        Формирует выборку для инференса (сабмита).
        Для теста мы просто берем предоставленных кандидатов и проставляем пустой таргет.
        """
        print("[CandidateGenerator] Generating test dataset from candidates file...")
        test_df = candidates_df[["user_id", "edition_id"]].copy()
        test_df["target"] = np.nan # На тесте таргета нет
        return test_df

    # ================= Внутренние методы (Mining) ================= #

    def _mine_als_hard_negatives(self, als_generator, user_history: Dict[int, Set[int]], 
                                 candidates_df: pd.DataFrame, n_negs: int, top_k_pool: int) -> pd.DataFrame:
        """
        Ищет айтемы с высоким ALS-скором, которых НЕТ в официальных кандидатах и НЕТ в истории.
        """
        model = als_generator.models["edition"]
        u_map = als_generator.user_maps["edition"]
        i_map = als_generator.item_maps["edition"]
        inv_i_map = {idx: eid for eid, idx in i_map.items()}
        
        # Собираем официальных кандидатов в сеты для быстрого исключения
        cand_sets = candidates_df.groupby("user_id")["edition_id"].apply(set).to_dict()
        
        user_ids = list(cand_sets.keys())
        hard_neg_records = []
        
        item_factors = model.item_factors
        
        for uid in user_ids:
            if uid not in u_map:
                continue
                
            u_idx = u_map[uid]
            u_vec = model.user_factors[u_idx]
            
            # Считаем скоры для всех айтемов
            scores = item_factors.dot(u_vec)
            
            # Получаем топ-K индексов (np.argpartition работает быстрее сортировки всего массива)
            top_k_indices = np.argpartition(scores, -top_k_pool)[-top_k_pool:]
            top_k_indices = top_k_indices[np.argsort(-scores[top_k_indices])]
            
            history_set = user_history.get(uid, set())
            cand_set = cand_sets.get(uid, set())
            
            added = 0
            for idx in top_k_indices:
                eid = inv_i_map.get(idx)
                if eid is None: continue
                
                # Условие: не читал раньше И не находится в кандидатах (outside candidates)
                if eid not in history_set and eid not in cand_set:
                    hard_neg_records.append({"user_id": uid, "edition_id": eid})
                    added += 1
                    if added >= n_negs:
                        break
                        
        return pd.DataFrame(hard_neg_records)

    def _mine_popular_negatives(self, history_df: pd.DataFrame, user_history: Dict[int, Set[int]], 
                                users: np.ndarray, n_negs: int, lookback_days: int, cutoff_ts: pd.Timestamp) -> pd.DataFrame:
        """
        Добавляет в качестве негативов самые популярные айтемы (например, за последние 30 дней),
        которые пользователь еще не видел. Помогает модели отличать мейнстрим от персонализации.
        """
        start_ts = cutoff_ts - pd.Timedelta(days=lookback_days)
        recent_df = history_df[history_df["event_ts"] >= start_ts]
        
        # Считаем глобальную популярность изданий за окно
        pop_counts = recent_df["edition_id"].value_counts()
        top_popular_eids = pop_counts.index.tolist()
        
        pop_records = []
        for uid in users:
            history_set = user_history.get(uid, set())
            added = 0
            for eid in top_popular_eids:
                if eid not in history_set:
                    pop_records.append({"user_id": uid, "edition_id": eid})
                    added += 1
                    if added >= n_negs:
                        break
                        
        return pd.DataFrame(pop_records)

In [ ]:
import numpy as np
import pandas as pd
from typing import Dict, List, Optional
import gc

class RankFeatureGenerator:
    """
    Финальный сборщик признаков (Абсолютная версия).
    Объединяет статику, динамику, CF-скоры, векторные расстояния, 
    а также всю сложную бизнес-логику (аффинити, тренды, геометрию слейта).
    """
    def __init__(self, cfg, const_gen, dyn_gen, als_gen=None, lfm_gen=None):
        self.cfg = cfg
        self.const_gen = const_gen
        self.dyn_gen = dyn_gen
        self.als_gen = als_gen
        self.lfm_gen = lfm_gen

    def transform(self, pairs_df: pd.DataFrame, history_df: pd.DataFrame, data_handler) -> pd.DataFrame:
        print(f"[RankFeatureGenerator] Assembling ALL features for {len(pairs_df)} pairs...")
        df = pairs_df.copy()

        # 1. Базовые фичи (Статика + Динамика)
        df = self.const_gen.transform(df)
        df = self.dyn_gen.transform(df, data_handler)
        df = self._add_historical_links(df, history_df, data_handler)

        # 2. CF-скоры (ALS / LightFM)
        if self.als_gen is not None:
            df = self.als_gen.transform(df, data_handler)
        if self.lfm_gen is not None:
            df = self.lfm_gen.transform(df, data_handler)

        # 3. Текстовые расстояния (SVD + Qwen)
        df = self._add_text_similarities(df, history_df)

        # 4. Ранги и доли (Relative Features)
        if getattr(self.cfg, "ENABLE_CANDIDATE_RELATIVE_FEATURES", True):
            df = self._add_entity_ranks(df)
            df = self._add_read_shares(df)
            df = self._add_book_structure_dynamics(df) # <--- ДОБАВИТЬ ЭТУ СТРОКУ

        # 5. НОВЫЙ БЛОК: Жанровое и издательское аффинити
        if getattr(self.cfg, "ENABLE_GENRE_AFFINITY_FEATURES", True):
            df = self._add_genre_affinity(df, history_df, data_handler)
            
        if getattr(self.cfg, "ENABLE_PUBLISHER_PREF_FEATURES", True):
            df = self._add_publisher_affinity(df, history_df, data_handler)

        # 6. НОВЫЙ БЛОК: Временные тренды и совпадения годов издания
        if getattr(self.cfg, "ENABLE_SHORT_WINDOW_TREND_7_14_VS_30D_FEATURES", True):
            df = self._add_trend_ratios(df)
            
        if getattr(self.cfg, "ENABLE_PUBYEAR_MATCH_FEATURES", True):
            df = self._add_pubyear_match(df, history_df, data_handler)

        if getattr(self.cfg, "ENABLE_PAIR_RECENCY_FEATURES", True):
            df = self._add_pair_recency(df)

        # 7. НОВЫЙ БЛОК: Новизна для юзера
        if getattr(self.cfg, "ENABLE_USER_NOVELTY_FEATURES", True):
            df = self._add_user_novelty(df)

        # 8. НОВЫЙ БЛОК: Геометрия слейта (Slate Consensus)
        if getattr(self.cfg, "ENABLE_SVD_GEOMETRY_FEATURES", True) or getattr(self.cfg, "ENABLE_SLATE_CONSENSUS_FEATURES", True):
            df = self._add_slate_geometry(df)

        # 9. Слейт-нормализация (Z-score внутри списка)
        if getattr(self.cfg, "ENABLE_CANDIDATE_SLATE_NORM_FEATURES", True):
            df = self._apply_slate_normalization(df)

        # 10. Относительные рейтинги
        df = self._add_user_relative_rating_stats(df)

        # 11. Очистка мусора
        drop_cols = getattr(self.cfg, "DROP_ZERO_IMPORTANCE_COLS", [])
        existing_drops = [c for c in drop_cols if c in df.columns]
        if existing_drops:
            df.drop(columns=existing_drops, inplace=True)
            
        print(f"[RankFeatureGenerator] Feature assembly done. Total columns: {len(df.columns)}")
        return df

    # ================= ВНУТРЕННИЕ МЕТОДЫ (СТАРЫЕ + ВОССТАНОВЛЕННЫЕ) ================= #

    def _add_text_similarities(self, df: pd.DataFrame, history_df: pd.DataFrame) -> pd.DataFrame:
        print("  -> Computing text similarities...")
        res = df.copy()
        hist = history_df.copy()
        
        w = np.where(hist["event_type"] == getattr(self.cfg, "READ_LABEL", 3), 
                     getattr(self.cfg, "ALS_READ_WEIGHT", 3.0), 
                     getattr(self.cfg, "ALS_WISHLIST_WEIGHT", 1.0)).astype(np.float32)
        
        if getattr(self.cfg, "ENABLE_RATING_AWARE_INTERACTION_WEIGHTS", False) and getattr(self.cfg, "RATING_AWARE_APPLY_TO_TEXT_USER_EMB", True):
            ratings = hist["rating"].fillna(0.0).to_numpy()
            mask = ratings > 0
            w[mask] = w[mask] * ((ratings[mask] / getattr(self.cfg, "RATING_AWARE_MAX_RATING", 10.0)) ** getattr(self.cfg, "RATING_AWARE_GAMMA", 1.0))
            
        hist["_weight"] = w

        cand_uids = res["user_id"].to_numpy()
        cand_eids = res["edition_id"].to_numpy()

        # --- TF-IDF + SVD ---
        tm = self.const_gen.text_model
        if tm is not None:
            u_embs = {}
            for uid, group in hist.groupby("user_id"):
                group_eids = group["edition_id"].to_numpy()
                weights = group["_weight"].to_numpy()
                rows = [tm.edition_id_to_row[e] for e in group_eids if e in tm.edition_id_to_row]
                valid_w = [weights[i] for i, e in enumerate(group_eids) if e in tm.edition_id_to_row]
                if rows:
                    vecs = tm.item_emb[rows]
                    u_vec = np.average(vecs, axis=0, weights=valid_w)
                    u_norm = np.linalg.norm(u_vec)
                    if u_norm > 1e-8: u_vec = u_vec / u_norm
                    u_embs[uid] = u_vec

            sims = np.zeros(len(res), dtype=np.float32)
            for i in range(len(res)):
                uid, eid = cand_uids[i], cand_eids[i]
                if uid in u_embs and eid in tm.edition_id_to_row:
                    sims[i] = np.dot(u_embs[uid], tm.item_emb_norm[tm.edition_id_to_row[eid]])
            res["text_cosine_similarity"] = sims

            # ВЕРНУЛИ: Разворачиваем SVD векторы в фичи
            if getattr(self.cfg, "ENABLE_TEXT_EMBEDDINGS_AS_FEATURES", True):
                u_svd = np.zeros((len(res), tm.item_emb.shape[1]), dtype=np.float32)
                i_svd = np.zeros((len(res), tm.item_emb.shape[1]), dtype=np.float32)
                for i in range(len(res)):
                    if cand_uids[i] in u_embs: u_svd[i] = u_embs[cand_uids[i]]
                    if cand_eids[i] in tm.edition_id_to_row: i_svd[i] = tm.item_emb_norm[tm.edition_id_to_row[cand_eids[i]]]
                u_df = pd.DataFrame(u_svd, columns=[f"text_svd_user_dim_{k}" for k in range(u_svd.shape[1])])
                i_df = pd.DataFrame(i_svd, columns=[f"text_svd_item_dim_{k}" for k in range(i_svd.shape[1])])
                res = pd.concat([res, u_df, i_df], axis=1)

        # --- Qwen ---
        qw = self.const_gen.qwen_store
        if qw is not None:
            u_qw_embs = {}
            for uid, group in hist.groupby("user_id"):
                group_eids = group["edition_id"].to_numpy()
                weights = group["_weight"].to_numpy()
                rows = [qw.edition_id_to_row[e] for e in group_eids if e in qw.edition_id_to_row]
                valid_w = [weights[i] for i, e in enumerate(group_eids) if e in qw.edition_id_to_row]
                if rows:
                    vecs = qw.edition_emb_norm[rows]
                    u_vec = np.average(vecs, axis=0, weights=valid_w)
                    u_norm = np.linalg.norm(u_vec)
                    if u_norm > 1e-8: u_vec = u_vec / u_norm
                    u_pca = qw.pca_transform(u_vec.reshape(1, -1))[0]
                    u_qw_embs[uid] = (u_vec, u_pca)

            q_sims, q_pca_sims = np.zeros(len(res), dtype=np.float32), np.zeros(len(res), dtype=np.float32)
            for i in range(len(res)):
                uid, eid = cand_uids[i], cand_eids[i]
                if uid in u_qw_embs and eid in qw.edition_id_to_row:
                    u_vec, u_pca = u_qw_embs[uid]
                    i_row = qw.edition_id_to_row[eid]
                    q_sims[i] = np.dot(u_vec, qw.edition_emb_norm[i_row])
                    q_pca_sims[i] = np.dot(u_pca, qw.edition_pca[i_row])
            res["qwen_cos_user_edition"] = q_sims
            res["qwen_pca_dot_user_edition"] = q_pca_sims
            
            # ВЕРНУЛИ: Разворачиваем Qwen PCA векторы в фичи
            if getattr(self.cfg, "ENABLE_QWEN_EMBEDDINGS_AS_FEATURES", True):
                u_pca_mat = np.zeros((len(res), qw.pca_components.shape[0]), dtype=np.float32)
                i_pca_mat = np.zeros((len(res), qw.pca_components.shape[0]), dtype=np.float32)
                for i in range(len(res)):
                    if cand_uids[i] in u_qw_embs: u_pca_mat[i] = u_qw_embs[cand_uids[i]][1] 
                    if cand_eids[i] in qw.edition_id_to_row: i_pca_mat[i] = qw.edition_pca[qw.edition_id_to_row[cand_eids[i]]]
                u_df = pd.DataFrame(u_pca_mat, columns=[f"qwen_pca_user_dim_{k}" for k in range(u_pca_mat.shape[1])])
                i_df = pd.DataFrame(i_pca_mat, columns=[f"qwen_pca_item_dim_{k}" for k in range(i_pca_mat.shape[1])])
                res = pd.concat([res, u_df, i_df], axis=1)

        return res
    
    def _add_historical_links(self, df: pd.DataFrame, history_df: pd.DataFrame, data_handler) -> pd.DataFrame:
        print("  -> Computing historical seen_* features...")
        res = df.copy()
        hist = history_df[history_df["event_type"] == 2].copy() # Только READ для простоты совпадения с оригиналом
        if hist.empty: return res
        
        # seen_edition_read_cnt
        ue = hist.groupby(["user_id", "edition_id"]).size().reset_index(name="seen_edition_read_cnt")
        ue["seen_edition_read"] = 1
        res = res.merge(ue, on=["user_id", "edition_id"], how="left")
        res["seen_edition_read_cnt"] = res["seen_edition_read_cnt"].fillna(0).astype(np.float32)
        res["seen_edition_read"] = res["seen_edition_read"].fillna(0).astype(np.int8)

        # seen_author_read_cnt
        hist = hist.merge(data_handler.editions[["edition_id", "author_id"]], on="edition_id", how="left")
        ua = hist.groupby(["user_id", "author_id"]).size().reset_index(name="seen_author_read_cnt")
        res = res.merge(ua, on=["user_id", "author_id"], how="left")
        res["seen_author_read_cnt"] = res["seen_author_read_cnt"].fillna(0).astype(np.float32)
        
        # seen_publisher_read_cnt
        hist = hist.merge(data_handler.editions[["edition_id", "publisher_id"]], on="edition_id", how="left")
        up = hist.groupby(["user_id", "publisher_id"]).size().reset_index(name="seen_publisher_read_cnt")
        res = res.merge(up, on=["user_id", "publisher_id"], how="left")
        res["seen_publisher_read_cnt"] = res["seen_publisher_read_cnt"].fillna(0).astype(np.float32)

        # seen_language_read_cnt
        hist = hist.merge(data_handler.editions[["edition_id", "language_id"]], on="edition_id", how="left")
        ul = hist.groupby(["user_id", "language_id"]).size().reset_index(name="seen_language_read_cnt")
        res = res.merge(ul, on=["user_id", "language_id"], how="left")
        res["seen_language_read_cnt"] = res["seen_language_read_cnt"].fillna(0).astype(np.float32)

        return res
    
    def _add_book_structure_dynamics(self, df: pd.DataFrame) -> pd.DataFrame:
        print("  -> Computing book structure dynamics...")
        res = df.copy()
        
        if "edition_popularity" in res.columns and "book_popularity" in res.columns:
            res["edition_popularity_share_of_book"] = (res["edition_popularity"] / np.maximum(res["book_popularity"], 1.0)).astype(np.float32)
            
        if "edition_rating_count" in res.columns and "book_rating_count" in res.columns:
            res["edition_read_share_of_book"] = (res["edition_rating_count"] / np.maximum(res["book_rating_count"], 1.0)).astype(np.float32)
            
        if "edition_popularity" in res.columns and "book_id" in res.columns:
            res["edition_popularity_rank_in_book"] = res.groupby("book_id")["edition_popularity"].rank(method="dense", ascending=False).astype(np.float32)
            
        if "edition_rating_count" in res.columns and "book_id" in res.columns:
            res["edition_read_rank_in_book"] = res.groupby("book_id")["edition_rating_count"].rank(method="dense", ascending=False).astype(np.float32)
            
        return res

    def _add_entity_ranks(self, df: pd.DataFrame) -> pd.DataFrame:
        res = df.copy()
        
        # ИСПРАВЛЕНИЕ: Добавлены ранги для edition, text_sim и qwen_sim
        rank_cols = ["edition_popularity", "text_cosine_similarity", "qwen_cos_user_edition"]
        for prefix in ["book", "author", "publisher"]:
            rank_cols.append(f"{prefix}_popularity")
            
        for col in rank_cols:
            if col in res.columns:
                rank_col_name = f"{col}_rank_in_candidates"
                if "popularity" in col and not col.startswith("edition"):
                    rank_col_name = f"{col}_rank_in_candidates"
                elif col == "edition_popularity":
                    rank_col_name = "popularity_rank_in_candidates"
                elif col == "text_cosine_similarity":
                    rank_col_name = "text_sim_rank_in_candidates"
                elif col == "qwen_cos_user_edition":
                    rank_col_name = "qwen_sim_rank_in_candidates"
                    
                res[rank_col_name] = res.groupby("user_id")[col].rank("first", ascending=False).astype(np.float32)
        return res

    def _add_read_shares(self, df: pd.DataFrame) -> pd.DataFrame:
        res = df.copy()
        if "user_activity_total" not in res.columns: return res
        u_act = np.maximum(res["user_activity_total"].to_numpy(dtype=np.float32), 1.0)
        for prefix in ["edition", "book", "author", "publisher"]:
            pop_col = f"{prefix}_popularity"
            if pop_col in res.columns:
                res[f"{prefix}_share_in_user_activity"] = (res[pop_col].to_numpy(dtype=np.float32) / u_act).astype(np.float32)
        return res

    # === ВОССТАНОВЛЕННАЯ ЛОГИКА ===

    def _add_genre_affinity(self, df: pd.DataFrame, history_df: pd.DataFrame, data_handler) -> pd.DataFrame:
        print("  -> Computing genre affinity...")
        res = df.copy()
        
        hist = history_df.merge(data_handler.editions[["edition_id", "genres"]], on="edition_id", how="left")
        user_genres = {}
        for uid, group in hist.groupby("user_id"):
            g_counts = {}
            for gl in group["genres"].dropna():
                if isinstance(gl, list):
                    for g in gl: g_counts[g] = g_counts.get(g, 0) + 1
            user_genres[uid] = g_counts

        # Считаем пересечение
        if "genres" not in res.columns:
            res = res.merge(data_handler.editions[["edition_id", "genres"]], on="edition_id", how="left")
            
        def calc_affinity(row):
            uid, cand_genres = row["user_id"], row["genres"]
            if not isinstance(cand_genres, list) or len(cand_genres) == 0: return 0.0
            u_g = user_genres.get(uid, {})
            overlap_score = sum(u_g.get(g, 0) for g in cand_genres)
            return np.log1p(overlap_score)

        res["genre_affinity_score"] = res.apply(calc_affinity, axis=1).astype(np.float32)
        return res.drop(columns=["genres"], errors="ignore")

    def _add_publisher_affinity(self, df: pd.DataFrame, history_df: pd.DataFrame, data_handler) -> pd.DataFrame:
        print("  -> Computing publisher affinity...")
        res = df.copy()
        hist = history_df.merge(data_handler.editions[["edition_id", "publisher_id"]], on="edition_id", how="left")
        
        pub_counts = hist.groupby(["user_id", "publisher_id"]).size().reset_index(name="user_pub_reads")
        
        if "publisher_id" not in res.columns:
            res = res.merge(data_handler.editions[["edition_id", "publisher_id"]], on="edition_id", how="left")
            
        res = res.merge(pub_counts, on=["user_id", "publisher_id"], how="left")
        res["user_pub_reads"] = res["user_pub_reads"].fillna(0).astype(np.float32)
        
        if "user_activity_total" in res.columns:
            res["publisher_affinity_share"] = (res["user_pub_reads"] / np.maximum(res["user_activity_total"], 1.0)).astype(np.float32)
            
        return res

    def _add_trend_ratios(self, df: pd.DataFrame) -> pd.DataFrame:
        print("  -> Computing trend ratios...")
        res = df.copy()
        for prefix in ["edition", "book", "author"]:
            c7, c14, c30 = f"{prefix}_popularity_7d", f"{prefix}_popularity_14d", f"{prefix}_popularity_30d"
            if all(c in res.columns for c in [c7, c14, c30]):
                res[f"{prefix}_trend_7_vs_30"] = (np.log1p(res[c7]) / np.log1p(res[c30])).astype(np.float32)
                res[f"{prefix}_trend_14_vs_30"] = (np.log1p(res[c14]) / np.log1p(res[c30])).astype(np.float32)
        return res

    def _add_pubyear_match(self, df: pd.DataFrame, history_df: pd.DataFrame, data_handler) -> pd.DataFrame:
        print("  -> Computing pubyear matches...")
        res = df.copy()
        hist = history_df.merge(data_handler.editions[["edition_id", "publication_year"]], on="edition_id", how="left")
        hist["publication_year"] = pd.to_numeric(hist["publication_year"], errors="coerce")
        
        u_years = hist.groupby("user_id")["publication_year"].agg(
            user_pubyear_min="min", user_pubyear_max="max", user_pubyear_median="median"
        ).reset_index()
        
        res = res.merge(u_years, on="user_id", how="left")
        if "publication_year" not in res.columns:
            res = res.merge(data_handler.editions[["edition_id", "publication_year"]], on="edition_id", how="left")
            
        py = pd.to_numeric(res["publication_year"], errors="coerce")
        res["is_pubyear_in_user_range"] = ((py >= res["user_pubyear_min"]) & (py <= res["user_pubyear_max"])).astype(np.int8)
        res["pubyear_diff_from_median"] = np.abs(py - res["user_pubyear_median"]).astype(np.float32)
        
        res.drop(columns=["user_pubyear_min", "user_pubyear_max", "user_pubyear_median"], inplace=True)
        return res

    def _add_pair_recency(self, df: pd.DataFrame) -> pd.DataFrame:
        res = df.copy()
        if "user_days_since_last_event" in res.columns and "edition_days_since_last_event" in res.columns:
            res["pair_recency_diff"] = (res["edition_days_since_last_event"] - res["user_days_since_last_event"]).astype(np.float32)
        return res

    def _add_user_novelty(self, df: pd.DataFrame) -> pd.DataFrame:
        res = df.copy()
        if "edition_popularity" in res.columns and "user_activity_total" in res.columns:
            # Насколько популярна книга глобально vs насколько активен юзер
            res["user_novelty_score"] = (res["edition_popularity"] / np.maximum(res["user_activity_total"], 1.0)).astype(np.float32)
        return res

    def _add_slate_geometry(self, df: pd.DataFrame) -> pd.DataFrame:
        print("  -> Computing slate consensus geometry & RRF...")
        res = df.copy()
        
        # 1. Центроид слейта по Qwen
        if "qwen_cos_user_edition" in res.columns:
            mean_slate_sim = res.groupby("user_id")["qwen_cos_user_edition"].transform("mean")
            res["qwen_dist_to_slate_centroid"] = np.abs(res["qwen_cos_user_edition"] - mean_slate_sim).astype(np.float32)
            
        # 2. RRF (Reciprocal Rank Fusion) и Percentile Ranks
        # Берем только те сигналы, которые сейчас активны в конфиге
        signals = []
        if "qwen_cos_user_edition" in res.columns: signals.append("qwen_cos_user_edition")
        if "text_cosine_similarity" in res.columns: signals.append("text_cosine_similarity")
        if "edition_bayes_avg_rating" in res.columns: signals.append("edition_bayes_avg_rating")
        if "log1p_edition_popularity" in res.columns: signals.append("log1p_edition_popularity")
        
        rrf_score = np.zeros(len(res), dtype=np.float32)
        
        for col in signals:
            # Считаем ранг внутри юзера (ascending=False, т.к. чем больше скор, тем лучше)
            rank = res.groupby("user_id")[col].rank(method="min", ascending=False)
            max_rank = res.groupby("user_id")[col].transform("count")
            
            # Percentile Rank (от 0 до 1, где 1 - лучший в слейте)
            res[f"{col}_pct_rank"] = (1.0 - (rank / max_rank)).astype(np.float32)
            
            # RRF Score (k=60 стандарт консенсуса)
            rrf_score += (1.0 / (60.0 + rank)).astype(np.float32)
            
        if signals:
            res["slate_consensus_rrf_score"] = rrf_score
            
        return res

    # ================= НОРМАЛИЗАЦИЯ И ПОСТ-ОБРАБОТКА ================= #

    def _apply_slate_normalization(self, df: pd.DataFrame) -> pd.DataFrame:
        print("  -> Applying slate normalizations...")
        res = df.copy()
        
        cols_to_norm = [
            "log1p_edition_popularity", "log1p_edition_popularity_30d", 
            "edition_days_since_last_event", "text_cosine_similarity", 
            "qwen_cos_user_edition", "edition_bayes_avg_rating", "genre_affinity_score"
        ]
        
        for level in getattr(self.cfg, "CAND_SLATE_NORM_ALS_LEVELS", ("edition",)):
            col = f"als_{level}_score"
            if col in res.columns: cols_to_norm.append(col)

        grouped = res.groupby("user_id")
        clip_val = getattr(self.cfg, "CAND_SLATE_NORM_Z_CLIP", 8.0)
        
        for col in cols_to_norm:
            if col not in res.columns: continue
            mean_series = grouped[col].transform("mean")
            std_series = grouped[col].transform("std").fillna(1.0).replace(0, 1e-6)
            
            z_col = f"z_{col}_user"
            res[z_col] = ((res[col] - mean_series) / std_series).astype(np.float32).clip(-clip_val, clip_val)
            
            if getattr(self.cfg, "CAND_SLATE_NORM_INCLUDE_ABS_DIFF", True):
                res[f"abs_diff_{col}_user"] = np.abs(res[col] - mean_series).astype(np.float32)
                
        return res

    def _add_user_relative_rating_stats(self, df: pd.DataFrame) -> pd.DataFrame:
        res = df.copy()
        if "user_mean_rating" in res.columns and "edition_bayes_avg_rating" in res.columns:
            res["rating_diff_from_user_mean"] = (res["edition_bayes_avg_rating"] - res["user_mean_rating"]).astype(np.float32)
            res["rating_zscore_user_relative"] = (res["rating_diff_from_user_mean"] / res["user_std_rating"].replace(0, 1e-6)).astype(np.float32)
        return res

In [ ]:
import numpy as np
import pandas as pd
import catboost as cb
import lightgbm as lgb
import xgboost as xgb
from typing import Dict, List, Optional
from itertools import product

class MetaRanker:

    """
    Класс для обучения ансамбля градиентных бустингов (CatBoost, LightGBM, XGBoost).
    Реализует YetiRank/LambdaRank, взвешивание групп (пользователей) и блендинг.
    """
    def __init__(self, cfg):
        self.cfg = cfg
        self.models = {}
        self.features = []
        self.cat_cols = []
        self.num_cols = []
        self.blend_weights = self.cfg.BLEND_MANUAL_WEIGHTS.copy()

    def fit(self, train_df: pd.DataFrame, val_df: Optional[pd.DataFrame] = None):
        print("[MetaRanker] Preparing data for ranking models...")
        
        # 1. Сортировка для стабильности qid (mergesort обязателен для XGBoost)
        train_df = train_df.sort_values("user_id", kind="mergesort").reset_index(drop=True)
        if val_df is not None:
            val_df = val_df.sort_values("user_id", kind="mergesort").reset_index(drop=True)

        # 2. Определение списка фичей
        numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
        drop_cols = ["user_id", "edition_id", "target", "event_ts", "event_type", 
                     "rating", "_weight", "days_to_cutoff", "pair_idx"]
        
        self.features = sorted([c for c in numeric_cols if c not in drop_cols])
        
        cat_cols_candidate = [
            "user_gender", "user_top_language_id", "user_top_publisher_id", 
            "author_id", "publisher_id", "language_id", "primary_genre_id"
        ]
        
        self.cat_cols = [c for c in cat_cols_candidate if c in self.features]
        self.num_cols = [c for c in self.features if c not in self.cat_cols]

        print(f"[MetaRanker] Using {len(self.features)} features ({len(self.cat_cols)} categorical).")

        # 3. БЕЗОПАСНАЯ ТИПИЗАЦИЯ
        def set_types(df):
            if df is None: return None
            df = df.copy()
            for c in self.num_cols:
                df[c] = df[c].fillna(0.0).astype(np.float32)
            for c in self.cat_cols:
                df[c] = df[c].fillna(-1).astype(np.int32)
            return df

        train_df = set_types(train_df)
        val_df = set_types(val_df)

        X_train = train_df[self.features]
        y_train = train_df["target"].fillna(0.0).astype(np.float32)
        
        # Размеры групп и веса (для CatBoost и LightGBM)
        group_train_sizes = train_df.groupby("user_id", sort=False).size().to_numpy(dtype=np.int32)
        weight_train = self._compute_group_weights(train_df, group_train_sizes)

        # 4. Обучение CatBoost
        print("[MetaRanker] Training CatBoost...")
        cb_train_pool = cb.Pool(
            data=X_train, label=y_train, 
            group_id=train_df["user_id"], 
            group_weight=weight_train, 
            cat_features=self.cat_cols
        )
        cb_model = cb.CatBoostRanker(**self.cfg.CATBOOST_WARM_FINAL_CONFIG)
        # Если не хотите спама логов при обучении, можно поставить verbose=False
        cb_model.fit(cb_train_pool, verbose=100)
        self.models["catboost"] = cb_model

        # 5. Обучение LightGBM
        if getattr(self.cfg, "USE_LGBM_BLEND_MODEL", True) :
            print("[MetaRanker] Training LightGBM...")
            lgb_model = lgb.LGBMRanker(**self.cfg.LGBM_BLEND_FINAL_CONFIG)
            lgb_model.fit(
                X_train, y_train, 
                group=group_train_sizes, 
                sample_weight=weight_train,
                categorical_feature=self.cat_cols
            )
            self.models["lightgbm"] = lgb_model

        # 6. Обучение XGBoost (С ИСПРАВЛЕНИЯМИ C++ КРАША)
        if getattr(self.cfg, "USE_XGB_BLEND_MODEL", True):
            print("[MetaRanker] Training XGBoost...")
            xgb_model = xgb.XGBRanker(**self.cfg.XGB_BLEND_FINAL_CONFIG)
            
            fit_kwargs = {}
            if val_df is not None:
                fit_kwargs["eval_set"] = [(val_df[self.features], val_df["target"].fillna(0.0).astype(np.float32))]
                # Каст qid валидации к int32 для безопасности памяти
                fit_kwargs["eval_qid"] = [val_df["user_id"].astype(np.int32).to_numpy()]
                fit_kwargs["verbose"] = False

            # ВАЖНО: 
            # 1. qid кастуется в int32
            # 2. sample_weight УБРАН, так как именно он крашит DMatrix в задаче rank:ndcg
            xgb_model.fit(
                X_train, y_train, 
                qid=train_df["user_id"].astype(np.int32).to_numpy(), 
                **fit_kwargs
            )
            self.models["xgboost"] = xgb_model

        return self
        
    def predict(self, test_df: pd.DataFrame) -> pd.DataFrame:
        print("[MetaRanker] Predicting on test data...")
        res = test_df.copy()
        
        # Восстанавливаем недостающие фичи (если на тесте чего-то нет)
        for c in self.features:
            if c not in res.columns:
                res[c] = 0.0
                
        # Строгая типизация
        for c in self.num_cols:
            res[c] = res[c].fillna(0.0).astype(np.float32)
        for c in self.cat_cols:
            res[c] = res[c].fillna(-1).astype(np.int32)
            
        X_test = res[self.features]
        
        # Предсказания от всех моделей
        for m_name, model in self.models.items():
            print(f"  -> Inferencing {m_name}...")
            res[f"score_{m_name}"] = model.predict(X_test)
            
        # Блендинг
        if getattr(self.cfg, "ENABLE_BLEND", True) and len(self.models) > 1:
            print("  -> Blending scores...")
            blend_score = np.zeros(len(res), dtype=np.float32)
            
            for m_name in self.models.keys():
                w = self.blend_weights.get(m_name, 1.0)
                score_col = f"score_{m_name}"
                
                # Z-score нормализация перед суммированием весов
                if getattr(self.cfg, "BLEND_PER_USER_ZSCORE", True):
                    norm_score = self._per_user_zscore_grouped(res, score_col)
                else:
                    norm_score = res[score_col].to_numpy()
                    
                blend_score += w * norm_score
                
            res["score_blended"] = blend_score

        return res

    def _compute_group_weights(self, df: pd.DataFrame, group_sizes: np.ndarray) -> np.ndarray:
        alpha = getattr(self.cfg, "PER_USER_TRAINING_WEIGHT_ALPHA", 1.0)
        cmin = getattr(self.cfg, "PER_USER_TRAINING_WEIGHT_CLIP_MIN", None)
        cmax = getattr(self.cfg, "PER_USER_TRAINING_WEIGHT_CLIP_MAX", None)
        
        group_weights = (1.0 / group_sizes) ** alpha
        if cmin is not None or cmax is not None:
            group_weights = np.clip(group_weights, a_min=cmin, a_max=cmax)
            
        user_ids = df["user_id"].unique()
        weight_map = dict(zip(user_ids, group_weights))
        return df["user_id"].map(weight_map).to_numpy(dtype=np.float32)

    def _per_user_zscore_grouped(self, df: pd.DataFrame, score_col: str) -> np.ndarray:
        g = df.groupby("user_id")[score_col]
        mean = g.transform("mean")
        std = g.transform("std").fillna(1.0).replace(0, 1e-6)
        return ((df[score_col] - mean) / std).to_numpy(dtype=np.float32)

In [ ]:
class GreedyReranker:
    """
    Класс для финального отбора Top-20 кандидатов.
    Реализует удаление прочитанных книг, обеспечение уникальности book_id 
    в выдаче и MMR-реранкинг (Maximal Marginal Relevance) для разнообразия жанров.
    """
    def __init__(self, cfg):
        self.cfg = cfg

    def generate_submissions(self, scored_df: pd.DataFrame, history_df: pd.DataFrame, data_handler):
        print("[GreedyReranker] Starting Post-processing and Submissions Generation...")
        
        # 1. Подготовка истории прочтений (чтобы не рекомендовать то, что уже прочитано)
        # Предполагаем, что READ_LABEL = 2 или 3 (согласно твоему конфигу)
        read_label = getattr(self.cfg, "READ_LABEL", 3)
        read_history = history_df[history_df["event_type"] == read_label].copy()
        
        # Подтягиваем book_id к истории, чтобы знать, какие именно произведения (не только издания) читал юзер
        read_history = read_history.merge(
            data_handler.editions[["edition_id", "book_id"]], 
            on="edition_id", 
            how="left"
        )
        
        user_read_books = read_history.groupby("user_id")["book_id"].apply(set).to_dict()
        user_read_editions = read_history.groupby("user_id")["edition_id"].apply(set).to_dict()

        # 2. Подготовка метаданных для кандидатов (нужно для фильтрации по уникальности книг)
        editions_meta = data_handler.editions[["edition_id", "book_id"]].copy()
        # Присоединяем book_id к скорам, если их там еще нет
        if "book_id" not in scored_df.columns:
            scored_df = scored_df.merge(editions_meta, on="edition_id", how="left")
            
        score_cols = [c for c in scored_df.columns if c.startswith("score_")]
        os.makedirs("submit", exist_ok=True)

        for col in score_cols:
            print(f"[GreedyReranker] Processing {col} to LONG format...")
            
            # Сортируем: сначала по юзеру, затем по скору (убывание), затем по ID (для стабильности)
            df_sorted = scored_df.sort_values(
                ["user_id", col, "edition_id"], 
                ascending=[True, False, True], 
                kind="stable"
            )
            
            final_rows = []
            
            # Группируем по юзеру и выбираем топ-20
            for uid, group in df_sorted.groupby("user_id", sort=False):
                picked_eids = []
                used_books_in_this_sub = set()
                
                # Книги, которые юзер уже читал (из истории)
                already_read_books = user_read_books.get(uid, set())
                already_read_editions = user_read_editions.get(uid, set())
                
                candidates = group.to_dict("records")
                
                # --- ПРОХОД 1: Выбираем уникальные новые книги ---
                for cand in candidates:
                    eid = int(cand["edition_id"])
                    bid = cand["book_id"]
                    # Если book_id нет, используем негативный eid как уникальный маркер
                    book_key = bid if pd.notna(bid) and bid >= 0 else f"no_book_{eid}"
                    
                    # Условие: Не читал это издание И (опционально) не читал эту книгу вообще
                    if self.cfg.POSTPROCESS_EXCLUDE_READ_BOOKS:
                        if book_key in already_read_books or eid in already_read_editions:
                            continue
                    
                    # Условие: Уникальность книги в рамках текущего сабмита (Top-20)
                    if self.cfg.POSTPROCESS_UNIQUE_BOOKS:
                        if book_key in used_books_in_this_sub:
                            continue
                    
                    picked_eids.append(eid)
                    used_books_in_this_sub.add(book_key)
                    
                    if len(picked_eids) >= 20:
                        break
                
                # --- ПРОХОД 2: Если не набрали 20 (фолбэк: дубликаты книг/прочитанное) ---
                if len(picked_eids) < 20:
                    for cand in candidates:
                        eid = int(cand["edition_id"])
                        if eid not in picked_eids:
                            picked_eids.append(eid)
                        if len(picked_eids) >= 20:
                            break

                # Формируем строки для итогового CSV (user_id, edition_id, rank)
                for rank, eid in enumerate(picked_eids[:20], 1):
                    final_rows.append({
                        "user_id": int(uid), 
                        "edition_id": int(eid), 
                        "rank": int(rank)
                    })

            # 3. Сохранение в длинном формате
            sub_df = pd.DataFrame(final_rows)
            
            # Важно: колонки именно в таком порядке
            sub_df = sub_df[["user_id", "edition_id", "rank"]]
            
            model_suffix = f'{col.replace("score_", "")}'
            out_path = f"submit/{model_suffix}_submissionAGI.csv"
            
            # index=False убирает колонку с индексами строк
            sub_df.to_csv(out_path, index=False)
            print(f"[GreedyReranker] Successfully saved: {out_path} ({len(sub_df)} rows)")
            
    def _exclude_read_books(self, df: pd.DataFrame, user_read_eids: Dict[int, Set[int]], 
                            user_read_bids: Dict[int, Set[int]]) -> pd.DataFrame:
        def is_unseen(row):
            uid, eid, bid = row["user_id"], row["edition_id"], row["book_id"]
            if eid in user_read_eids.get(uid, set()): return False
            if bid in user_read_bids.get(uid, set()): return False
            return True
            
        mask = df.apply(is_unseen, axis=1)
        # Не удаляем book_id и genres, они пригодятся на следующих этапах
        return df[mask]

    def _enforce_unique_books(self, df: pd.DataFrame) -> pd.DataFrame:
        # Датафрейм не перемешивается мержем, drop_duplicates работает честно
        return df.drop_duplicates(subset=["user_id", "book_id"], keep="first")

    def _apply_mmr(self, df: pd.DataFrame, score_col: str, lambda_val: float) -> pd.DataFrame:
        def jaccard(g1, g2):
            if not isinstance(g1, list) or not isinstance(g2, list): return 0.0
            s1, s2 = set(g1), set(g2)
            if not s1 or not s2: return 0.0
            return len(s1 & s2) / len(s1 | s2)

        reranked_rows = []
        for uid, group in df.groupby("user_id"):
            candidates = group.to_dict("records")
            selected = []
            
            while candidates and len(selected) < 20: 
                best_idx = -1
                best_mmr_score = -float("inf")
                
                for i, cand in enumerate(candidates):
                    rel_score = cand[score_col]
                    max_sim = 0.0
                    
                    if selected:
                        sims = [jaccard(cand.get("genres", []), sel.get("genres", [])) for sel in selected]
                        max_sim = max(sims) if sims else 0.0
                        
                    mmr_score = (1.0 - lambda_val) * rel_score - lambda_val * max_sim
                    
                    if mmr_score > best_mmr_score:
                        best_mmr_score = mmr_score
                        best_idx = i
                        
                best_cand = candidates.pop(best_idx)
                selected.append(best_cand)
                
            selected.extend(candidates)
            reranked_rows.extend(selected)
            
        return pd.DataFrame(reranked_rows)

In [ ]:
import gc
import pandas as pd
import numpy as np

def run_pipeline():
    print("="*60)
    print("🚀 STARTING CORRECTED RECSYS PIPELINE 🚀")
    print("="*60)
    
    # 1. Инициализация конфигурации и загрузка данных
    cfg = PipelineConfig()
    data_handler = DataHandler(cfg)
    data_handler.load_data()
    
    # Определение временных точек
    max_ts = data_handler.interactions["event_ts"].max()
    # Точка отсечки для обучения: за 30 дней до конца данных
    train_cutoff_ts = (max_ts - pd.Timedelta(days=getattr(cfg, "TRAIN_WINDOW_DAYS", 30))).floor("D")
    
    # Инференс делается на "завтра" относительно последней записи
    inference_ts = max_ts + pd.Timedelta(seconds=1)
    
    print(f"\n[TimeSplit] Global Max TS: {max_ts}")
    print(f"[TimeSplit] Train Cutoff TS: {train_cutoff_ts}")
    print(f"[TimeSplit] Inference (Today) TS: {inference_ts}")

    # =========================================================================
    # СТАДИЯ 1: ПОДГОТОВКА ДАННЫХ ДЛЯ ОБУЧЕНИЯ (TRAIN)
    # =========================================================================
    print("\n--- STAGE 1: FITTING GENERATORS FOR TRAINING ---")
    
    # История строго ДО точки отсечки обучения
    history_train = data_handler.interactions[data_handler.interactions["event_ts"] < train_cutoff_ts].copy()
    
    # 1.1 Статические фичи (не зависят от времени)
    const_gen = ConstantFeatureGenerator(cfg)
    const_gen.fit(data_handler)
    
    # 1.2 Динамические фичи на момент train_cutoff_ts
    dyn_gen = DynamicFeatureGenerator(cfg)
    dyn_gen.fit(history_train, data_handler, train_cutoff_ts)
    
    # 1.3 ALS на истории до train_cutoff_ts
    als_gen = ALSFeatureGenerator(cfg)
    als_gen.fit(history_train, data_handler)
    
    # 1.4 Генерация пар для обучения (Labeling)
    cand_gen = CandidateGenerator(cfg)
    train_pairs = cand_gen.generate_train(
        interactions=data_handler.interactions,
        candidates_df=data_handler.candidates,
        cutoff_ts=train_cutoff_ts,
        als_generator=als_gen
    )
    
    # 1.5 Сборка признаков для обучения
    rank_gen = RankFeatureGenerator(cfg, const_gen, dyn_gen, als_gen)
    print("[Assembly] Building Train Feature Matrix...")
    train_df = rank_gen.transform(train_pairs, history_train, data_handler)
    
    # 1.6 Обучение ранжирующей модели
    print("\n--- STAGE 2: TRAINING META-RANKER ---")
    meta_ranker = MetaRanker(cfg)
    meta_ranker.fit(train_df)
    
    # Очистка памяти после обучения (удаляем тренировочные объекты)
    del train_df, train_pairs, history_train
    gc.collect()

    # =========================================================================
    # СТАДИЯ 2: ПОДГОТОВКА ГЕНЕРАТОРОВ ДЛЯ ИНФЕРЕНСА (REFIT)
    # =========================================================================
    print("\n--- STAGE 3: RE-FITTING GENERATORS FOR INFERENCE (FULL DATA) ---")
    
    # Теперь мы используем ВСЮ историю, чтобы фичи на тесте были актуальными
    full_history = data_handler.interactions.copy()
    
    # 2.1 Обновляем динамические статистики до текущего момента (inference_ts)
    # Это пересчитает популярность за последние 30 дней и recency (дни с последнего события)
    dyn_gen.fit(full_history, data_handler, inference_ts)
    
    # 2.2 Переобучаем ALS на всей истории
    # Чтобы модель знала о последних действиях пользователей за "валидационный" месяц
    if getattr(cfg, "USE_IMPLICIT_ALS_FEATURES", False):
        als_gen.fit(full_history, data_handler)
        
    # 2.3 Создаем новый RankFeatureGenerator с обновленными данными
    # Важно: const_gen не меняется, так как тексты книг статичны
    rank_gen_inference = RankFeatureGenerator(cfg, const_gen, dyn_gen, als_gen)

    # =========================================================================
    # СТАДИЯ 3: ИНФЕРЕНС (PREDICT)
    # =========================================================================
    print("\n--- STAGE 4: INFERENCE ON TEST CANDIDATES ---")
    
    # Тестовые пары из файла candidates.csv
    test_pairs = cand_gen.generate_test(data_handler.candidates)
    
    # Сборка признаков для теста на полной истории
    print("[Assembly] Building Test Feature Matrix (using full history)...")
    test_df = rank_gen_inference.transform(test_pairs, full_history, data_handler)
    
    # Предсказание скоров
    scored_test_df = meta_ranker.predict(test_df)
    
    # Очистка памяти
    del test_df, full_history
    gc.collect()

    # =========================================================================
    # СТАДИЯ 4: ПОСТПРОЦЕССИНГ
    # =========================================================================
    print("\n--- STAGE 5: POST-PROCESSING & SUBMISSION ---")
    reranker = GreedyReranker(cfg)
    # Передаем исходные интеракции для фильтрации уже прочитанного (если нужно)
    reranker.generate_submissions(scored_test_df, data_handler.interactions, data_handler)
    
    print("\n" + "="*60)
    print("✅ PIPELINE FINISHED SUCCESSFULLY ✅")
    print("="*60)

if __name__ == "__main__":
    run_pipeline()

In [ ]:

import pandas as pd
import numpy as np
import os
import random
import warnings
from dataclasses import dataclass, field
from typing import Dict, Optional, List, Set, Tuple
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors
import implicit
from tqdm import tqdm
from catboost import CatBoostRanker, Pool, EFeaturesSelectionAlgorithm, EShapCalcType

# Models
from catboost import CatBoostRanker
import lightgbm as lgb
import xgboost as xgb

from dataclasses import dataclass, field
from typing import Dict, Optional, Any, List

@dataclass
class PipelineConfig:
    # =========================================================================
    # 1. PATHS & FILES (Пути к данным)
    # =========================================================================
    data_path: str = './data'
    submit_path: str = './submit'
    output_path: str = './output'


    save_raw_scores: bool = True
    val_scores_path: str = './ansamb/val_raw_scores.csv'   # CSV формат
    test_scores_path: str = './ansamb/test_raw_scores.csv' # CSV формат

    files: Dict[str, str] = field(default_factory=lambda: {
        'users': 'users.csv',
        'items': 'editions.csv',
        'interactions': 'interactions.csv',
        'authors': 'authors.csv',
        'genres': 'genres.csv',
        'book_genres': 'book_genres.csv',
        'targets': 'targets.csv',
        'candidates': 'candidates.csv'
    })
    
    # Файл для стратегии "from_file" (Hack Strategy)
    #hack_candidates_path: str = "output/smart_hard_negatives_v1.csv"
    hack_candidates_path: str = "submit/candidates.csv"

    ref_rank_file_path: str = "submit/candidates.csv"

    # =========================================================================
    # 2. VALIDATION STRATEGY (Валидация и Окна)
    # =========================================================================
    window_days: int = 119      # Размер окна истории
    target_days: int = 30        # Горизонт предсказания
    step_days: int = 30       # Шаг сдвига


    # 'union' - (было раньше) берем всех из окна И всех из таргета (много мусора)
    # 'intersection' - (то, что ты просил) только те, кто был активен И в прошлом, И в будущем
    # 'window_only' - (строгая симуляция) только те, кого видели в прошлом
    training_user_strategy: str = 'intersection'
    
    # Если None, валидация считается от конца данных. Формат: 'YYYY-MM-DD'
    force_validation_date: Optional[str] = None 
    
    # Делать ли финальный рефит на ВСЕХ данных перед сабмитом?
    refit_on_full_data: bool = True

    # =========================================================================
    # 3. CANDIDATE GENERATION (Генерация кандидатов)
    # =========================================================================
    # Сколько кандидатов брать из каждого источника
    candidates_strategy: Dict[str, int] = field(default_factory=lambda: {
        'als': 0,             # Коллаборативная фильтрация
        'top_pop_window': 0,  # Популярное в окне
        'top_pop_global': 0,  # Популярное за все время
        'wishlist': 0,        # вишлист
        'random': 0,           # Random negative sampling
        'from_file': 200,       # Из внешнего файла
        'als_plus_toppop': 60
    })
    hybrid_strategy_config: Dict[str, Any] = field(default_factory=lambda: {
        'alpha': 0.5,        # Вес персонализации (ALS Z-Score)
        'beta': 0.5,         # Вес популярности (LogPop Z-Score)
        'pool_factor': 30     # Во сколько раз больше кандидатов брать для просеивания (N * 5)
    })

    # Настройки "Умной фильтрации" внешнего файла
    hack_strategy_config: Dict[str, Any] = field(default_factory=lambda: {
        'filter_mode': 'per_user_bottom_k', 
        #'filter_mode': 'none', 
        'quantile': 0.65,
        'fallback_n': 0
    })

    # --- INTERNAL ALS PARAMS (Для генерации кандидатов) ---
    als_factors: int = 64
    als_regularization: float = 0.05
    als_iterations: int = 15
    als_random_state: int = 42
    
    # Веса событий для обучения ALS (Read > Wishlist)
    als_event_weights: Dict[int, float] = field(default_factory=lambda: {
        2: 3.0,  # Read весит в 3 раза больше
        1: 1.0   # Wishlist стандарт
    })

    # =========================================================================
    # 4. FEATURE ENGINEERING (Фичи)
    # =========================================================================
    # --- Text / Content ---
    tfidf_max_features: int = 3000
    svd_components: int = 32
    genre_svd_components: int = 16
    
    # --- Флаги генерации статических фичей ---
    calc_user_stats: bool = True           
    calc_item_stats: bool = True           
    calc_user_author_affinity: bool = True 
    calc_user_genre_affinity: bool = True  
    calc_user_meta: bool = True             # Пол, возраст
    calc_detailed_stats: bool = True        # Кол-во оценок и т.д.
    use_author_reads_metric: bool = True 

    # --- Флаги динамических фичей (DynamicFeatureGenerator) ---
    dyn_features: Dict[str, bool] = field(default_factory=lambda: {
        'split_read_wishlist': True,  # Считать отдельно cnt_read, cnt_wish
        'calc_ratios': True,          # Считать доли
        'calc_deltas': True,          # Дни с последнего события
        'author_split': True,         # Разделять автора на read/wishlist
        'genre_split': True,          # Разделять жанры на read/wishlist
        'rating_details': True,       # Доп. статы по рейтингу
        'calc_time_decay': True,      # Включить генерацию фич с затуханием (tau=7, 30)
        'calc_diffs_and_z': True,  
        'calc_global_recency': True,      # Жива ли книга (глобально)
        'calc_user_pop_zscore': True,     # Z-score популярности под юзера
        'calc_user_year_zscore': True,    # Z-score года издания под юзера
        'calc_rating_confidence': True    
    })
    
    

    # --- Флаги константных фичей (ConstantFeatureGenerator) ---
    const_features: Dict[str, bool] = field(default_factory=lambda: {
        'calc_svd_cosine': True,      # Косинусное расстояние SVD User-Item
        'calc_bayes_mean': True,      # Включить байесовское сглаживание рейтинга
        'calc_genre_cosine': True,
        'calc_loyalty_fractions': True
    })


    rank_features: Dict[str, bool] = field(default_factory=lambda: {
        'enabled': True,                  # Включить генерацию рангов
        'rank_popularity': True,          # Ранг по популярности в батче
        'rank_als_score': True,           # Ранг по скору ALS (если есть)
        'rank_year': True,                # Ранг по новизне
        'rank_tfidf_sim': False,             # Ранг по текстовой похожести
        'rank_qwen_sim': False,        # Rank по косинусу Qwen (User-Item)
        'rank_book_pop': True,        # Rank по популярности Книги (Work)
        'rank_author_pop': True,      # Rank по популярности Автора
        'rank_publisher_pop': True    # Rank по популярности Издателя
    })
    
    embeddings_config: Dict[str, Any] = field(default_factory=lambda: {
        'enabled': True,
        'external_path': "data/edition_qwen3_embedding.feather", # Путь к внешним эмбеддингам
        'use_pca': True,
        'pca_components': 64,
        'calculate_pca_features': True,   
        'calculate_cosines': True,        
        'use_hierarchical': True,         
        'event_weights': {2: 3.0, 1: 1.0} 
    })

    # --- Настройки ALS-фичей (ConstantFeatureGenerator) ---
    als_ensemble_config: Dict[str, Any] = field(default_factory=lambda: {
        # Какие фичи генерировать для КАЖДОГО включенного уровня
        'features_to_calculate': [
                           # Raw Dot Product
            'norms',       # User/Item Euclidean Norms (Confidence)
            'global_z'     # Z-Score относительно среднего скора юзера по всей базе (Global Context)
                  
        ],
        
        # Настройки уровней
        'levels': {
            'edition': {
                'enabled': True,
                'col': 'edition_id',
                'factors': 64,
                'regularization': 0.05,
                'iterations': 15,
                'use_weights': True  # Использовать 3.0/1.0 веса
            },
            'book': {
                'enabled': True,
                'col': 'book_id',     # Агрегация по книге
                'factors': 64,
                'regularization': 0.1,
                'iterations': 15,
                'use_weights': True
            },
            'publisher': {
                'enabled': True,       
                'col': 'publisher_id', # Убедись, что колонка так называется в items.csv
                'factors': 32,         # Издателей обычно не супер много, 32 хватит
                'regularization': 0.1,
                'iterations': 15,
                'use_weights': True
            },
            'author': {
                'enabled': True,       # Включить User-Author ALS
                'col': 'author_id',
                'factors': 32,         # Факторов меньше, так как авторов меньше
                'regularization': 0.1,
                'iterations': 15,
                'use_weights': True
            }
            
        },
        
        # Настройки для "Is Read" (бинарная история)
        'history_check': ['book', 'author', 'genre'], # Для каких уровней считать флаг "читал ранее"
        
        # Общие настройки
        'global_sample_size': 10000,   # Для Global Z-Score
        'random_state': 42
    })


    lightfm_config: Dict[str, Any] = field(default_factory=lambda: {
        'enabled': True,
        'random_state': 42,
        
        # --- Гиперпараметры модели ---
        'dims': 64,             # Размер вектора (можно 128, если памяти хватает)
        'learning_rate': 0.05,
        'loss': 'warp',         # WARP лучше всего для ранжирования
        'epochs': 20,           # Чуть увеличил эпохи, так как фич много
        'num_threads': 4,
        
        # --- Входные фичи (Input Mixture) ---
        'use_content_svd': True,  # Подмешивает SVD (Text для Edition, Genre для Book)
        'use_global_stats': True, # Подмешивает log(pop) и rating (только для Edition)
        
        # --- Уровни агрегации (Levels) ---
        'levels': {
            # 1. Уровень издания (Самый мощный, гибридный)
            'edition': {
                'enabled': True,
                'col': 'edition_id',
                'sample_size_for_z': 10000
            },
            # 2. Уровень книги (Гибридный с жанрами)
            'book': {
                'enabled': True,
                'col': 'book_id',
                'sample_size_for_z': 10000
            },
            # 3. Уровень автора (Pure CF - ловит стиль автора через поведение юзеров)
            'author': {
                'enabled': True,
                'col': 'author_id',
                'sample_size_for_z': 5000  # Авторов меньше, семпл можно меньше
            },
            # 4. Уровень издательства (Pure CF - ловит "формат" издательства)
            'publisher': {
                'enabled': True,
                'col': 'publisher_id',
                'sample_size_for_z': 5000
            }
        },
        
        # --- Выходные фичи (Output Columns) ---
        # Включаем ВСЁ. Для каждого уровня сгенерируется 6 колонок.
        # Итого: 4 уровня * 6 колонок = 24 новые фичи.
        'features_to_calculate': [
            'biases',      # lfm_{level}_u_bias, lfm_{level}_i_bias
            'norms',       # lfm_{level}_u_norm, lfm_{level}_i_norm
            'global_z'     # lfm_{level}_global_z (Z-Score)
        ]
    })


    
    # =========================================================================
    # 5. MODEL CONFIG (CatBoost / LGBM)
    # =========================================================================
    models_config: Dict = field(default_factory=lambda: {
        'catboost_yeti': {
            'type': 'catboost',
            'enabled': True,
            'weight': 0.4,
            'params': {
                'iterations': 10000,
                'learning_rate': 0.01,
                'depth': 12,
                'loss_function': 'YetiRank',
                'eval_metric': 'NDCG:top=20',
                'task_type': 'CPU',
                'random_seed': 42,
                'verbose': 100,
                'early_stopping_rounds': 50
            }
        },

        'lgbm_lambdarank': {
            'type': 'lgbm',
            'enabled': True,
            'weight': 0.3,
            'params': {
                'objective': 'lambdarank',
                'metric': 'ndcg',
                'n_estimators': 10000,
                'learning_rate': 0.01,
                
                'max_depth': 10,
                'min_data_in_leaf': 31,
               
                'random_state': 42,
                'n_jobs': -1,
                'verbose': -1,
                'eval_at': 20
            }
        },
        'xgboost_ranker': {
            'type': 'xgboost',
            'enabled': True,
            'weight': 0.3,
            'params': {
                'objective': 'rank:ndcg',
                'eval_metric': 'ndcg@20',
                'n_estimators': 10000,
                'learning_rate': 0.01,
                'max_depth': 10,
                'random_state': 42,
                'min_child_weight': 1,
                'n_jobs': -1,
                # 'tree_method': 'hist' # Если есть GPU или для ускорения
            }
        }
    })


    feature_selection: Dict[str, Any] = field(default_factory=lambda: {
        'enabled': True,                  # Включить/Выключить
        'num_features_to_select': 220,     # Сколько лучших фичей оставить 
        'steps': 1,                       # Сколько фичей выкидывать за одну итерацию (выше = быстрее)
        'algorithm': 'ByShapValues', # Самый точный метод для ранжирования
        # Параметры для черновой модели (должна быть быстрой)
        'selection_params': {
            'iterations': 1000,            # Меньше итераций для скорости
            'learning_rate': 0.05,         # Чуть выше LR для быстрого схождения
            'depth': 10,
            'loss_function': 'YetiRank',
            'task_type': 'CPU',
            'random_seed': 42,
            
            'allow_writing_files': False
        }
    })

    

    target_scores: Dict[float, float] = field(default_factory=lambda: {
        2: 3.0,  # За чтение даем 3 (можно поменять на 8, 10 и т.д.)
        1: 1,  # За вишлист даем 1
        0: 0.0   # За отсутствие взаимодействия
    })

    # =========================================================================
    # 6. RERANKER (Greedy)
    # =========================================================================
    enable_reranker: bool = False
    auto_tune_reranker: bool = False
    reranker_grid: Dict = field(default_factory=lambda: {'alpha': [1.0], 'beta': [0.5]})
    default_alpha: float = 1.0
    default_beta: float = 0.5

    # =========================================================================
    # 7. SYSTEM
    # =========================================================================
    n_cpu: int = 4
    random_seed: int = 42
    verbose: bool = True

In [ ]:
import pandas as pd
import numpy as np
import implicit
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from typing import Dict, List, Optional, Set, Any
from tqdm import tqdm
import lightfm
from scipy import sparse

class ConstantFeatureGenerator:
    """
    Отвечает за генерацию статических фичей и эмбеддингов.
    Обучается 1 раз на всей доступной истории (до точки валидации).
    
    [Refactored v11]: 
      - Удален легаси код (_fit_als, _add_advanced_als_features).
      - Исправлена логика ALS Ensemble (динамический мердж колонок).
      - Реализована правильная обработка Жанров (Explode) для ALS.
      - Сохранены все SVD и Profile фичи.
    """
    
    def __init__(self, config):
        self.config = config
        
        # Models Containers
        self.als_models = {}      # {'book': model_obj, ...}
        self.als_maps = {}        # {'book': {'u_map':..., 'i_map':...}, ...}
        self.als_stats = {}       # {'book': {internal_idx: (mean, std)}, ...} для Global Z
        
        # SVD Models
        self.tfidf = None
        self.text_svd = None
        self.genre_svd = None
        
        # Caches & Embeddings
        self.text_emb_map = {}
        self.genre_emb_map = {}
        self.user_svd_profiles = {} # {uid: mean_vector}
        
        # History Sets for Flags
        self.history_sets = {}    # {'book': set((uid, bid)), ...}
        self.seen_books_set = set()
        
        # Profile Stats
        self.user_top_lang = {}   
        self.user_top_genres = {} 
        
        # Metadata Cache (для transform)
        self.editions_meta_cached = None
        self.book_genres_map_cached = None


        self.lfm_models = {}        # {level: model} - на всякий случай
        self.lfm_repr = {}          # {level: {'u_vecs':..., 'i_vecs':..., 'u_biases':..., 'i_biases':...}}
        self.lfm_maps = {}          # {level: {'u_map':..., 'i_map':...}}
        self.lfm_stats = {}

        self.lfm_features_cache = {}

        self.qwen_item_embs = {}   # {edition_id: np.array(64)}
        self.qwen_user_embs = {}   # {user_id: np.array(64)}
        self.qwen_author_embs = {} # {author_id: np.array(64)}


        self.user_pub_frac = {}   # { (uid, pub_id): fraction }
        self.user_lang_frac = {}




    def fit(self, interactions: pd.DataFrame, editions_meta: pd.DataFrame, book_genres_map: Dict):
        """
        Главный метод обучения.
        """
        self.editions_meta_cached = editions_meta
        self.book_genres_map_cached = book_genres_map

        print("[Constant] Fitting ALS Ensemble (Levels: Edition, Book, Author, etc.)...")
        # 1. Обучаем ансамбль ALS (исправленная логика)
        self._fit_als_ensemble(interactions, editions_meta, book_genres_map)
        
        # 2. Обучаем текстовые модели (без изменений)
        print("[Constant] Fitting Text & Genre SVD Models...")
        self._fit_text_models(editions_meta)
        self._fit_genre_models(editions_meta, book_genres_map)

        # --- ВСТАВИТЬ В МЕТОД fit() ПОСЛЕ _fit_genre_models ---
        
        if self.config.const_features.get('calc_bayes_mean', False):
            print("[Constant] Calculating Bayesian Mean Ratings...")
            self._calc_bayes_ratings(interactions, editions_meta)

        if self.config.const_features.get('calc_genre_cosine', False):
            print("[Constant] Building User Genre Profiles (Cosine)...")
            self._build_user_genre_profiles(interactions)
        
        # 3. Строим профили пользователя по контенту (SVD Cosine) (без изменений)
        if self.config.const_features.get('calc_svd_cosine', False):
            print("[Constant] Building User Content Profiles (SVD)...")
            self._build_user_svd_profiles(interactions)

        # 4. Строим эвристические профили (Язык, Топ Жанры) (без изменений)
        print("[Constant] Fitting content profiles (Lang, Genre, History)...")
        self._fit_content_profiles(interactions, editions_meta, book_genres_map)


        if self.config.lightfm_config['enabled']:
            print("[Constant] Fitting LightFM Ensemble (Hybrid)...")
            # Важно: передаем глобальную историю взаимодействий
            self._fit_lightfm_ensemble(interactions, editions_meta)

        

        emb_cfg = self.config.embeddings_config
        if emb_cfg['enabled']:
            print("[Constant] Fitting External Qwen/LLM Embeddings...")
            self._fit_external_embeddings(interactions, editions_meta)
            
        return self
    

    def transform(self, candidates: pd.DataFrame) -> pd.DataFrame:
        """
        Применяет обученные модели к кандидатам.
        """
        result = candidates.copy()
        cfg = self.config.als_ensemble_config
        
        # 0. Обогащаем метаданными (Dynamic Enrich)
        # Нам нужно убедиться, что в кандидатах есть ВСЕ колонки, требуемые включенными уровнями ALS.
        # Например, если включен 'publisher', а в кандидатах его нет -> надо подтянуть.
        
        cols_needed = set()
        # Базовые колонки, которые нужны почти всегда
        cols_needed.add('book_id') 
        cols_needed.add('author_id')
        cols_needed.add('language_id') # для profile features
        
        # 1. Проверяем ALS конфиг
        als_cfg = self.config.als_ensemble_config
        for level_name, level_cfg in als_cfg['levels'].items():
            if level_cfg['enabled']:
                cols_needed.add(level_cfg['col'])

        # 2. Проверяем LightFM конфиг (ДОБАВЛЕНО)
        lfm_cfg = self.config.lightfm_config
        if lfm_cfg['enabled']:
            for level_name, level_cfg in lfm_cfg['levels'].items():
                if level_cfg['enabled']:
                    cols_needed.add(level_cfg['col'])
        
        # Оставляем только те, которых нет в result, но есть в метаданных
        cols_to_merge = [c for c in cols_needed 
                         if c not in result.columns 
                         and c in self.editions_meta_cached.columns]
        
        if cols_to_merge:
            # Используем left join, чтобы не потерять кандидатов
            result = result.merge(self.editions_meta_cached[['edition_id'] + cols_to_merge], 
                                  on='edition_id', how='left')
        # 1. Цикл по уровням ALS (Extract Features)
        for level_name, level_cfg in cfg['levels'].items():
            if not level_cfg['enabled']: continue
            if level_name not in self.als_models: continue
            
            col_id = level_cfg['col']
            
            # Для Жанров (1-to-Many) мы не можем просто посчитать Score (User * Item), 
            # так как у Item много жанров. 
            # Поэтому для жанров считаем ТОЛЬКО флаг истории (читал ли этот жанр).
            # (Если нужно полноценное ALS-расстояние до жанров книги, нужна агрегация векторов, 
            #  но пока оставим классический скоринг только для 1-to-1 сущностей).
            
            if level_name == 'genre':
                # Только история
                if level_name in cfg['history_check']:
                     # Для жанров нужно проверить вхождение ЛЮБОГО жанра книги в историю юзера
                     # Это сложно векторизовать быстро здесь. 
                     # Упрощение: используем логику topg_match_frac или пропускаем флаг здесь, 
                     # так как _fit_content_profiles уже считает похожее.
                     pass
            else:
                # Стандартный 1-to-1 скоринг (Edition, Book, Author, Publisher)
                prefix = f"{level_name}_als_"
                result = self._extract_level_features(result, level_name, col_id, prefix)
                
                # Генерируем "Is Read" флаг
                if level_name in cfg['history_check']:
                    result = self._add_history_flag(result, level_name, col_id)   

        # 2. Text & Genre Embeddings (SVD)
        result = self._add_embeddings(result, self.text_emb_map, prefix="txt_svd_")
        result = self._add_embeddings(result, self.genre_emb_map, prefix="genre_svd_")
        
        # 3. SVD Cosine Similarity
        if self.config.const_features.get('calc_svd_cosine', False) and hasattr(self, 'user_svd_profiles'):
            # Векторизованный расчет через apply (можно оптимизировать, но логику сохраняем)
            def get_sim(row):
                u_vec = self.user_svd_profiles.get(row['user_id'])
                i_vec = self.text_emb_map.get(row['edition_id'])
                if u_vec is None or i_vec is None: return 0.0
                i_norm = np.linalg.norm(i_vec)
                if i_norm == 0: return 0.0
                return np.dot(u_vec, i_vec) / i_norm
            
            result['svd_cosine_sim'] = result.apply(get_sim, axis=1).astype(np.float32)

        # 4. Profile Features (Lang, TopGenres)
        result = self._add_profile_features(result, self.book_genres_map_cached)



        # --- ВСТАВИТЬ В МЕТОД transform() ПЕРЕД return ---
        
        # 1. Apply Bayesian Rating
        if self.config.const_features.get('calc_bayes_mean', False) and hasattr(self, 'bayes_rating_map'):
            # Мапим, если нет значения - ставим глобальное среднее
            result['edition_bayes_avg_rating'] = result['edition_id'].map(self.bayes_rating_map).fillna(self.global_rating_mean)

        # 2. Apply Genre Cosine Similarity
        if self.config.const_features.get('calc_genre_cosine', False) and hasattr(self, 'user_genre_profiles'):
            # Векторизованный расчет через apply (можно оптимизировать через matrix mult, но пока так надежнее)
            def get_genre_sim(row):
                u_vec = self.user_genre_profiles.get(row['user_id'])
                i_vec = self.genre_emb_map.get(row['edition_id'])
                if u_vec is None or i_vec is None: return 0.0
                # i_vec уже должен быть нормирован при создании, но проверим dot
                return float(np.dot(u_vec, i_vec)) # предполагаем что векторы нормированы
            
            result['genre_svd_cosine'] = result.apply(get_genre_sim, axis=1).astype(np.float32)

        

        # [NEW] LightFM Features Transform
        lfm_cfg = self.config.lightfm_config
        if lfm_cfg['enabled']:
            for level_name, level_cfg in lfm_cfg['levels'].items():
                if not level_cfg['enabled']: continue
                if level_name not in self.lfm_repr: continue
                
                col_id = level_cfg['col']
                prefix = f"lfm_{level_name}_"
                
                # Вызываем метод извлечения (векторизованный)
                result = self._extract_lfm_features(result, level_name, col_id, prefix)
        


        if self.config.embeddings_config['enabled']:
            result = self._add_external_embedding_features(result)
        
        return result
    


    def _fit_external_embeddings(self, interactions, editions_meta):
        import os
        from sklearn.decomposition import PCA
        cfg = self.config.embeddings_config
        
        if not os.path.exists(cfg['external_path']):
            print(f"[Warning] External embeddings not found at {cfg['external_path']}")
            return

        # 1. Загрузка из Feather
        print(f"  -> Loading wide-format Qwen embeddings from {cfg['external_path']}...")
        df_emb = pd.read_feather(cfg['external_path'])
        
        # 2. Извлечение матрицы векторов
        emb_cols = [c for c in df_emb.columns if c.startswith('emb_')]
        if not emb_cols:
            # Fallback если имена другие (на всякий случай)
            emb_cols = [c for c in df_emb.columns if c != 'edition_id']
            
        print(f"  -> Found {len(emb_cols)} embedding dimensions.")
        
        raw_embs = df_emb[emb_cols].values.astype(np.float32)
        edition_ids = df_emb['edition_id'].values
        
        # 3. PCA сжатие
        if cfg['use_pca']:
            print(f"  -> Fitting PCA: {raw_embs.shape[1]} -> {cfg['pca_components']} components...")
            pca = PCA(n_components=cfg['pca_components'], random_state=42)
            compressed_embs = pca.fit_transform(raw_embs)
        else:
            compressed_embs = raw_embs

        # 4. Нормализация и создание словаря
        norms = np.linalg.norm(compressed_embs, axis=1, keepdims=True)
        norm_embs = compressed_embs / (norms + 1e-9)
        self.qwen_item_embs = {eid: vec for eid, vec in zip(edition_ids, norm_embs)}

        # 5. Построение профилей пользователей (ИСПРАВЛЕННАЯ ЛОГИКА)
        print("  -> Building User Qwen Profiles...")
        weights = cfg.get('event_weights', {2: 3.0, 1: 1.0})
        hist = interactions[['user_id', 'edition_id', 'event_type']].copy()
        hist['w'] = hist['event_type'].map(weights).fillna(1.0)
        
        # Используем tqdm для прогресса
        for uid, group in tqdm(hist.groupby('user_id'), desc="User LLM Profiles"):
            # [FIX] Фильтруем группу целиком через isin.
            # Это гарантирует, что мы берем только те строки взаимодействий, для книг которых есть эмбеддинги.
            valid_group = group[group['edition_id'].isin(self.qwen_item_embs)]
            
            if not valid_group.empty:
                # Извлекаем векторы строго в порядке следования строк в valid_group
                vecs = np.array([self.qwen_item_embs[eid] for eid in valid_group['edition_id']])
                
                # Извлекаем веса строго из тех же строк
                ws = valid_group['w'].values
                
                # Теперь len(vecs) == len(ws) гарантированно
                user_vec = np.average(vecs, axis=0, weights=ws)
                self.qwen_user_embs[uid] = user_vec / (np.linalg.norm(user_vec) + 1e-9)

        # 6. Построение профилей авторов
        if cfg.get('use_hierarchical', False):
            print("  -> Building Author Qwen Profiles...")
            author_data = editions_meta[['edition_id', 'author_id']].dropna()
            # Фильтруем сразу те, что есть в эмбеддингах
            author_data = author_data[author_data['edition_id'].isin(self.qwen_item_embs)]
            
            for aid, group in author_data.groupby('author_id'):
                vecs = np.array([self.qwen_item_embs[eid] for eid in group['edition_id']])
                if len(vecs) > 0:
                    auth_vec = np.mean(vecs, axis=0)
                    self.qwen_author_embs[aid] = auth_vec / (np.linalg.norm(auth_vec) + 1e-9)


    

    def _add_external_embedding_features(self, df):
        """
        Добавляет фичи на основе внешних эмбеддингов (Qwen/LLM):
        1. Cosine Similarity (User-Item, User-Author)
        2. PCA Components (координаты векторов)
        """
        cfg = self.config.embeddings_config
        dim = cfg['pca_components']
        
        # 1. Векторизованный Lookup (быстрое извлечение векторов из словарей)
        # Используем списковые включения, так как они быстрее apply для словарей
        
        # Векторы Юзеров
        u_vecs = np.array([
            self.qwen_user_embs.get(uid, np.zeros(dim, dtype=np.float32)) 
            for uid in df['user_id']
        ])
        
        # Векторы Книг
        i_vecs = np.array([
            self.qwen_item_embs.get(eid, np.zeros(dim, dtype=np.float32)) 
            for eid in df['edition_id']
        ])
        
        # 2. Расчет Косинусной близости (User-Item)
        if cfg['calculate_cosines']:
            # Т.к. векторы уже нормированы в fit, косинус = скалярное произведение
            df['qwen_cos_user_edition'] = np.sum(u_vecs * i_vecs, axis=1).astype(np.float32)

        # 3. Расчет Косинусной близости (User-Author) - Иерархия
        if cfg['use_hierarchical']:
            # Проверяем наличие author_id (обычно есть после enrich в начале transform)
            if 'author_id' in df.columns:
                a_vecs = np.array([
                    self.qwen_author_embs.get(aid, np.zeros(dim, dtype=np.float32)) 
                    for aid in df['author_id']
                ])
                df['qwen_cos_user_author'] = np.sum(u_vecs * a_vecs, axis=1).astype(np.float32)
            else:
                # Если вдруг колонки нет, заполняем нулями
                df['qwen_cos_user_author'] = 0.0

        # 4. Добавление самих координат (PCA features)
        # Это важно для CatBoost, чтобы он сам искал нелинейные связи
        if cfg['calculate_pca_features']:
            # Координаты книги
            item_cols = [f'qwen_pca_edition_{i}' for i in range(dim)]
            df_item_pca = pd.DataFrame(i_vecs, columns=item_cols, index=df.index)
            
            # Координаты юзера
            user_cols = [f'qwen_pca_user_{i}' for i in range(dim)]
            df_user_pca = pd.DataFrame(u_vecs, columns=user_cols, index=df.index)
            
            # Конкатенация (быстрее, чем присваивание по одной колонке)
            df = pd.concat([df, df_item_pca, df_user_pca], axis=1)

        return df




                    
    def _precalc_lightfm_data(self, interactions, editions_meta):
        """
        [NEW] Единый метод подготовки фичей для всех уровней (Edition, Book, Author, Publisher).
        Считает статистику и агрегирует векторы SVD.
        """
        print("[Constant] Pre-calculating Stats & Content Vectors for LightFM...")
        self.lfm_features_cache = {}
        
        # 1. Подготовка векторов в удобном формате DataFrame
        # Text Embeddings (Edition Level)
        text_vec_df = pd.DataFrame.from_dict(self.text_emb_map, orient='index')
        text_vec_df.index.name = 'edition_id'
        
        # Genre Embeddings (Book Level)
        genre_vec_df = pd.DataFrame.from_dict(self.genre_emb_map, orient='index')
        genre_vec_df.index.name = 'book_id'
        
        # Объединенная мета для агрегации (Edition -> Book -> Author -> Publisher)
        # Оставляем только нужные ID
        meta_cols = ['edition_id', 'book_id', 'author_id', 'publisher_id']
        meta_cols = [c for c in meta_cols if c in editions_meta.columns]
        full_meta = editions_meta[meta_cols].copy()

        # Мержим векторы к мете для агрегации
        # (Осторожно с памятью: если падает, можно делать циклом, но пока так быстрее)
        if not text_vec_df.empty:
            full_meta = full_meta.merge(text_vec_df, on='edition_id', how='left')
        
        # 2. Цикл по уровням
        levels_to_proc = ['edition', 'book', 'author', 'publisher']
        
        for level in levels_to_proc:
            col_id = f"{level}_id"
            if col_id not in full_meta.columns:
                continue
                
            # --- A. Interaction Stats (Log Vol, Avg Rate, Audience) ---
            # Группируем взаимодействия по этому ID
            if col_id == 'edition_id':
                level_interactions = interactions
            else:
                # Подтягиваем ID уровня к интеракциям
                level_interactions = interactions.merge(editions_meta[['edition_id', col_id]], on='edition_id', how='inner')

            stats = level_interactions.groupby(col_id).agg(
                cnt=('user_id', 'count'),
                rating_sum=('rating', 'sum'),
                rating_cnt=('rating', lambda x: (x>0).sum()),
                users=('user_id', 'nunique')
            )
            
            # Derived Stats
            stats['log_vol'] = np.log1p(stats['cnt'])
            stats['avg_rating'] = stats['rating_sum'] / (stats['rating_cnt'] + 1e-9)
            stats['log_aud'] = np.log1p(stats['users'])
            
            # Normalization (MinMax roughly)
            for col in ['log_vol', 'avg_rating', 'log_aud']:
                max_val = stats[col].max()
                if max_val > 0:
                    stats[col] /= max_val
            
            stats_dict = stats[['log_vol', 'avg_rating', 'log_aud']].to_dict('index')
            
            # --- B. Content Aggregation (Propagation) ---
            # Агрегируем текстовые векторы (они уже вмержены в full_meta)
            # Для Edition это будет Identity (один к одному), для Author - среднее
            
            content_dict = {}
            # Берем колонки векторов (они интовые 0..31 в names после merge, если создавались из np.array)
            # Или строковые, зависит от того, как pandas создал DF. 
            # text_vec_df имеет колонки 0, 1, ... 
            
            vec_cols = [c for c in full_meta.columns if isinstance(c, int) or (isinstance(c, str) and c.isdigit())]
            
            if vec_cols:
                # Группируем и усредняем
                agg_vecs = full_meta.groupby(col_id)[vec_cols].mean()
                # Превращаем в dict {id: np.array}
                # Это может быть медленно, но надежно
                content_dict = {idx: row.values.astype(np.float32) for idx, row in agg_vecs.iterrows()}
            
            # Отдельно Жанры (если это не Book и не Edition, где мы уже знаем маппинг)
            # Для Author/Publisher усредняем жанры книг
            if level in ['author', 'publisher'] and not genre_vec_df.empty:
                # Нам нужно map: author -> [books] -> [genre_vecs]
                # Проще: merge book_id к full_meta (уже есть), потом merge genre_vec
                # Но чтобы не раздувать full_meta, сделаем агрегацию отдельно
                
                # author -> book map
                ab_map = full_meta[[col_id, 'book_id']].drop_duplicates()
                ab_merged = ab_map.merge(genre_vec_df, on='book_id', how='inner')
                
                g_cols = genre_vec_df.columns
                g_agg = ab_merged.groupby(col_id)[g_cols].mean()
                
                # Добавляем к content_dict (конкатенация или сложение? Конкатенация лучше)
                for idx, row in g_agg.iterrows():
                    g_vec = row.values.astype(np.float32)
                    if idx in content_dict:
                        content_dict[idx] = np.concatenate([content_dict[idx], g_vec])
                    else:
                        content_dict[idx] = g_vec

            # Save to cache
            self.lfm_features_cache[level] = {
                'stats': stats_dict,
                'content': content_dict
            }


    def _extract_lfm_features(self, df, level_name, col_id, prefix):
        """
        Быстрый расчет фичей через предрасчитанные векторы.
        Score = Dot(U, I) + Bu + Bi
        """
        reprs = self.lfm_repr[level_name]
        maps = self.lfm_maps[level_name]
        stats = self.lfm_stats.get(level_name, {})
        
        active_feats = self.config.lightfm_config['features_to_calculate']
        
        # 1. Map Indices
        u_map = maps['u_map']
        i_map = maps['i_map']
        
        u_idxs = df['user_id'].map(u_map).fillna(-1).astype(int)
        i_idxs = df[col_id].map(i_map).fillna(-1).astype(int)
        
        valid_mask = (u_idxs >= 0) & (i_idxs >= 0)
        n_rows = len(df)
        
        # Arrays for results
        scores = np.zeros(n_rows, dtype=np.float32)
        u_biases = np.zeros(n_rows, dtype=np.float32)
        i_biases = np.zeros(n_rows, dtype=np.float32)
        u_norms = np.zeros(n_rows, dtype=np.float32)
        i_norms = np.zeros(n_rows, dtype=np.float32)
        
        if valid_mask.any():
            v_u = u_idxs[valid_mask].values
            v_i = i_idxs[valid_mask].values
            
            # Fetch Vectors
            uv = reprs['u_vecs'][v_u] # (N, D)
            iv = reprs['i_vecs'][v_i] # (N, D)
            ub = reprs['u_biases'][v_u]
            ib = reprs['i_biases'][v_i]
            
            # Calc Score
            dot = np.sum(uv * iv, axis=1)
            scores[valid_mask] = dot + ub + ib
            
            if 'biases' in active_feats:
                u_biases[valid_mask] = ub
                i_biases[valid_mask] = ib
                
            if 'norms' in active_feats:
                u_norms[valid_mask] = np.linalg.norm(uv, axis=1)
                i_norms[valid_mask] = np.linalg.norm(iv, axis=1)

        # Write to DF
        if 'score' in active_feats:
            df[f'{prefix}score'] = scores
            
        if 'biases' in active_feats:
            df[f'{prefix}u_bias'] = u_biases
            df[f'{prefix}i_bias'] = i_biases
            
        if 'norms' in active_feats:
            df[f'{prefix}u_norm'] = u_norms
            df[f'{prefix}i_norm'] = i_norms
            
        # Global Z-Score (Normalization)
        if 'global_z' in active_feats and stats:
            def get_stat(uidx):
                return stats.get(uidx, (0.0, 1.0))
            
            stats_list = [get_stat(x) if x >= 0 else (0.0, 1.0) for x in u_idxs]
            stats_arr = np.array(stats_list, dtype=np.float32)
            
            means = stats_arr[:, 0]
            stds = stats_arr[:, 1]
            
            # Z = (Score - Mean) / Std
            df[f'{prefix}global_z'] = (scores - means) / stds
            df.loc[~valid_mask, f'{prefix}global_z'] = 0.0

        return df
    


    # =========================================================================
    # [NEW] LIGHTFM ENSEMBLE LOGIC
    # =========================================================================

    def _fit_lightfm_ensemble(self, interactions, editions_meta):
        import lightfm
        from scipy import sparse
        
        cfg = self.config.lightfm_config
        levels = cfg['levels']
        
        # [NEW] 1. Запуск прекалькуляции (Stats + Content Propagation)
        self._precalc_lightfm_data(interactions, editions_meta)
        
        # Подготовка весов (как раньше)
        weights_map = self.config.als_event_weights
        base_df = interactions[['user_id', 'edition_id', 'event_type']].copy()
        base_df['weight'] = base_df['event_type'].map(weights_map).fillna(1.0)
        
        # --- Loop over levels ---
        for level_name, level_cfg in levels.items():
            if not level_cfg['enabled']: continue
            
            print(f"  -> Training LightFM Level: {level_name.upper()}...")
            col_target = level_cfg['col']
            
            # Агрегация взаимодействий (как раньше)
            if col_target not in base_df.columns:
                temp_df = base_df.merge(editions_meta[['edition_id', col_target]], on='edition_id', how='inner')
            else:
                temp_df = base_df
                
            grouped = temp_df.groupby(['user_id', col_target])['weight'].max().reset_index()
            
            # Mappings
            users = grouped['user_id'].astype('category')
            items = grouped[col_target].astype('category')
            
            u_map = {uid: idx for idx, uid in enumerate(users.cat.categories)}
            i_map = {iid: idx for idx, iid in enumerate(items.cat.categories)}
            
            self.lfm_maps[level_name] = {'u_map': u_map, 'i_map': i_map}
            
            # Interaction Matrix
            row = users.cat.codes.values.copy()
            col = items.cat.codes.values.copy()
            data = grouped['weight'].values.astype(np.float32).copy()
            
            interactions_csr = sparse.coo_matrix((data, (row, col)), shape=(len(u_map), len(i_map)))
            
            # [NEW] 3. Build Item Features (Updated)
            # Теперь передаем только level_name, остальное берется из кэша
            item_features_csr = self._build_lightfm_features(
                level_name, i_map, 
                use_svd=cfg['use_content_svd'], 
                use_stats=cfg['use_global_stats']
            )
            
            # 4. Train (как раньше)
            model = lightfm.LightFM(
                no_components=cfg['dims'],
                learning_rate=cfg['learning_rate'],
                loss=cfg['loss'],
                random_state=cfg['random_state']
            )
            
            model.fit(
                interactions_csr, 
                item_features=item_features_csr, 
                epochs=cfg['epochs'], 
                num_threads=cfg['num_threads'],
                verbose=False
            )
            
            # 5. Extract Representations
            u_biases, u_vecs = model.get_user_representations()
            i_biases, i_vecs = model.get_item_representations(features=item_features_csr)
            
            self.lfm_repr[level_name] = {
                'u_vecs': u_vecs.astype(np.float32),
                'u_biases': u_biases.astype(np.float32),
                'i_vecs': i_vecs.astype(np.float32),
                'i_biases': i_biases.astype(np.float32)
            }
            
            # 6. Global Stats Precalc (для Z-score)
            if 'global_z' in cfg['features_to_calculate']:
                self.lfm_stats[level_name] = self._precalc_lfm_stats(
                    u_vecs, u_biases, i_vecs, i_biases, 
                    sample_size=level_cfg['sample_size_for_z']
                )

    

    
    def _build_lightfm_features(self, level_name, i_map, use_svd, use_stats):
        """
        [NEW] Собирает матрицу: [Identity | Stats | Content]
        Использует self.lfm_features_cache.
        """
        from scipy import sparse
        
        n_items = len(i_map)
        feature_blocks = []
        
        # 1. Identity Block (Always required for CF)
        feature_blocks.append(sparse.identity(n_items, format='csr', dtype=np.float32))
        
        # Получаем данные из кэша
        cached_data = self.lfm_features_cache.get(level_name, {})
        stats_map = cached_data.get('stats', {})
        content_map = cached_data.get('content', {})
        
        # Инвертируем мап для порядка
        idx_to_real = {v: k for k, v in i_map.items()}
        sorted_real_ids = [idx_to_real[i] for i in range(n_items)]
        
        # 2. Stats Features
        if use_stats and stats_map:
            # [log_vol, avg_rating, log_aud] -> 3 колонки
            mat = np.zeros((n_items, 3), dtype=np.float32)
            for i, real_id in enumerate(sorted_real_ids):
                if real_id in stats_map:
                    s = stats_map[real_id]
                    mat[i, 0] = s.get('log_vol', 0)
                    mat[i, 1] = s.get('avg_rating', 0)
                    mat[i, 2] = s.get('log_aud', 0)
            feature_blocks.append(sparse.csr_matrix(mat))
            
        # 3. Content Features (Propagated SVD)
        if use_svd and content_map:
            # Определяем размерность вектора по первому попавшемуся элементу
            sample_key = next(iter(content_map))
            dims = len(content_map[sample_key])
            
            mat = np.zeros((n_items, dims), dtype=np.float32)
            for i, real_id in enumerate(sorted_real_ids):
                if real_id in content_map:
                    mat[i] = content_map[real_id]
            feature_blocks.append(sparse.csr_matrix(mat))
            
        # Combine
        full_matrix = sparse.hstack(feature_blocks, format='csr')
        return full_matrix

    def _precalc_lfm_stats(self, u_vecs, u_biases, i_vecs, i_biases, sample_size):
        """
        Считает Mean/Std скоров для каждого юзера для Z-Score.
        Score = U*I + Bu + Bi
        """
        n_items = i_vecs.shape[0]
        n_users = u_vecs.shape[0]
        
        # Sample items
        if n_items > sample_size:
            idxs = np.random.choice(n_items, sample_size, replace=False)
        else:
            idxs = np.arange(n_items)
            
        samp_i_vecs = i_vecs[idxs]   # (S, D)
        samp_i_biases = i_biases[idxs] # (S,)
        
        stats = {}
        chunk_size = 1000
        
        for start in range(0, n_users, chunk_size):
            end = min(start + chunk_size, n_users)
            
            u_chunk = u_vecs[start:end]    # (C, D)
            u_b_chunk = u_biases[start:end] # (C,)
            
            # Dot: (C, D) @ (D, S) -> (C, S)
            dots = u_chunk.dot(samp_i_vecs.T)
            
            # Add biases: Score = Dot + Ub + Ib
            # Broadcasting: (C, S) + (C, 1) + (1, S)
            scores = dots + u_b_chunk[:, None] + samp_i_biases[None, :]
            
            means = scores.mean(axis=1)
            stds = scores.std(axis=1) + 1e-9
            
            for i in range(len(means)):
                stats[start + i] = (means[i], stds[i])
                
        return stats

    # =========================================================================
    # CORE LOGIC: ALS ENSEMBLE TRAINING
    # =========================================================================
    
    def _fit_als_ensemble(self, interactions, editions_meta, book_genres_map):
        cfg = self.config.als_ensemble_config
        levels = cfg['levels']
        
        # 1. Подготовка базового датасета с весами
        weights_map = self.config.als_event_weights
        base_df = interactions[['user_id', 'edition_id', 'event_type']].copy()
        base_df['weight'] = base_df['event_type'].map(weights_map).fillna(1.0)
        
        # 2. Определяем, какие колонки нужны для мерджа
        # Проходимся по всем включенным уровням (кроме жанров, они отдельно)
        cols_to_fetch = set(['book_id', 'author_id']) # всегда нужны базово
        
        for lname, lcfg in levels.items():
            if lcfg['enabled'] and lname != 'genre':
                cols_to_fetch.add(lcfg['col'])
        
        # === FIX: Исключаем edition_id из списка фичей, так как он будет ключом мерджа ===
        # Если этого не сделать, будет ['edition_id', 'edition_id', ...], что вызовет ошибку
        if 'edition_id' in cols_to_fetch:
            cols_to_fetch.remove('edition_id')
        
        # Оставляем только те, что реально есть в метаданных
        cols_to_fetch = [c for c in cols_to_fetch if c in editions_meta.columns]
        
        # 3. Создаем Rich DF (1-to-1 mapping)
        # Теперь дубликатов в колонках выборки нет
        rich_df = base_df.merge(
            editions_meta[['edition_id'] + cols_to_fetch], 
            on='edition_id', 
            how='inner'
        )
        
        # 4. Цикл обучения
        for level_name, level_cfg in levels.items():
            if not level_cfg['enabled']:
                continue
                
            print(f"  -> Training ALS Level: {level_name.upper()}...")
            col_target = level_cfg['col']
            
            # --- A. Data Aggregation ---
            if level_name == 'genre':
                # ЛОГИКА ДЛЯ 1-to-MANY (Explode)
                # 1. Берем уникальные пары (User, Book) с макс весом
                # Важно: book_id должен быть в rich_df (мы его добавили в cols_to_fetch)
                if 'book_id' not in rich_df.columns:
                     print("     [Warning] book_id missing for Genre ALS. Skipping.")
                     continue

                user_book_weights = rich_df.groupby(['user_id', 'book_id'])['weight'].max().reset_index()
                
                # 2. Разворачиваем жанры
                bg_items = []
                for bid, g_list in book_genres_map.items():
                    for gid in g_list:
                        bg_items.append((bid, gid))
                
                if not bg_items:
                    print("     [Warning] No genre data map found. Skipping Genre ALS.")
                    continue
                    
                bg_df = pd.DataFrame(bg_items, columns=['book_id', 'genre_id'])
                
                # 3. Merge & GroupBy User-Genre
                genre_interactions = user_book_weights.merge(bg_df, on='book_id', how='inner')
                grouped = genre_interactions.groupby(['user_id', 'genre_id'])['weight'].max().reset_index()
                item_col_data = grouped['genre_id']
                
            else:
                # ЛОГИКА ДЛЯ 1-to-1 (Edition, Book, Author, Publisher)
                if col_target not in rich_df.columns:
                    # Если таргет это edition_id, он есть точно, иначе проверим
                    if col_target == 'edition_id':
                         pass
                    else:
                        print(f"     [Warning] Column {col_target} not found. Skipping {level_name}.")
                        continue
                
                grouped = rich_df.groupby(['user_id', col_target])['weight'].max().reset_index()
                item_col_data = grouped[col_target]

            # --- B. Create Matrices ---
            users = grouped['user_id'].astype('category')
            items = item_col_data.astype('category')
            
            # Маппинги: Internal Index <-> Real ID
            u_map = {idx: uid for idx, uid in enumerate(users.cat.categories)} # idx -> real
            i_map = {idx: iid for idx, iid in enumerate(items.cat.categories)} # idx -> real
            
            # Инвертированные для поиска (Real -> idx)
            u_map_inv = {uid: idx for idx, uid in u_map.items()}
            i_map_inv = {iid: idx for idx, iid in i_map.items()}
            
            self.als_maps[level_name] = {'u_map': u_map_inv, 'i_map': i_map_inv}
            
            # CSR
            row = users.cat.codes
            col = items.cat.codes
            data = grouped['weight'].values.astype(np.float32)
            sparse_matrix = csr_matrix((data, (row, col)), shape=(len(u_map), len(i_map)))
            
            # --- C. Fit Model ---
            model = implicit.als.AlternatingLeastSquares(
                factors=level_cfg['factors'],
                regularization=level_cfg['regularization'],
                iterations=level_cfg['iterations'],
                random_state=cfg.get('random_state', 42)
            )
            model.fit(sparse_matrix)
            self.als_models[level_name] = model
            
            # --- D. Precalc Stats (Global Z) ---
            if 'global_z' in cfg['features_to_calculate']:
                self.als_stats[level_name] = self._precalc_stats_for_model(
                    model, cfg['global_sample_size']
                )

            # --- E. Build History Set (Is Read) ---
            reads = grouped[grouped['weight'] >= 2.0]
            self.history_sets[level_name] = set(zip(reads['user_id'], reads[col_target]))

    

    def _calc_bayes_ratings(self, interactions, editions_meta):
        """
        Считает байесовский средний рейтинг для каждой книги.
        Formula: (v / (v + C)) * R + (C / (v + C)) * m
        """
        # Берем только реальные оценки
        rated = interactions[interactions['rating'] > 0]
        
        # Глобальное среднее (m)
        global_mean = rated['rating'].mean()
        
        # Агрегация по книгам
        item_stats = rated.groupby('edition_id').agg(
            v=('rating', 'count'),
            R=('rating', 'mean')
        )
        
        # Константа C (например, 25-й перцентиль количества оценок, чтобы не сглаживать хиты)
        C = item_stats['v'].quantile(0.25)
        
        # Расчет
        item_stats['edition_bayes_avg_rating'] = (
            (item_stats['v'] / (item_stats['v'] + C)) * item_stats['R'] +
            (C / (item_stats['v'] + C)) * global_mean
        )
        
        # Сохраняем маппинг
        self.bayes_rating_map = item_stats['edition_bayes_avg_rating'].to_dict()
        self.global_rating_mean = global_mean # Нужно для fillna

    def _build_user_genre_profiles(self, interactions):
        """
        Строит векторные профили юзеров на основе SVD-векторов жанров книг, которые они читали.
        Аналог _build_user_svd_profiles, но для жанров.
        """
        reads = interactions[interactions['event_type'] == 2]
        user_items = reads.groupby('user_id')['edition_id'].apply(list).to_dict()
        
        self.user_genre_profiles = {}
        
        # self.genre_emb_map должен быть уже заполнен в _fit_genre_models
        if not self.genre_emb_map:
            print("[Warning] Genre Embeddings not found. Skipping user genre profiles.")
            return

        for uid, items in tqdm(user_items.items(), desc="Building User Genre Profiles"):
            # Собираем векторы всех прочитанных книг
            vecs = [self.genre_emb_map[eid] for eid in items if eid in self.genre_emb_map]
            
            if vecs:
                mean_vec = np.mean(vecs, axis=0)
                # L2 Normalization для косинусного расстояния
                norm = np.linalg.norm(mean_vec)
                if norm > 0: mean_vec /= norm
                self.user_genre_profiles[uid] = mean_vec

    def _precalc_stats_for_model(self, model, sample_size):
        """
        Считает Mean/Std скоров для каждого юзера (по внутренним индексам).
        Возвращает dict: {internal_user_idx: (mean, std)}
        """
        n_items = model.item_factors.shape[0]
        if n_items > sample_size:
            sample_idxs = np.random.choice(n_items, sample_size, replace=False)
        else:
            sample_idxs = np.arange(n_items)
            
        sample_items = model.item_factors[sample_idxs] # (Sample, F)
        stats = {}
        
        # Чанками по юзерам
        chunk_size = 2000
        n_users = model.user_factors.shape[0]
        
        for start in range(0, n_users, chunk_size):
            end = min(start + chunk_size, n_users)
            u_chunk = model.user_factors[start:end] # (Chunk, F)
            
            # Dot: (Chunk, F) @ (F, Sample).T -> (Chunk, Sample)
            scores = u_chunk.dot(sample_items.T)
            
            means = scores.mean(axis=1)
            stds = scores.std(axis=1) + 1e-9
            
            # Сохраняем по внутреннему индексу (i + start)
            for i in range(len(means)):
                stats[start + i] = (means[i], stds[i])
                 
        return stats

    def _extract_level_features(self, df, level_name, col_id, prefix):
        """
        Универсальный расчет фичей для уровня.
        Raw Score, Norms, Global Z, Local Z.
        """
        model = self.als_models[level_name]
        maps = self.als_maps[level_name]
        u_map = maps['u_map'] # real -> idx
        i_map = maps['i_map'] # real -> idx
        stats = self.als_stats.get(level_name, {}) # internal_idx -> (mean, std)
        
        active_feats = self.config.als_ensemble_config['features_to_calculate']
        missing_val = 0.0
        
        # 1. Map Indices (Fast map)
        u_idxs = df['user_id'].map(u_map).fillna(-1).astype(int)
        i_idxs = df[col_id].map(i_map).fillna(-1).astype(int)
        
        # 2. Valid Mask (знаем и юзера, и объект)
        valid_mask = (u_idxs >= 0) & (i_idxs >= 0)
        n_rows = len(df)
        
        # Init Arrays
        raw_scores = np.full(n_rows, missing_val, dtype=np.float32)
        u_norms = np.zeros(n_rows, dtype=np.float32)
        i_norms = np.zeros(n_rows, dtype=np.float32)
        
        # 3. Vector Ops
        if valid_mask.any():
            v_u = u_idxs[valid_mask].values
            v_i = i_idxs[valid_mask].values
            
            u_vecs = model.user_factors[v_u]
            i_vecs = model.item_factors[v_i]
            
            # Dot Product
            raw_scores[valid_mask] = np.sum(u_vecs * i_vecs, axis=1)
            
            if 'norms' in active_feats:
                u_norms[valid_mask] = np.linalg.norm(u_vecs, axis=1)
                i_norms[valid_mask] = np.linalg.norm(i_vecs, axis=1)

        # 4. Save Features
        if 'raw' in active_feats:
            df[f'{prefix}score'] = raw_scores
        
        if 'norms' in active_feats:
            df[f'{prefix}u_norm'] = u_norms
            df[f'{prefix}i_norm'] = i_norms
            
        # 5. Global Z-Score
        if 'global_z' in active_feats and stats:
            # Stats хранятся по internal index, поэтому лукапим по u_idxs
            # Функция для map
            def get_stat(uidx):
                return stats.get(uidx, (0.0, 1.0))
            
            # Векторизуем через список (map быстрее apply для dict lookup)
            stats_list = [get_stat(x) if x >= 0 else (0.0, 1.0) for x in u_idxs]
            stats_arr = np.array(stats_list, dtype=np.float32)
            
            means = stats_arr[:, 0]
            stds = stats_arr[:, 1]
            
            df[f'{prefix}global_z'] = (raw_scores - means) / stds
            # Зануляем там, где данных не было
            df.loc[~valid_mask, f'{prefix}global_z'] = 0.0

        # 6. Local Z-Score (Contextual)
        if 'local_z' in active_feats:
            df['_tmp_raw'] = raw_scores
            grouped = df.groupby('user_id')['_tmp_raw']
            z_vals = grouped.transform(lambda x: (x - x.mean()) / (x.std() + 1e-9)).fillna(0)
            df[f'{prefix}local_z'] = z_vals
            df.drop(columns=['_tmp_raw'], inplace=True)
            
        return df

    def _add_history_flag(self, df, level_name, col_id):
        hist_set = self.history_sets.get(level_name, set())
        pairs = list(zip(df['user_id'], df[col_id]))
        df[f'is_{level_name}_read'] = [1 if p in hist_set else 0 for p in pairs]
        return df

    # =========================================================================
    # PRESERVED FUNCTIONALITY: SVD, TEXT, PROFILES
    # =========================================================================
    
    def _fit_text_models(self, editions):
        texts = (editions['title'].fillna('') + ' ' + editions['description'].fillna('')).astype(str)
        
        self.tfidf = TfidfVectorizer(max_features=self.config.tfidf_max_features, stop_words='english')
        tfidf_matrix = self.tfidf.fit_transform(texts)
        
        self.text_svd = TruncatedSVD(n_components=self.config.svd_components, random_state=self.config.random_seed)
        vecs = self.text_svd.fit_transform(tfidf_matrix)
        
        self.text_emb_map = {eid: vec for eid, vec in zip(editions['edition_id'], vecs)}

    def _fit_genre_models(self, editions, book_genres_map):
        unique_books = editions['book_id'].unique()
        book_to_idx = {bid: i for i, bid in enumerate(unique_books)}
        
        rows, cols, data = [], [], []
        for bid in unique_books:
            if bid in book_genres_map:
                for gid in book_genres_map[bid]:
                    rows.append(book_to_idx[bid])
                    cols.append(gid)
                    data.append(1)
        
        if not rows: return
        
        max_genre_id = max(cols) + 1
        sparse_genres = csr_matrix((data, (rows, cols)), shape=(len(unique_books), max_genre_id))
        
        self.genre_svd = TruncatedSVD(n_components=self.config.genre_svd_components, random_state=self.config.random_seed)
        g_vecs = self.genre_svd.fit_transform(sparse_genres)
        
        book_vec_map = {bid: vec for bid, vec in zip(unique_books, g_vecs)}
        self.genre_emb_map = {}
        for _, row in editions[['edition_id', 'book_id']].iterrows():
            if row['book_id'] in book_vec_map:
                self.genre_emb_map[row['edition_id']] = book_vec_map[row['book_id']]

    def _build_user_svd_profiles(self, interactions):
        reads = interactions[interactions['event_type'] == 2]
        user_items = reads.groupby('user_id')['edition_id'].apply(list).to_dict()
        
        self.user_svd_profiles = {}
        for uid, items in tqdm(user_items.items(), desc="User Profiles"):
            vecs = [self.text_emb_map[eid] for eid in items if eid in self.text_emb_map]
            if vecs:
                mean_vec = np.mean(vecs, axis=0)
                norm = np.linalg.norm(mean_vec)
                if norm > 0: mean_vec /= norm
                self.user_svd_profiles[uid] = mean_vec

    def _fit_content_profiles(self, interactions, editions_meta, book_genres_map):
        """
        [FIXED & FULL] Рассчитывает профили юзера (Топ-жанры, Топ-язык, Loyalty).
        Безопасен к отсутствию колонок publisher_id/language_id.
        Полностью сохраняет логику работы с genres_map.
        """
        # 1. Подготовка данных: берем только позитив (лайк/чтение)
        pos_interactions = interactions[interactions['event_type'].isin([1, 2])].copy()
        
        cols_to_fetch = ['edition_id', 'book_id'] # Обязательные
        
        # Ищем издателя (publisher_id ИЛИ publisher)
        pub_col = 'publisher_id' if 'publisher_id' in editions_meta.columns else ('publisher' if 'publisher' in editions_meta.columns else None)
        if pub_col: cols_to_fetch.append(pub_col)
            
        # Ищем язык (language_id ИЛИ language)
        lang_col = 'language_id' if 'language_id' in editions_meta.columns else ('language' if 'language' in editions_meta.columns else None)
        if lang_col: cols_to_fetch.append(lang_col)
            
        # 2. Безопасный Merge (берем только то, что нашли)
        merged = pos_interactions.merge(editions_meta[cols_to_fetch], on='edition_id', how='inner')
        
        # 3. Локальное переименование (чтобы дальше код работал с publisher_id как обычно)
        rename_map = {}
        if pub_col and pub_col != 'publisher_id': rename_map[pub_col] = 'publisher_id'
        if lang_col and lang_col != 'language_id': rename_map[lang_col] = 'language_id'
        
        if rename_map:
            merged.rename(columns=rename_map, inplace=True)

        has_lang = 'language_id' in editions_meta.columns
        has_pub = 'publisher_id' in editions_meta.columns

        # ---------------------------------------------------------
        # A. Логика Top Language (Сохраняем функционал)
        # ---------------------------------------------------------
        if has_lang:
            # Находим моду (самый частый язык) для каждого юзера
            # Используем pd.Series.mode, берем первый элемент
            # Оптимизация: groupby + agg
            self.user_top_lang = merged.groupby('user_id')['language_id'].agg(
                lambda x: x.mode().iloc[0] if not x.mode().empty else -1
            ).to_dict()
        else:
            self.user_top_lang = {}

        # ---------------------------------------------------------
        # B. Логика Top Genres (Восстанавливаем полную логику)
        # ---------------------------------------------------------
        # Нам нужно развернуть списки жанров из book_genres_map для каждой книги юзера
        # book_genres_map: {book_id: [g1, g2, ...]}
        
        # 1. Получаем список книг для каждого юзера
        user_books = merged[['user_id', 'book_id']].drop_duplicates()
        
        # 2. Аккумулируем жанры (через словарь для скорости, т.к. explode может быть медленным)
        from collections import Counter, defaultdict
        user_genre_counts = defaultdict(Counter)
        
        # Преобразуем map в быстрый доступ (если это не dict)
        genres_lookup = book_genres_map
        
        # Итерируемся и считаем (это быстрее чем explode на миллионах строк)
        # Но для вектора Python медленный, используем Pandas explode если данных не терабайты.
        # Вариант Pandas (надежнее для поддержки):
        
        # Мапим жанры на книги в датафрейме
        # Примечание: map вернет список для каждой строки
        user_books['genres'] = user_books['book_id'].map(genres_lookup)
        
        # Удаляем пустые (где нет жанров)
        user_books = user_books.dropna(subset=['genres'])
        
        if not user_books.empty:
            # Разворачиваем списки: одна строка = одна пара (user, genre)
            ug_exploded = user_books.explode('genres')
            
            # Считаем топы: groupby user -> value_counts -> head(3) -> index
            # Но нам нужен просто "Топ-1 жанр" или список? 
            # Обычно для фичи 'genre_match' нужен set топ жанров или самый топ.
            # Сохраним топ-3 жанра как set для быстрого поиска пересечений
            
            top_genres_df = (
                ug_exploded.groupby(['user_id', 'genres'])
                .size()
                .reset_index(name='cnt')
                .sort_values(['user_id', 'cnt'], ascending=[True, False])
                .groupby('user_id')
                .head(5) # Берем топ-5 любимых жанров
            )
            
            # Собираем в словарь {user_id: {g1, g2, g3}}
            self.user_top_genres = (
                top_genres_df.groupby('user_id')['genres']
                .apply(set)
                .to_dict()
            )
        else:
            self.user_top_genres = {}

        # ---------------------------------------------------------
        # C. Логика Loyalty Fractions (Новая фича)
        # ---------------------------------------------------------
        if self.config.const_features.get('calc_loyalty_fractions', False):
            # Общее кол-во взаимодействий юзера (знаменатель)
            user_counts = merged.groupby('user_id').size()
            
            # Publisher Fractions
            if has_pub:
                pub_counts = merged.groupby(['user_id', 'publisher_id']).size()
                # Делим count(u, p) на count(u)
                self.user_pub_frac = pub_counts.div(user_counts, level='user_id').to_dict()
            else:
                self.user_pub_frac = {}
            
            # Language Fractions
            if has_lang:
                lang_counts = merged.groupby(['user_id', 'language_id']).size()
                self.user_lang_frac = lang_counts.div(user_counts, level='user_id').to_dict()
            else:
                self.user_lang_frac = {}
        else:
            self.user_pub_frac = {}
            self.user_lang_frac = {}

        # ---------------------------------------------------------
        # D. Логика Seen Books (Для фильтрации)
        # ---------------------------------------------------------
        # Здесь берем только тип 2 (Read), чтобы не рекомендовать прочитанное
        reads = interactions[interactions['event_type'] == 2].merge(
            editions_meta[['edition_id', 'book_id']], on='edition_id', how='inner'
        )
        self.seen_books_set = set(zip(reads['user_id'], reads['book_id']))
        
        return self

    def _add_profile_features(self, df, book_genres_map):
        df['user_top_lang'] = df['user_id'].map(self.user_top_lang).fillna(-1).astype(int)
        df['language_id'] = df['language_id'].fillna(-2).astype(int)
        df['lang_match'] = (df['language_id'] == df['user_top_lang']).astype(int)
        df.loc[df['user_top_lang'] == -1, 'lang_match'] = 0

        pairs = list(zip(df['user_id'], df['book_id']))
        df['is_book_read'] = [1 if p in self.seen_books_set else 0 for p in pairs]

        def calc_match(row):
            bid = row['book_id']
            uid = row['user_id']
            b_genres = book_genres_map.get(bid, [])
            if not b_genres: return 0.0
            u_genres = self.user_top_genres.get(uid)
            if not u_genres: return 0.0
            match_cnt = sum(1 for g in b_genres if g in u_genres)
            return match_cnt / len(b_genres)

        df['topg_match_frac'] = df.apply(calc_match, axis=1).astype(np.float32)

        if self.config.const_features.get('calc_loyalty_fractions', False):
            # Векторизованный lookup через map (Tuple lookup в pandas медленный, используем MultiIndex map или apply)
            # Быстрый способ:
            
            # 1. Publisher Frac
            # Если publisher_id нет, подтягиваем (обычно уже есть)
            if 'publisher_id' in df.columns:
                # Создаем серию ключей для маппинга
                keys = list(zip(df['user_id'], df['publisher_id']))
                df['user_pub_loyalty'] = [self.user_pub_frac.get(k, 0.0) for k in keys]
                
            # 2. Language Frac
            if 'language_id' in df.columns:
                keys_lang = list(zip(df['user_id'], df['language_id']))
                df['user_lang_loyalty'] = [self.user_lang_frac.get(k, 0.0) for k in keys_lang]
        return df

    def _add_embeddings(self, df, emb_map, prefix):
        if not emb_map: return df
        sample_vec = next(iter(emb_map.values()))
        dim = len(sample_vec)
        emb_matrix = np.array([emb_map.get(eid, np.zeros(dim)) for eid in df['edition_id']])
        cols = [f"{prefix}{i}" for i in range(dim)]
        emb_df = pd.DataFrame(emb_matrix, columns=cols, index=df.index)
        return pd.concat([df, emb_df], axis=1)

In [ ]:
class DynamicFeatureGenerator:
    """
    Генерирует фичи на лету на основе СРЕЗА взаимодействий (Window Slice).
    Stateless (не хранит модели), считает агрегаты здесь и сейчас.
    ОБНОВЛЕНИЕ v10.1: Добавлена логика User-Genre Affinity.
    """
    
    def __init__(self, config: PipelineConfig):
        self.config = config
        
    def transform(self, interactions_slice: pd.DataFrame, candidates: pd.DataFrame, 
                  editions_meta: pd.DataFrame, book_genres_map: Dict, 
                  users_meta: Optional[pd.DataFrame] = None) -> pd.DataFrame: # <--- ДОБАВЛЕН users_meta
        """
        interactions_slice: Данные только за последние X дней (Local Context)
        candidates: Пары (user, edition)
        users_meta: (NEW) Датафрейм с пользователями (sex, age)
        """
        result = candidates.copy()
        
        # 0. Предварительная проверка: нужен book_id в кандидатах
        cols_to_add = []
        if 'book_id' not in result.columns and 'book_id' in editions_meta.columns:
            cols_to_add.append('book_id')
        if 'author_id' not in result.columns and 'author_id' in editions_meta.columns:
            cols_to_add.append('author_id')
            
        if cols_to_add:
            result = result.merge(editions_meta[['edition_id'] + cols_to_add], on='edition_id', how='left')
        
        # --- 1. Base Window Stats ---
        result = self._add_window_stats(result, interactions_slice)
        
        # --- 2. Static Metadata ---
        result = self._add_static_meta(result, editions_meta)
        
        # --- 2.1 User Meta (NEW v7 Logic) ---
        if self.config.calc_user_meta and users_meta is not None:
            result = self._add_user_meta(result, users_meta)

        # --- 3. Derived Features ---
        if 'user_window_events' in result.columns:
            result['is_cold_user'] = (result['user_window_events'] < 5).astype(int)
        
        # --- 4. User-Author Affinity ---
        if self.config.calc_user_author_affinity:
            result = self._add_author_affinity(result, interactions_slice, editions_meta)
            
        # --- 5. User-Genre Affinity ---
        if self.config.calc_user_genre_affinity:
            result = self._add_user_genre_affinity(result, interactions_slice, editions_meta, book_genres_map)

        result = self._add_parent_level_stats(result, interactions_slice, editions_meta)


        # 1. Time Decay Features
        if self.config.dyn_features.get('calc_time_decay', False):
            result = self._add_time_decay_stats(result, interactions_slice)
            
        # 2. Interaction Diffs & Z-scores
        # Важно: вызываем это ПОСЛЕ того, как ConstantGen добавил bayes_rating (в оркестраторе)
        if self.config.dyn_features.get('calc_diffs_and_z', False):
            result = self._add_interaction_diffs(result, interactions_slice)
        
        if self.config.dyn_features.get('calc_global_recency', False):
            result = self._add_global_recency(result, interactions_slice)
            
        return result

    
    def _add_static_meta(self, df, meta):
        # Добавляем год, возраст и производные
        features = ['publication_year', 'age_restriction', 'pages']
        # Берем только те колонки, которых еще нет в df
        cols_to_merge = [c for c in features if c in meta.columns and c not in df.columns]
        
        if cols_to_merge:
            df = df.merge(meta[['edition_id'] + cols_to_merge], on='edition_id', how='left')
        
        # Derived
        if 'publication_year' in df.columns:
            df['year_scaled'] = (df['publication_year'] - 2000) / 20
            df['year_scaled'] = df['year_scaled'].fillna(0)

        if self.config.dyn_features.get('calc_rating_confidence', False):
            # Пытаемся найти колонку с кол-вом оценок. 
            # Если её нет в editions_meta, это поле будет 0. 
            # (Предполагается, что вы подготовили editions_meta с агрегатами заранее)
            col_cnt = 'rating_cnt' if 'rating_cnt' in meta.columns else None
            
            if col_cnt:
                # Мержим, если еще нет
                if col_cnt not in df.columns:
                    df = df.merge(meta[['edition_id', col_cnt]], on='edition_id', how='left')
                
                # Считаем log1p (Confidence)
                df['edition_rating_conf'] = np.log1p(df[col_cnt].fillna(0))
                # Удаляем сырой каунт, если не нужен
                # df.drop(columns=[col_cnt], inplace=True)
            else:
                # Если в мете нет, можно попробовать посчитать через slice_df (но это будет локальный)
                df['edition_rating_conf'] = 0.0
            
            
        return df
    
    def _add_time_decay_stats(self, candidates, slice_df):
        """
        Считает экспоненциально затухающие счетчики (Tau=7, 30).
        Дает больше веса недавним событиям.
        """
        # 1. Подготовка весов в слайсе
        max_date = slice_df['event_ts'].max()
        # Дни с момента события: (Now - EventTime)
        slice_df = slice_df.copy() # Safe copy
        slice_df['days_diff'] = (max_date - slice_df['event_ts']).dt.days
        
        # Веса: exp(-days / tau)
        slice_df['w_tau7'] = np.exp(-slice_df['days_diff'] / 7.0)
        slice_df['w_tau30'] = np.exp(-slice_df['days_diff'] / 30.0)
        
        # 2. Агрегация по Item (Edition)
        # Считаем сумму весов (это и есть "decayed count")
        item_decay = slice_df.groupby('edition_id').agg({
            'w_tau7': 'sum',
            'w_tau30': 'sum'
        }).reset_index().rename(columns={
            'w_tau7': 'edition_decay_tau7',
            'w_tau30': 'edition_decay_tau30'
        })
        
        # 3. Агрегация по Author (если есть author_id в слайсе)
        author_decay = None
        if 'author_id' in slice_df.columns:
            author_decay = slice_df.groupby('author_id').agg({
                'w_tau7': 'sum',
                'w_tau30': 'sum'
            }).reset_index().rename(columns={
                'w_tau7': 'author_decay_tau7',
                'w_tau30': 'author_decay_tau30'
            })
            
        # 4. Мержим к кандидатам
        df = candidates.merge(item_decay, on='edition_id', how='left').fillna(0)
        
        # Log1p трансформа
        df['log1p_edition_decay_tau7'] = np.log1p(df['edition_decay_tau7'])
        df['log1p_edition_decay_tau30'] = np.log1p(df['edition_decay_tau30'])
        
        if author_decay is not None:
             if 'author_id' not in df.columns:
                 # Пытаемся найти author_id через merge (если его нет)
                 pass # Предполагаем, что он есть после transform enrich
             else:
                df = df.merge(author_decay, on='author_id', how='left').fillna(0)
                df['log1p_author_decay_tau7'] = np.log1p(df['author_decay_tau7'])
                df['log1p_author_decay_tau30'] = np.log1p(df['author_decay_tau30'])
        
        return df

    def _add_interaction_diffs(self, df, slice_df):
        """
        Считает разницы между свойствами книги и привычками юзера (Diffs, Z-scores).
        Требует, чтобы в df уже были: 'edition_bayes_avg_rating' (из const), 'book_pop_30' (из dyn).
        """
        # 1. Считаем профиль юзера в текущем окне (Mean Rating, Mean Pop)
        # Mean Rating юзера
        u_rating_stats = slice_df[slice_df['rating'] > 0].groupby('user_id')['rating'].agg(['mean', 'std']).reset_index()
        u_rating_stats.columns = ['user_id', 'user_mean_rating', 'user_std_rating']
        
        # 2. Мержим статистику юзера к кандидатам
        df = df.merge(u_rating_stats, on='user_id', how='left')
        
        # Заполняем пропуски (если юзер ничего не оценивал, ставим среднее 4.0 и std 1.0)
        df['user_mean_rating'] = df['user_mean_rating'].fillna(4.0)
        df['user_std_rating'] = df['user_std_rating'].fillna(1.0)
        
        # 3. Считаем Diffs (Rating)
        if 'edition_bayes_avg_rating' in df.columns:
            # Насколько книга лучше того, что обычно ставит юзер
            df['diff_bayes_rate_user_mean'] = df['edition_bayes_avg_rating'] - df['user_mean_rating']
            df['abs_diff_bayes_rate_user_mean'] = df['diff_bayes_rate_user_mean'].abs()
            
            # Z-score по рейтингу: (BookRate - UserMean) / UserStd
            # (насколько эта оценка аномальна для юзера)
            df['z_bayes_rate_user'] = (df['edition_bayes_avg_rating'] - df['user_mean_rating']) / (df['user_std_rating'] + 1e-9)

        if self.config.dyn_features.get('calc_user_pop_zscore') and 'item_win_total_events' in df.columns:
            # Считаем среднюю популярность кандидатов, которые мы ему предложили (контекст сессии)
            # Или лучше: насколько этот кандидат популярнее среднего кандидата для этого юзера
            grp = df.groupby('user_id')['item_win_total_events']
            u_pop_mean = grp.transform('mean')
            u_pop_std = grp.transform('std').fillna(1.0)
            
            df['z_log1p_edition_popularity_user'] = (df['item_win_total_events'] - u_pop_mean) / (u_pop_std + 1e-9)

        if self.config.dyn_features.get('calc_user_year_zscore') and 'publication_year' in df.columns:
            grp_year = df.groupby('user_id')['publication_year']
            u_year_mean = grp_year.transform('mean')
            u_year_std = grp_year.transform('std').fillna(5.0) # std по умолчанию 5 лет
            
            df['z_years_since_publication_user'] = (df['publication_year'] - u_year_mean) / (u_year_std + 1e-9)
            
        return df

    def _add_parent_level_stats(self, candidates, slice_df, meta):
        """
        Считает популярность Книги (Work) и Автора внутри окна (30 дней).
        Помогает модели понять, если издание новое, но сама книга/автор сейчас в хайпе.
        """
        # 1. Обогащаем слайс метаданными (если их нет), чтобы группировать по book/author
        cols_needed = []
        if 'book_id' not in slice_df.columns: cols_needed.append('book_id')
        if 'author_id' not in slice_df.columns: cols_needed.append('author_id')
        
        if cols_needed:
            # Используем inner, так как нам нужна стата только по существующим книгам
            slice_enriched = slice_df.merge(meta[['edition_id'] + cols_needed], on='edition_id', how='inner')
        else:
            slice_enriched = slice_df

        # 2. Агрегаты
        # Book Pop 30 (Популярность произведения)
        book_stats = slice_enriched.groupby('book_id').size().reset_index(name='book_pop_30')
        
        # Author Pop 30 (Популярность автора)
        author_stats = slice_enriched.groupby('author_id').size().reset_index(name='author_pop_30')
        
        # 3. Мержим к кандидатам
        # Сначала убедимся, что в кандидатах есть book_id/author_id (обычно добавляется в начале transform)
        if 'book_id' not in candidates.columns:
            candidates = candidates.merge(meta[['edition_id', 'book_id']], on='edition_id', how='left')
        if 'author_id' not in candidates.columns:
            candidates = candidates.merge(meta[['edition_id', 'author_id']], on='edition_id', how='left')

        candidates = candidates.merge(book_stats, on='book_id', how='left')
        candidates = candidates.merge(author_stats, on='author_id', how='left')
        
        # Заполняем нулями (если в окне не читали - значит популярность 0)
        candidates['book_pop_30'] = candidates['book_pop_30'].fillna(0).astype(np.int32)
        candidates['author_pop_30'] = candidates['author_pop_30'].fillna(0).astype(np.int32)
        
        # (Опционально) Log-transform для нормализации распределения
        candidates['book_pop_30_log'] = np.log1p(candidates['book_pop_30'])
        candidates['author_pop_30_log'] = np.log1p(candidates['author_pop_30'])
        
        return candidates

    def _add_window_stats(self, candidates, slice_df):
        flags = self.config.dyn_features
        
        # 1. Разделяем потоки
        reads = slice_df[slice_df['event_type'] == 2]
        wishes = slice_df[slice_df['event_type'] == 1]
        
        # --- A. Item Stats ---
        item_stats_dfs = []
        
        # Базовые (общие)
        base_stats = slice_df.groupby('edition_id').size().reset_index(name='item_win_total_events')
        item_stats_dfs.append(base_stats)
        
        if flags.get('split_read_wishlist'):
            i_reads = reads.groupby('edition_id').size().reset_index(name='item_win_reads')
            i_wishes = wishes.groupby('edition_id').size().reset_index(name='item_win_wishes')
            item_stats_dfs.append(i_reads)
            item_stats_dfs.append(i_wishes)
            
        if flags.get('rating_details'):
            # Средний рейтинг и кол-во оценок (только из reads, где рейтинг > 0)
            rated = slice_df[slice_df['rating'] > 0]
            i_rating = rated.groupby('edition_id').agg(
                item_win_mean_rating=('rating', 'mean'),
                item_win_rating_cnt=('rating', 'count')
            ).reset_index()
            item_stats_dfs.append(i_rating)
            
        # Мержим всё для айтемов
        item_feat = item_stats_dfs[0]
        for d in item_stats_dfs[1:]:
            item_feat = item_feat.merge(d, on='edition_id', how='outer')
        item_feat = item_feat.fillna(0)
        
        # Ratios (Item)
        if flags.get('calc_ratios') and flags.get('split_read_wishlist'):
            item_feat['item_read_share'] = item_feat['item_win_reads'] / (item_feat['item_win_total_events'] + 1)

        # --- B. User Stats ---
        user_stats_dfs = []
        u_base = slice_df.groupby('user_id').size().reset_index(name='user_win_total_events')
        user_stats_dfs.append(u_base)
        
        if flags.get('split_read_wishlist'):
            u_reads = reads.groupby('user_id').size().reset_index(name='user_win_reads')
            u_wishes = wishes.groupby('user_id').size().reset_index(name='user_win_wishes')
            user_stats_dfs.append(u_reads)
            user_stats_dfs.append(u_wishes)
            
        if flags.get('rating_details'):
            # Доля высоких оценок
            rated_u = slice_df[slice_df['rating'] > 0]
            # (count >= 4) / count
            u_high = rated_u.groupby('user_id')['rating'].apply(lambda x: (x >= 4).mean()).reset_index(name='user_high_rate_share')
            user_stats_dfs.append(u_high)
            
        # Мержим юзеров
        user_feat = user_stats_dfs[0]
        for d in user_stats_dfs[1:]:
            user_feat = user_feat.merge(d, on='user_id', how='outer')
        user_feat = user_feat.fillna(0)
        
        # Ratios (User)
        if flags.get('calc_ratios') and flags.get('split_read_wishlist'):
             user_feat['user_read_share'] = user_feat['user_win_reads'] / (user_feat['user_win_total_events'] + 1)
        
        # --- C. Merge to Candidates ---
        df = candidates.merge(item_feat, on='edition_id', how='left').fillna(0)
        df = df.merge(user_feat, on='user_id', how='left').fillna(0)
        
        # --- D. Interaction Deltas (Last Event) ---
        if flags.get('calc_deltas'):
            # Ищем макс дату для пары (u, i) отдельно для Read и Wish
            # Оптимизация: группируем только слайс, где есть события
            
            # Функция для расчета дней
            window_end = slice_df['event_ts'].max()
            
            def add_delta(sub_df, prefix):
                last = sub_df.groupby(['user_id', 'edition_id'])['event_ts'].max().reset_index()
                last[f'{prefix}_days_ago'] = (window_end - last['event_ts']).dt.days
                return last[['user_id', 'edition_id', f'{prefix}_days_ago']]
            
            if not reads.empty:
                r_delta = add_delta(reads, 'last_read')
                df = df.merge(r_delta, on=['user_id', 'edition_id'], how='left')
                
            if not wishes.empty:
                w_delta = add_delta(wishes, 'last_wish')
                df = df.merge(w_delta, on=['user_id', 'edition_id'], how='left')
            
            # Заполняем пропуски (если не читал - значит давно, ставим window_days + 1)
            max_days = self.config.window_days + 1
            if 'last_read_days_ago' in df.columns:
                df['last_read_days_ago'] = df['last_read_days_ago'].fillna(max_days)
                df['has_read_in_window'] = (df['last_read_days_ago'] < max_days).astype(int)
                
            if 'last_wish_days_ago' in df.columns:
                df['last_wish_days_ago'] = df['last_wish_days_ago'].fillna(max_days)
                df['has_wish_in_window'] = (df['last_wish_days_ago'] < max_days).astype(int)

        return df

    def _add_author_affinity(self, candidates, slice_df, meta):
        flags = self.config.dyn_features
        
        # 1. Enrich
        if 'author_id' not in slice_df.columns:
            slice_enriched = slice_df.merge(meta[['edition_id', 'author_id']], on='edition_id', how='inner')
        else:
            slice_enriched = slice_df
            
        # 2. Base Counts
        ua_stats = slice_enriched.groupby(['user_id', 'author_id']).size().reset_index(name='ua_total_events')
        
        # 3. Split Counts
        if flags.get('author_split'):
            reads = slice_enriched[slice_enriched['event_type'] == 2]
            wishes = slice_enriched[slice_enriched['event_type'] == 1]
            
            ua_reads = reads.groupby(['user_id', 'author_id']).size().reset_index(name='ua_read_events')
            ua_wishes = wishes.groupby(['user_id', 'author_id']).size().reset_index(name='ua_wish_events')
            
            ua_stats = ua_stats.merge(ua_reads, on=['user_id', 'author_id'], how='left')
            ua_stats = ua_stats.merge(ua_wishes, on=['user_id', 'author_id'], how='left')
            ua_stats = ua_stats.fillna(0)
            
        # 4. Merge
        if 'author_id' not in candidates.columns:
            candidates = candidates.merge(meta[['edition_id', 'author_id']], on='edition_id', how='left')
            
        merged = candidates.merge(ua_stats, on=['user_id', 'author_id'], how='left').fillna(0)
        
        # 5. Ratios
        # Чтобы посчитать доли, нам нужны общие счетчики юзера (они уже есть в candidates после _add_window_stats)
        # ua_read_share = ua_read_events / user_win_reads
        if flags.get('author_split') and flags.get('calc_ratios'):
            if 'user_win_reads' in merged.columns:
                merged['ua_read_share'] = merged['ua_read_events'] / (merged['user_win_reads'] + 1)
                
        return merged

    def _add_user_genre_affinity(self, candidates, slice_df, meta, book_genres_map):
        flags = self.config.dyn_features
        
        # Если сплит выключен - старая логика (просто общая сумма)
        # Если включен - делаем отдельно для read и wish
        
        if not flags.get('genre_split'):
            return super()._add_user_genre_affinity(candidates, slice_df, meta, book_genres_map) # Или старый код
            
        # --- New Split Logic ---
        
        # 1. Prepare Slice with Genres
        if 'book_id' not in slice_df.columns:
            slice_with_book = slice_df.merge(meta[['edition_id', 'book_id']], on='edition_id', how='inner')
        else:
            slice_with_book = slice_df.copy()
            
        # Explode Genres in Slice
        unique_books_slice = slice_with_book['book_id'].unique()
        bg_data = []
        for bid in unique_books_slice:
            if bid in book_genres_map:
                for gid in book_genres_map[bid]:
                    bg_data.append((bid, gid))
        
        if not bg_data: return candidates # Empty
        
        bg_df_slice = pd.DataFrame(bg_data, columns=['book_id', 'genre_id'])
        slice_long = slice_with_book[['user_id', 'book_id', 'event_type']].merge(bg_df_slice, on='book_id', how='inner')
        
        # 2. Calc User Genre Profile (Split)
        # Group by User, Genre, EventType
        # Pivot table: index=[user, genre], columns=type, values=count
        ug_stats = slice_long.groupby(['user_id', 'genre_id', 'event_type']).size().unstack(fill_value=0).reset_index()
        
        # Rename columns (1->wish, 2->read)
        if 1 not in ug_stats.columns: ug_stats[1] = 0
        if 2 not in ug_stats.columns: ug_stats[2] = 0
        
        ug_stats.rename(columns={1: 'g_wish_cnt', 2: 'g_read_cnt'}, inplace=True)
        
        # 3. Prepare Candidates (Book -> Genres)
        unique_books_cands = candidates['book_id'].unique()
        bg_data_cands = []
        for bid in unique_books_cands:
            if bid in book_genres_map:
                for gid in book_genres_map[bid]:
                    bg_data_cands.append((bid, gid))
                    
        if not bg_data_cands: return candidates
        bg_df_cands = pd.DataFrame(bg_data_cands, columns=['book_id', 'genre_id'])
        
        cand_long = candidates[['user_id', 'edition_id', 'book_id']].merge(bg_df_cands, on='book_id', how='inner')
        
        # 4. Merge & Aggregate
        cand_long = cand_long.merge(ug_stats[['user_id', 'genre_id', 'g_wish_cnt', 'g_read_cnt']], on=['user_id', 'genre_id'], how='left')
        
        # Sum scores per edition
        final_scores = cand_long.groupby(['user_id', 'edition_id'])[['g_wish_cnt', 'g_read_cnt']].sum().reset_index()
        final_scores.rename(columns={'g_wish_cnt': 'ug_wish_affinity', 'g_read_cnt': 'ug_read_affinity'}, inplace=True)
        
        # 5. Final Merge
        result = candidates.merge(final_scores, on=['user_id', 'edition_id'], how='left').fillna(0)
        return result
    


    def _add_user_meta(self, df, users_meta):
        """Добавляет соц-дем характеристики пользователя (из v7)."""
        # Проверяем, какие колонки есть (обычно sex, age)
        available_cols = [c for c in ['sex', 'age'] if c in users_meta.columns]
        
        if not available_cols:
            return df
            
        # Мержим
        df = df.merge(users_meta[['user_id'] + available_cols], on='user_id', how='left')
        
        # Обработка пропусков (как в v7)
        if 'sex' in df.columns:
            df['sex'] = df['sex'].fillna('unknown').astype(str)
            # Если нужно закодировать в int, можно сделать map, но CatBoost ест строки
            
        if 'age' in df.columns:
            # Заполняем медианой по датасету (или -1)
            median_age = users_meta['age'].median()
            df['age'] = df['age'].fillna(median_age)
            
            # (Опционально) Бакеты возраста, если были в v7
            # df['age_bucket'] = pd.cut(df['age'], bins=[0, 18, 25, 35, 50, 100], labels=False)
            
        return df
    
    def _add_global_recency(self, df, slice_df):
        """
        Считает, сколько дней прошло с последнего взаимодействия с книгой (во всей системе).
        Использует slice_df как "текущее знание о мире".
        """
        # Находим последнюю дату события для каждой edition_id в текущем окне/слайсе
        last_events = slice_df.groupby('edition_id')['event_ts'].max()
        window_end = slice_df['event_ts'].max()
        
        # Считаем дни
        days_diff = (window_end - last_events).dt.days
        days_map = days_diff.to_dict()
        
        # Мапим. Если книги нет в slice (нет событий), ставим заглушку (например, 365*10)
        df['global_days_since_last_interaction'] = df['edition_id'].map(days_map).fillna(3650)
        return df

In [ ]:
import pandas as pd
import numpy as np
import implicit
from scipy.sparse import csr_matrix
from typing import List, Dict, Optional, Set, Any
import os

class CandidateGenerator:
    """
    Генерирует кандидатов используя разные стратегии.
    Полностью изолированный класс: сам готовит индексы и модели в fit().
    
    Восстановленный функционал v7 + NEW Hybrid Strategy:
      - ALS (Collaborative Filtering)
      - Top Popular (Global & Window)
      - Wishlist (Explicit Intent)
      - Random (Negative Sampling with History Exclusion)
      - External File Injection + Smart ALS Scoring Filter
      - [NEW] ALS + TopPop Hybrid (Z-Score Fusion)
    """
    
    def __init__(self, config):
        self.config = config
        
        # Models
        self.als_model = None
        
        # Data Structures (Learned in fit)
        self.window_top_pop = []
        self.global_top_pop = []
        self.user_history = {} # {uid: {items...}} для исключения из Random
        self.wishlist_map = {} # {uid: {items...}} для стратегии Wishlist
        
        # Mappings for ALS
        self.user_map = {} 
        self.item_map = {}
        self.item_inv_map = {}
        self.user_inv_map = {}
        
        # Global Item Pool (for Random)
        self.all_items = np.array([])
        
        # [NEW] Structures for Hybrid Strategy
        self.item_pop_log_score = None # np.array aligned with item_map
        self.pop_mean = 0.0
        self.pop_std = 1.0

    def fit(self, global_history: pd.DataFrame, window_data: pd.DataFrame):
        """
        Обучает внутренние модели на переданной истории.
        """
        # 1. Prepare Data Structures
        print("[CandidateGen] Indexing User History & Wishlist...")
        self.user_history = global_history.groupby('user_id')['edition_id'].apply(set).to_dict()
        
        wish_df = global_history[global_history['event_type'] == 1]
        self.wishlist_map = wish_df.groupby('user_id')['edition_id'].apply(list).to_dict()
        
        self.all_items = global_history['edition_id'].unique()
        
        # 2. Global ALS for Candidates & Filtering
        print("[CandidateGen] Fitting internal ALS...")
        self._fit_als(global_history)
        
        # 3. Top Popular (Global & Window)
        global_reads = global_history[global_history['event_type'] == 2]
        window_reads = window_data[window_data['event_type'] == 2]
        
        self.global_top_pop = global_reads['edition_id'].value_counts().head(200).index.tolist()
        self.window_top_pop = window_reads['edition_id'].value_counts().head(200).index.tolist()
        
        # [NEW] 4. Pre-calculate Popularity Scores for Hybrid Strategy
        # Считаем логарифм популярности для ВСЕХ айтемов, которые знает ALS
        # Используем global_reads для чистоты (популярность по чтениям)
        print("[CandidateGen] Pre-calculating hybrid popularity scores...")
        pop_counts = global_reads['edition_id'].value_counts()
        
        # Создаем массив, выравненный по self.item_map (internal idx)
        num_items = len(self.item_map)
        self.item_pop_log_score = np.zeros(num_items, dtype=np.float32)
        
        # Заполняем массив (векторизовать сложно, так как map dict, но цикл быстрый)
        # Оптимизация: итерируемся по items, которые есть в pop_counts
        for iid, count in pop_counts.items():
            if iid in self.item_map:
                idx = self.item_map[iid]
                self.item_pop_log_score[idx] = np.log1p(count)
                
        # Глобальные статистики для Z-Score (Mean/Std по всему корпусу)
        # Исключаем нули из статистики, чтобы не смещать среднее (опционально, но логично)
        mask_nonzero = self.item_pop_log_score > 0
        if mask_nonzero.any():
            self.pop_mean = self.item_pop_log_score[mask_nonzero].mean()
            self.pop_std = self.item_pop_log_score[mask_nonzero].std() + 1e-9
        else:
            self.pop_mean = 0.0
            self.pop_std = 1.0
            
        return self

    def generate(self, target_users: list) -> pd.DataFrame:
        """
        Генерирует DataFrame [user_id, edition_id, strategy]
        для списка пользователей.
        """
        candidates_buffer = []
        
        # --- 1. ALS Strategy ---
        n_als = self.config.candidates_strategy.get('als', 0)
        if n_als > 0:
            df_als = self._generate_als(target_users, n_als)
            if not df_als.empty:
                df_als['strategy'] = 'als'
                candidates_buffer.append(df_als)

        # --- 2. Window Top Popular ---
        n_win = self.config.candidates_strategy.get('top_pop_window', 0)
        if n_win > 0:
            df_win = self._generate_top_pop(target_users, self.window_top_pop, n_win)
            if not df_win.empty:
                df_win['strategy'] = 'top_pop_window'
                candidates_buffer.append(df_win)
                
        # --- 3. Global Top Popular ---
        n_glob = self.config.candidates_strategy.get('top_pop_global', 0)
        if n_glob > 0:
            df_glob = self._generate_top_pop(target_users, self.global_top_pop, n_glob)
            if not df_glob.empty:
                df_glob['strategy'] = 'top_pop_global'
                candidates_buffer.append(df_glob)
        
        # --- 4. Wishlist ---
        n_wish = self.config.candidates_strategy.get('wishlist', 0)
        if n_wish > 0: 
            df_wish = self._generate_wishlist(target_users)
            if not df_wish.empty:
                df_wish['strategy'] = 'wishlist'
                candidates_buffer.append(df_wish)

        # --- 5. Random Negatives ---
        n_rnd = self.config.candidates_strategy.get('random', 0)
        if n_rnd > 0:
            df_rnd = self._generate_random(target_users, n_rnd)
            if not df_rnd.empty:
                df_rnd['strategy'] = 'random'
                candidates_buffer.append(df_rnd)

        # --- 6. Hack / From File ---
        n_file = self.config.candidates_strategy.get('from_file', 0)
        if n_file > 0:
            df_hack = self._generate_from_file(target_users)
            if not df_hack.empty:
                df_hack['strategy'] = 'hack_file'
                candidates_buffer.append(df_hack)
                
        # --- [NEW] 7. Hybrid ALS + TopPop ---
        n_hybrid = self.config.candidates_strategy.get('als_plus_toppop', 0)
        if n_hybrid > 0:
            df_hybrid = self._generate_hybrid_als_pop(target_users, n_hybrid)
            if not df_hybrid.empty:
                df_hybrid['strategy'] = 'als_plus_toppop'
                candidates_buffer.append(df_hybrid)

        # Merge & Deduplicate
        if not candidates_buffer:
            return pd.DataFrame(columns=['user_id', 'edition_id', 'strategy'])
            
        full_df = pd.concat(candidates_buffer, ignore_index=True)
        
        # Склеиваем стратегии для аналитики
        full_df = full_df.groupby(['user_id', 'edition_id'])['strategy'].apply(
            lambda x: '+'.join(sorted(set(x)))
        ).reset_index()
        
        return full_df

    # -------------------------------------------------------------------------
    # Internal Strategy Implementations
    # -------------------------------------------------------------------------

    def _fit_als(self, df):
        weights_map = self.config.als_event_weights
        temp_df = df[['user_id', 'edition_id', 'event_type']].copy()
        temp_df['weight'] = temp_df['event_type'].map(weights_map).fillna(1.0)
        
        grouped = temp_df.groupby(['user_id', 'edition_id'])['weight'].max().reset_index()
        
        users = grouped['user_id'].astype('category')
        items = grouped['edition_id'].astype('category')
        
        self.user_map = {u: i for i, u in enumerate(users.cat.categories)}
        self.user_inv_map = {i: u for u, i in self.user_map.items()}
        self.item_map = {item: i for i, item in enumerate(items.cat.categories)}
        self.item_inv_map = {i: item for item, i in self.item_map.items()}        
        
        row = users.cat.codes
        col = items.cat.codes
        data = grouped['weight'].values.astype(np.float32)
        
        sparse = csr_matrix((data, (row, col)), shape=(len(self.user_map), len(self.item_map)))
        self.sparse_matrix = sparse 
        
        self.als_model = implicit.als.AlternatingLeastSquares(
            factors=self.config.als_factors, 
            regularization=self.config.als_regularization,
            iterations=self.config.als_iterations, 
            random_state=self.config.als_random_state
        )
        self.als_model.fit(sparse)

    def _generate_als(self, users, n):
        u_idxs = [self.user_map[u] for u in users if u in self.user_map]
        if not u_idxs: return pd.DataFrame()
        
        ids, _ = self.als_model.recommend(u_idxs, self.sparse_matrix[u_idxs], N=n, filter_already_liked_items=False)
        
        res_users, res_items = [], []
        for i, u_idx in enumerate(u_idxs):
            real_user = self.user_inv_map[u_idx]
            for item_idx in ids[i]:
                res_users.append(real_user)
                res_items.append(self.item_inv_map[item_idx])
                
        return pd.DataFrame({'user_id': res_users, 'edition_id': res_items})

    def _generate_top_pop(self, users, top_items, n):
        items = top_items[:n]
        if not items: return pd.DataFrame()
        
        n_users = len(users)
        n_items = len(items)
        user_col = np.repeat(users, n_items)
        item_col = np.tile(items, n_users)
        
        return pd.DataFrame({'user_id': user_col, 'edition_id': item_col})

    def _generate_wishlist(self, users):
        pairs = []
        for u in users:
            if u in self.wishlist_map:
                for item in self.wishlist_map[u]:
                    pairs.append((u, item))
        return pd.DataFrame(pairs, columns=['user_id', 'edition_id'])

    def _generate_random(self, users, n):
        pairs = []
        rng = np.random.default_rng(self.config.random_seed)
        n_raw = n * 2 
        
        for u in users:
            seen = self.user_history.get(u, set())
            rand_items = rng.choice(self.all_items, size=n_raw, replace=True)
            count = 0
            for item in rand_items:
                if item not in seen:
                    pairs.append((u, item))
                    count += 1
                if count >= n:
                    break
                    
        return pd.DataFrame(pairs, columns=['user_id', 'edition_id'])

    def _generate_from_file(self, target_users):
        path = self.config.hack_candidates_path
        if not os.path.exists(path):
            return pd.DataFrame()
            
        try:
            df = pd.read_csv(path)
            target_users_set = set(target_users)
            df = df[df['user_id'].isin(target_users_set)].copy()
            if df.empty: return df
            
            conf = self.config.hack_strategy_config
            mode = conf.get('filter_mode', 'none')
            quantile = conf.get('quantile', 0.3)
            
            if self.als_model is None or mode == 'none':
                return df[['user_id', 'edition_id']]
                
            u_indices = df['user_id'].map(self.user_map).fillna(-1).astype(int)
            i_indices = df['edition_id'].map(self.item_map).fillna(-1).astype(int)
            
            valid_mask = (u_indices != -1) & (i_indices != -1)
            scores = np.zeros(len(df), dtype=np.float32)
            
            if valid_mask.any():
                u_factors = self.als_model.user_factors[u_indices[valid_mask]]
                i_factors = self.als_model.item_factors[i_indices[valid_mask]]
                scores[valid_mask] = np.sum(u_factors * i_factors, axis=1)
            
            df['als_score'] = scores
            df['is_valid'] = valid_mask
            
            if mode == 'per_user_bottom_k':
                df['pct_rank'] = df.groupby('user_id')['als_score'].rank(pct=True, method='min')
                mask_keep = (df['pct_rank'] <= quantile) | (~df['is_valid'])
                filtered_df = df[mask_keep].copy()
                return filtered_df[['user_id', 'edition_id']]
                
            elif mode == 'low_als_score':
                 threshold = np.percentile(scores[valid_mask], conf.get('als_percentile_cutoff', 70))
                 return df[scores <= threshold][['user_id', 'edition_id']]
            
            return df[['user_id', 'edition_id']]

        except Exception as e:
            print(f"[CandidateGen] Error reading hack file: {e}")
            return pd.DataFrame()

    def _generate_hybrid_als_pop(self, users, n_final):
        """
        [NEW] Гибридная генерация: ALS + Popularity.
        1. Берем широкий пул (x5) от ALS и от TopPop.
        2. Объединяем их.
        3. Считаем взвешенный скор: alpha * Z(ALS) + beta * Z(Pop).
        4. Отбираем топ-N.
        """
        cfg = self.config.hybrid_strategy_config
        alpha = cfg.get('alpha', 0.7)
        beta = cfg.get('beta', 0.3)
        pool_factor = cfg.get('pool_factor', 5)
        
        # Расширенный размер пула
        pool_size = int(n_final * pool_factor)
        
        # 1. Формируем пул кандидатов
        # А. ALS Candidates (Retrieval)
        u_idxs = [self.user_map[u] for u in users if u in self.user_map]
        if not u_idxs: return pd.DataFrame()
        
        # Берем кандидатов с запасом. 
        # filter_already_liked_items=False, чтобы не фильтровать потенциально релевантные перечитывания
        # (или можно True, зависит от бизнес-логики, но для ранжирования лучше False и фильтровать позже)
        ids_als_batch, _ = self.als_model.recommend(
            u_idxs, 
            self.sparse_matrix[u_idxs], 
            N=pool_size, 
            filter_already_liked_items=False
        )
        
        # B. Top Pop Candidates (Retrieval)
        # Получаем внутренние ID топ популярных
        top_pop_ids = [self.item_map[iid] for iid in self.global_top_pop[:pool_size] if iid in self.item_map]
        
        # 2. Ранжирование пула
        results = []
        
        # Pre-fetch factors (Numpy array view)
        u_factors = self.als_model.user_factors[u_idxs] # (n_batch, dim)
        item_factors = self.als_model.item_factors      # (n_items, dim)
        
        # Цикл по юзерам (для объединения множеств и расчета)
        for i, u_idx in enumerate(u_idxs):
            real_user = self.user_inv_map[u_idx]
            
            # Объединяем кандидатов (Set Union)
            candidates_indices = set(ids_als_batch[i])
            candidates_indices.update(top_pop_ids)
            candidates_indices = list(candidates_indices)
            
            if not candidates_indices: continue
            
            cand_idxs_arr = np.array(candidates_indices, dtype=int)
            
            # --- A. ALS Component ---
            # Dot Product для всех кандидатов в пуле (даже тех, кого ALS не вернул в топе)
            u_vec = u_factors[i]
            i_vecs = item_factors[cand_idxs_arr]
            
            als_raw = i_vecs.dot(u_vec)
            
            # In-Pool Normalization (Standardization)
            # Нормируем скоры внутри текущего пула кандидата, чтобы привести к шкале Z
            if len(als_raw) > 1:
                als_mean = als_raw.mean()
                als_std = als_raw.std() + 1e-9
                als_z = (als_raw - als_mean) / als_std
            else:
                als_z = np.zeros_like(als_raw)
                
            # --- B. Popularity Component ---
            # Берем прекалькулированный Log(Pop)
            pop_raw = self.item_pop_log_score[cand_idxs_arr]
            
            # Global Normalization (используем глобальные статистики, посчитанные в fit)
            pop_z = (pop_raw - self.pop_mean) / self.pop_std
            
            # --- C. Final Score ---
            final_score = alpha * als_z + beta * pop_z
            
            # --- D. Selection ---
            # Берем топ N
            if len(final_score) > n_final:
                # argpartition быстрее для топ-k
                top_k_idx = np.argpartition(final_score, -n_final)[-n_final:]
                # Сортируем этот топ (так как argpartition не гарантирует порядок внутри топа)
                top_k_sorted_local = np.argsort(final_score[top_k_idx])[::-1]
                best_indices_local = top_k_idx[top_k_sorted_local]
            else:
                best_indices_local = np.argsort(final_score)[::-1]
                
            best_item_idxs = cand_idxs_arr[best_indices_local]
            
            for item_idx in best_item_idxs:
                results.append((real_user, self.item_inv_map[item_idx]))
        
        return pd.DataFrame(results, columns=['user_id', 'edition_id'])

In [ ]:
class RankFeatureGenerator:
    """
    [UPGRADED v17] Virtual Ranking Feature Generator.
    Генерирует ранговые фичи относительно фиксированного Reference Set.
    Теперь умеет сам считать агрегаты по истории и использовать эмбеддинги из ConstantGen.
    """
    def __init__(self, config):
        self.config = config
        self.active = config.rank_features['enabled']
        self.features = config.rank_features
        
        # Хранилище распределений ("Линеек")
        self.reference_distributions = {} 
        self.reference_users = set()

        # Хранилище глобальных статистик (вычисляются в fit)
        self.book_pop_map = {}
        self.author_pop_map = {}
        self.publisher_pop_map = {}
        self.edition_pop_map = {}

    def fit(self, reference_path: str, 
            history_df: pd.DataFrame,          # <--- Полная история (Global)
            editions_meta: pd.DataFrame,       # <--- Метаданные
            const_gen_wrapper,                 # <--- Доступ к эмбеддингам (Qwen/SVD)
            als_model_wrapper=None):           # <--- Доступ к ALS
        """
        Готовит эталонные распределения.
        1. Считает глобальную популярность по history_df.
        2. Считает скоры (Qwen, SVD, Pop) для Reference Set.
        3. Строит индексы распределений.
        """
        if not self.active: return self
        if not os.path.exists(reference_path):
            print(f"[RankGen] Reference file not found at {reference_path}. Skipping.")
            return self

        print(f"[RankGen] Loading reference candidates from {reference_path}...")
        try:
            ref_df = pd.read_csv(reference_path)
        except Exception as e:
            print(f"[RankGen] Error loading reference: {e}")
            return self
            
        self.reference_users = set(ref_df['user_id'].unique())

        # =========================================================
        # 1. CALCULATE GLOBAL POPULARITY STATS (Internal Logic)
        # =========================================================
        print("[RankGen] Aggregating Global Popularity Stats from History...")
        
        # Обогащаем историю, чтобы группировать по book/author
        # Используем inner, нам нужны только валидные ID
        hist_enriched = history_df.merge(
            editions_meta[['edition_id', 'book_id', 'author_id', 'publisher_id']], 
            on='edition_id', how='inner'
        )

        # A. Edition Popularity
        if self.features.get('rank_popularity'):
            self.edition_pop_map = history_df['edition_id'].value_counts().to_dict()

        # B. Book Popularity
        if self.features.get('rank_book_pop'):
            self.book_pop_map = hist_enriched['book_id'].value_counts().to_dict()

        # C. Author Popularity
        if self.features.get('rank_author_pop'):
            self.author_pop_map = hist_enriched['author_id'].value_counts().to_dict()

        # D. Publisher Popularity
        if self.features.get('rank_publisher_pop'):
            self.publisher_pop_map = hist_enriched['publisher_id'].value_counts().to_dict()


        # =========================================================
        # 2. ENRICH REFERENCE DATASET (Calculations for Ruler)
        # =========================================================
        print("[RankGen] Enriching Reference Set & Building Indices...")

        # Предварительно обогащаем ref_df метаданными
        cols_needed = ['book_id', 'author_id', 'publisher_id', 'publication_year']
        cols_to_merge = [c for c in cols_needed if c in editions_meta.columns]
        ref_df = ref_df.merge(editions_meta[['edition_id'] + cols_to_merge], on='edition_id', how='left')

        # --- A. POPULARITY RANKS ---
        if self.features.get('rank_popularity'):
            ref_df['val_pop'] = ref_df['edition_id'].map(self.edition_pop_map).fillna(0)
            self._build_dist_index('pop', ref_df, 'val_pop')

        if self.features.get('rank_book_pop'):
            ref_df['val_book_pop'] = ref_df['book_id'].map(self.book_pop_map).fillna(0)
            self._build_dist_index('book_pop', ref_df, 'val_book_pop')

        if self.features.get('rank_author_pop'):
            ref_df['val_author_pop'] = ref_df['author_id'].map(self.author_pop_map).fillna(0)
            self._build_dist_index('author_pop', ref_df, 'val_author_pop')
            
        if self.features.get('rank_publisher_pop'):
            ref_df['val_publisher_pop'] = ref_df['publisher_id'].map(self.publisher_pop_map).fillna(0)
            self._build_dist_index('publisher_pop', ref_df, 'val_publisher_pop')

        # --- B. YEAR RANK ---
        if self.features.get('rank_year') and 'publication_year' in ref_df.columns:
            ref_df['val_year'] = ref_df['publication_year'].fillna(0)
            self._build_dist_index('year', ref_df, 'val_year')

        # --- C. QWEN SIMILARITY RANK (Using ConstGen Embeddings) ---
        # Логика: берем user_vec и item_vec из const_gen, считаем dot product для Reference Set
        if self.features.get('rank_qwen_sim') and const_gen_wrapper:
            print("   -> Calculating Qwen Similarity for Reference Set...")
            u_embs = const_gen_wrapper.qwen_user_embs
            i_embs = const_gen_wrapper.qwen_item_embs
            
            # Функция быстрого расчета (предполагаем векторы нормированы в const_gen)
            def get_qwen_sim(uid, eid):
                u = u_embs.get(uid)
                i = i_embs.get(eid)
                if u is None or i is None: return 0.0
                return np.dot(u, i)

            # Применяем (можно ускорить через zip, если критично по времени)
            ref_df['val_qwen'] = [get_qwen_sim(u, i) for u, i in zip(ref_df['user_id'], ref_df['edition_id'])]
            self._build_dist_index('qwen', ref_df, 'val_qwen')

        # --- D. TF-IDF/SVD SIMILARITY RANK ---
        if self.features.get('rank_tfidf_sim') and const_gen_wrapper:
            print("   -> Calculating TF-IDF/SVD Similarity for Reference Set...")
            u_profs = const_gen_wrapper.user_svd_profiles
            i_vecs = const_gen_wrapper.text_emb_map

            def get_svd_sim(uid, eid):
                u = u_profs.get(uid)
                i = i_vecs.get(eid)
                if u is None or i is None: return 0.0
                return np.dot(u, i) # Assuming normalized

            ref_df['val_tfidf'] = [get_svd_sim(u, i) for u, i in zip(ref_df['user_id'], ref_df['edition_id'])]
            self._build_dist_index('tfidf', ref_df, 'val_tfidf')

        # --- E. ALS SCORE RANK (Existing) ---
        if self.features.get('rank_als_score') and als_model_wrapper and als_model_wrapper.als_model:
            # ... (Код для ALS оставляем как был, он сложнее из-за индексов) ...
            # Для краткости: если вы используете ALS rank, добавьте сюда логику из старого кода
            pass

        return self

    def _build_dist_index(self, key, df, val_col):
        """Создает словарь {uid: sorted_array} для быстрого searchsorted."""
        sorted_groups = df.sort_values(['user_id', val_col])
        self.reference_distributions[key] = {}
        for uid, grp in tqdm(sorted_groups.groupby('user_id'), desc=f"Indexing {key}", leave=False):
            self.reference_distributions[key][uid] = grp[val_col].values.astype(np.float32)

    def transform(self, candidates: pd.DataFrame) -> pd.DataFrame:
        if not self.active or candidates.empty or not self.reference_distributions:
            return candidates
            
        df = candidates.copy()
        print("[RankGen] Applying Virtual Ranking Stats...")

        # 1. Popularity Ranks (Map GLOBAL stats from self.*_pop_map)
        # Важно: Мы мапим значения из словарей, подсчитанных в FIT. 
        # Это гарантирует, что шкала (Scale) у кандидатов такая же, как у референса.
        
        # Edition Pop
        if self.features.get('rank_popularity') and 'pop' in self.reference_distributions:
            # Мапим сами, не полагаясь на внешние колонки
            vals = df['edition_id'].map(self.edition_pop_map).fillna(0).values
            df['rank_popularity_virtual'] = self._apply_virtual_rank(df['user_id'].values, vals, 'pop')

        # Book Pop
        if self.features.get('rank_book_pop') and 'book_pop' in self.reference_distributions:
            # Убедимся, что book_id есть
            if 'book_id' not in df.columns: 
                print("[RankGen Warning] book_id missing for ranking")
            else:
                vals = df['book_id'].map(self.book_pop_map).fillna(0).values
                df['rank_book_pop_virtual'] = self._apply_virtual_rank(df['user_id'].values, vals, 'book_pop')

        # Author Pop
        if self.features.get('rank_author_pop') and 'author_pop' in self.reference_distributions:
            if 'author_id' in df.columns:
                vals = df['author_id'].map(self.author_pop_map).fillna(0).values
                df['rank_author_pop_virtual'] = self._apply_virtual_rank(df['user_id'].values, vals, 'author_pop')

        # Publisher Pop
        if self.features.get('rank_publisher_pop') and 'publisher_pop' in self.reference_distributions:
            if 'publisher_id' in df.columns:
                vals = df['publisher_id'].map(self.publisher_pop_map).fillna(0).values
                df['rank_publisher_pop_virtual'] = self._apply_virtual_rank(df['user_id'].values, vals, 'publisher_pop')

        # 2. Embedding Ranks (Qwen & TFIDF)
        # Здесь мы ожидаем, что ConstantFeatureGenerator УЖЕ посчитал косинусы для текущих кандидатов.
        # Обычно колонки называются 'qwen_cos_user_edition' и 'svd_cosine_sim'.
        
        if self.features.get('rank_qwen_sim') and 'qwen' in self.reference_distributions:
            col = 'qwen_cos_user_edition'
            if col in df.columns:
                df['rank_qwen_sim_virtual'] = self._apply_virtual_rank(df['user_id'].values, df[col].values, 'qwen')
        
        if self.features.get('rank_tfidf_sim') and 'tfidf' in self.reference_distributions:
            col = 'svd_cosine_sim'
            if col in df.columns:
                df['rank_tfidf_sim_virtual'] = self._apply_virtual_rank(df['user_id'].values, df[col].values, 'tfidf')

        # 3. Year Rank
        if self.features.get('rank_year') and 'year' in self.reference_distributions:
            if 'publication_year' in df.columns:
                df['rank_year_virtual'] = self._apply_virtual_rank(df['user_id'].values, df['publication_year'].fillna(0).values, 'year')

        return df

    def _apply_virtual_rank(self, user_ids, values, dist_key):
        """Векторизованный расчет ранга через searchsorted"""
        ref_dict = self.reference_distributions[dist_key]
        results = np.zeros(len(user_ids), dtype=np.float32)
        results[:] = 0.5 # Default

        # Группируем по юзерам для применения searchsorted
        # (Используем DF wrapper для удобства группировки)
        temp = pd.DataFrame({'u': user_ids, 'v': values, 'idx': range(len(user_ids))})
        
        for uid, grp in temp.groupby('u'):
            if uid in ref_dict:
                ref_arr = ref_dict[uid]
                if len(ref_arr) == 0: continue
                
                # Ищем позицию наших значений в эталонном распределении
                ranks = np.searchsorted(ref_arr, grp['v'].values, side='left')
                norm_ranks = ranks / len(ref_arr)
                results[grp['idx'].values] = norm_ranks
                
        return results

In [ ]:
from datetime import timedelta

class DataOrchestrator:
    """
    Управляет нарезкой времени и запуском генераторов.
    Гарантирует отсутствие лика.
    """
    def __init__(self, config: PipelineConfig):
        self.config = config
    
    def run(self, full_interactions: pd.DataFrame, editions_meta: pd.DataFrame, 
            book_genres_map: Dict, users_meta: pd.DataFrame):
        """
        Возвращает: train_df, val_df (готовые для MetaRanker)
        """
        # Сортировка по времени
        df = full_interactions.sort_values('event_ts').copy()
        
        # Определяем точку валидации
        max_date = df['event_ts'].max()
        if self.config.force_validation_date:
            val_split = pd.to_datetime(self.config.force_validation_date)
        else:
            val_split = max_date - timedelta(days=self.config.target_days)
            
        print(f"[Orchestrator] Validation Split: {val_split}")
        
        train_dfs = []
        dyrting_dfs = [] 
        val_dfs = []
        
        # --- Sliding Window Loop ---
        current_split = val_split
        fold_idx = 0
        is_validation = True # Первый проход - это валидационный фолд
        
        while True:
            # Окна
            window_start = current_split - timedelta(days=self.config.window_days)
            target_end = current_split + timedelta(days=self.config.target_days)
            
            if window_start < df['event_ts'].min():
                break # Данные кончились
                
            print(f"Processing Fold {fold_idx}: Window [{window_start} -> {current_split}] | Target -> {target_end}")
            
            # Срез данных
            # Global History: всё до момента split (для ALS)
            global_hist = df[df['event_ts'] < current_split]
            # Window Data: только окно (для динамических фичей)
            window_data = df[(df['event_ts'] >= window_start) & (df['event_ts'] < current_split)]
            # Target Data: будущее
            target_data = df[(df['event_ts'] >= current_split) & (df['event_ts'] < target_end)]


            
            if len(target_data) == 0 and not is_validation:
                current_split -= timedelta(days=self.config.step_days)
                continue


            

            # 1. Init Generators
            cand_gen = CandidateGenerator(self.config).fit(df, window_data)
            const_gen = ConstantFeatureGenerator(self.config).fit(global_hist, editions_meta, book_genres_map)
            dyn_gen = DynamicFeatureGenerator(self.config)
            rank_gen = RankFeatureGenerator(self.config)
            current_pop_map = window_data['edition_id'].value_counts().to_dict()
            
            # Запускаем fit ранкера с НОВЫМИ аргументами
            rank_gen.fit(
                reference_path=self.config.ref_rank_file_path,
                history_df=global_hist,        # <--- Передаем полную историю для честной статистики
                editions_meta=editions_meta,   # Метаданные (авторы, книги)
                const_gen_wrapper=const_gen,   # <--- Передаем генератор с эмбеддингами
                als_model_wrapper=cand_gen     # ALS модель (если нужна)
            )
            
            # 2. Identify Target Users (Active in window)
            

            u_window = set(window_data['user_id'].unique())
            u_target = set(target_data['user_id'].unique())
            
            strat = self.config.training_user_strategy
            
            if strat == 'intersection':
                # Только те, кто был активен в окне И продолжил активность в таргете
                # Это очистит обучающую выборку от случайных "залетных" и "уснувших"
                target_users = list(u_window.intersection(u_target))
                
            elif strat == 'window_only':
                # Честная симуляция: предсказываем для всех, кого видели, 
                # даже если они потом ничего не лайкнули (Target=0)
                target_users = list(u_window)
                
            else: # 'union' (старое поведение)
                target_users = list(u_window.union(u_target))
            
            # Если после пересечения никого не осталось (бывает на краях), берем хотя бы окно
            if not target_users and strat == 'intersection':
                print("Warning: Intersection is empty! Fallback to window_only.")
                target_users = list(u_window)

            # 3. Generate Candidates (base pool)
            cands = cand_gen.generate(target_users)

            # 3.1 Inject ALL future positives into candidates (GT positives)
            # Берём уникальные пары (user_id, edition_id) из target_data (wishlist/read),
            # чтобы recall позитивов в кандидатах был = 1.0.
            # [FIX START] Фильтруем GT, оставляя только целевых юзеров!
            pos_pairs = target_data[['user_id', 'edition_id']].drop_duplicates()
            pos_pairs = pos_pairs[pos_pairs['user_id'].isin(target_users)] # <--- ДОБАВИТЬ ЭТУ СТРОКУ
            # [FIX END]
            # Если в candidate df есть колонка strategy — пометим источник,
            # чтобы потом можно было анализировать.
            if 'strategy' in cands.columns:
               pos_pairs = pos_pairs.copy()
               pos_pairs['strategy'] = 'gt_pos'

            # Мержим и дедупляем
            cands = pd.concat([cands, pos_pairs], ignore_index=True)
            cands = cands.drop_duplicates(['user_id', 'edition_id']).reset_index(drop=True)
            # 4. Calculate Features
            cands = const_gen.transform(cands)
            cands = dyn_gen.transform(window_data, cands, editions_meta, book_genres_map, users_meta)
            cands = rank_gen.transform(cands)
            # 5. Calculate Targets (0/1/3)
            final_df = self._add_targets(cands, target_data)

            final_df['fold_id'] = fold_idx
            
            # 6. Store
            if is_validation:
                val_dfs.append(final_df)
                is_validation = False
                print(" -> Validation Fold stored.")
            else:
                # Проверка на перекрытие с валидацией (Dirty Check)
                if target_end > val_split:
                    dyrting_dfs.append(final_df)
                    print(" -> Fold overlaps with validation, (Dirty).")
                else:
                    train_dfs.append(final_df)
                    print(" -> Train Fold stored.")
            
            # Shift back
            current_split -= timedelta(days=self.config.step_days)
            fold_idx += 1
        train_final = pd.concat(train_dfs, ignore_index=True) if train_dfs else pd.DataFrame()
        dirty_final = pd.concat(dyrting_dfs, ignore_index=True) if dyrting_dfs else pd.DataFrame()
        val_final = pd.concat(val_dfs, ignore_index=True) if val_dfs else pd.DataFrame()
            
        print(f"[Orchestrator] Done. Train: {len(train_final)}, Dirty: {len(dirty_final)}, Val: {len(val_final)}")
        
        return train_final, dirty_final, val_final

    def _add_targets(self, candidates, target_interactions):
        # Агрегация реальных событий: берем Max (Read(2) > Wishlist(1))
        # Важно: это выбирает "сильнейшее" событие по ID, а не по весу
        real = target_interactions.groupby(['user_id', 'edition_id'])['event_type'].max().reset_index()
        
        merged = candidates.merge(real, on=['user_id', 'edition_id'], how='left')
        
        # [UPDATED] Лаконичный маппинг через конфиг
        # Берём словарь из конфига
        mapping = self.config.target_scores
        # Дефолтное значение для нулей (negatives)
        fill_val = mapping.get(0, 0.0)
        
        # Map работает быстрее apply, fillna закрывает пропуски
        merged['target_score'] = merged['event_type'].map(mapping).fillna(fill_val)
        
        merged = merged.drop(columns=['event_type'])
        return merged

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import timedelta

# =========================================================================
# 8. DATA HANDLER & DATE ANALYZER (Helpers)
# =========================================================================

class DateAnalyzer:
    """Анализатор временной шкалы для авто-сплита."""
    def __init__(self, interactions_df):
        self.df = interactions_df.copy()
        if not np.issubdtype(self.df['event_ts'].dtype, np.datetime64):
            self.df['event_ts'] = pd.to_datetime(self.df['event_ts'])
        self.min_date = self.df['event_ts'].min()
        self.max_date = self.df['event_ts'].max()

    def suggest_split(self, test_days=30):
        cutoff_date = self.max_date - timedelta(days=test_days)
        cutoff_date = cutoff_date.replace(hour=0, minute=0, second=0, microsecond=0)
        
        train_mask = self.df['event_ts'] < cutoff_date
        val_mask = self.df['event_ts'] >= cutoff_date
        
        print(f"\n[Auto-Split Analysis]")
        print(f"Last Date: {self.max_date} | Window: {test_days} days")
        print(f"Suggested Split: {cutoff_date}")
        print(f"Train: {train_mask.sum()} | Valid: {val_mask.sum()} ({val_mask.sum()/len(self.df):.1%})")
        
        return str(cutoff_date)

class DataHandler:
    """Загрузка данных и Time-Series Split."""
    def __init__(self, config: PipelineConfig):
        self.cfg = config
        self.users = None
        self.items = None
        self.interactions = None
        self.book_to_genres = None
        self.items_with_genres = None # items + genres text

    def load_data(self):
        print(">>> Loading Raw Data...")
        self.users = pd.read_csv(os.path.join(self.cfg.data_path, self.cfg.files['users']))
        self.items = pd.read_csv(os.path.join(self.cfg.data_path, self.cfg.files['items']))
        
        # Authors
        authors = pd.read_csv(os.path.join(self.cfg.data_path, self.cfg.files['authors']))
        self.items = self.items.merge(authors, on='author_id', how='left')
        
        # Genres
        genres = pd.read_csv(os.path.join(self.cfg.data_path, self.cfg.files['genres']))
        book_genres = pd.read_csv(os.path.join(self.cfg.data_path, self.cfg.files['book_genres']))
        book_genres = book_genres.merge(genres, on='genre_id', how='left')
        
        self.book_to_genres = book_genres.groupby('book_id')['genre_id'].apply(list).to_dict()
        
        # Genres Text for TF-IDF
        bg_str = book_genres.groupby('book_id')['genre_name'].apply(lambda x: ' '.join(x)).reset_index()
        self.items = self.items.merge(bg_str, on='book_id', how='left').rename(columns={'genre_name': 'genres_text'})
        self.items['genres_text'] = self.items['genres_text'].fillna('')
        
        # Interactions
        self.interactions = pd.read_csv(os.path.join(self.cfg.data_path, self.cfg.files['interactions']))
        self.interactions['event_ts'] = pd.to_datetime(self.interactions['event_ts'])
        self.interactions = self.interactions.sort_values('event_ts').reset_index(drop=True)
        
        print(f"Data Loaded. Interactions: {len(self.interactions)}")

    def get_train_validation_split(self):
        if not self.cfg.valid_start_date:
            raise ValueError("Valid start date not set!")
            
        split_date = pd.to_datetime(self.cfg.valid_start_date)
        train_mask = self.interactions['event_ts'] < split_date
        valid_mask = self.interactions['event_ts'] >= split_date
        
        return self.interactions[train_mask].copy(), self.interactions[valid_mask].copy()

    def get_full_data(self):
        return self.interactions.copy()

In [ ]:
import numpy as np
import pandas as pd
import math
from tqdm import tqdm
import numpy as np
import pandas as pd
import math
from tqdm import tqdm

class MetricCalculator:
    """
    Реализует официальные метрики задачи строго по формулам.
    Используется для валидации.
    """
    def __init__(self, book_genres_map: dict):
        """
        Args:
            book_genres_map: Словарь {book_id: [genre_id, ...]}
        """
        # FIX: Принудительно конвертируем списки в сеты для быстрой математики множеств
        self.book_genres = {k: set(v) for k, v in book_genres_map.items()}
        
        # Константы из условия
        self.alpha = 0.7  # Вес NDCG в итоговом скоре
        self.beta = 0.5   # Вес Coverage в Diversity

    def _get_dcg(self, relevance_list):
        """Discounted Cumulative Gain"""
        dcg = 0.0
        for i, rel in enumerate(relevance_list):
            if rel > 0:
                gain = (2 ** rel) - 1
                dcg += gain / np.log2(i + 2)
        return dcg

    def _get_ndcg(self, pred_relevance, true_user_relevance, k=20):
        """
        Считает NDCG@K корректно.
        pred_relevance: список релевантности предсказанных книг (в порядке ранга модели)
        true_user_relevance: список релевантности ВСЕХ книг, которые юзер реально лайкнул (Ground Truth)
        """
        # DCG считаем по предсказаниям
        pred_relevance = pred_relevance[:k]
        dcg = self._get_dcg(pred_relevance)
        
        # IDCG считаем по ИДЕАЛЬНОМУ топу из всей истории юзера
        # Сортируем все реальные оценки юзера по убыванию и берем топ-K
        ideal_relevance = sorted(true_user_relevance, reverse=True)[:k]
        idcg = self._get_dcg(ideal_relevance)
        
        if idcg == 0:
            return 0.0
        return dcg / idcg

    
    def _jaccard_distance(self, set_a, set_b):
        """1 - Jaccard Index"""
        if not set_a and not set_b:
            return 0.0 
        intersection = len(set_a.intersection(set_b))
        union = len(set_a.union(set_b))
        if union == 0:
            return 1.0
        return 1.0 - (intersection / union)

    def calculate_diversity_components(self, recommended_books, relevant_mask, k=20):
        """
        Считает Coverage@20 и ILD@20.
        Args:
            recommended_books: список book_id 
            relevant_mask: список булевых значений
        """
        discounts = np.array([1 / np.log2(i + 2) for i in range(k)])
        norm_factor_coverage = 1 / discounts.sum()
        
        # --- 1. Coverage (relevance-weighted) ---
        coverage_score = 0.0
        known_genres = set() 
        
        # Проходим только до k или длины списка
        limit = min(len(recommended_books), k)
        
        for i in range(limit):
            if not relevant_mask[i]:
                continue 
            
            book_id = recommended_books[i]
            # Получаем жанры (уже set)
            genres = self.book_genres.get(book_id, set())
            
            if not genres:
                continue
                
            # |G(r) \ S_{k-1}|
            # Здесь раньше была ошибка, теперь genres и known_genres - это sets
            new_genres_count = len(genres - known_genres)
            total_genres_count = len(genres)
            
            fraction = new_genres_count / total_genres_count if total_genres_count > 0 else 0
            
            coverage_score += discounts[i] * fraction
            known_genres.update(genres)
            
        coverage_at_20 = norm_factor_coverage * coverage_score
        
        # --- 2. Intra-List Diversity (ILD) ---
        relevant_indices = [i for i, is_rel in enumerate(relevant_mask) if is_rel][:k]
        L_u = len(relevant_indices)
        
        ild_at_20 = 0.0
        if L_u >= 2:
            sum_dist = 0.0
            for i in range(L_u):
                for j in range(L_u):
                    if i == j: continue
                    
                    idx_a = relevant_indices[i]
                    idx_b = relevant_indices[j]
                    
                    genres_a = self.book_genres.get(recommended_books[idx_a], set())
                    genres_b = self.book_genres.get(recommended_books[idx_b], set())
                    
                    sum_dist += self._jaccard_distance(genres_a, genres_b)
            
            ild_at_20 = sum_dist / (L_u * (L_u - 1))
            
        return coverage_at_20, ild_at_20

    def evaluate(self, predictions_df, target_df, items_df):
        """
        Полный пайплайн оценки.
        items_df: должен быть с columns=['edition_id', 'book_id']
        """


        valid_users_with_gt = target_df[target_df['target_score'] > 0]['user_id'].unique()
        predictions_df = predictions_df[predictions_df['user_id'].isin(valid_users_with_gt)]
        
        if predictions_df.empty:
            return {'NDCG@20': 0.0, 'Diversity@20': 0.0, 'Final_Score': 0.0}
        # =====================
        # Оптимизация: создаем маппинг edition -> book один раз
        ed_to_book = dict(zip(items_df['edition_id'], items_df['book_id']))
        
        # --- 1. Подготовка Ground Truth для быстрого доступа ---
        # Нам нужно два словаря:
        # a) {uid: {eid: score}} -> чтобы проверять, угадали ли мы конкретную книгу
        # b) {uid: [score1, score2, ...]} -> чтобы посчитать IDCG (идеальный знаменатель)
        
        gt_dict = {}       # a)
        gt_lists = {}      # b)
        
        # Фильтруем таргет
        val_users = predictions_df['user_id'].unique()
        target_df_filtered = target_df[target_df['user_id'].isin(val_users)]
        
        for uid, grp in target_df_filtered.groupby('user_id'):
            scores = grp['target_score'].values
            eids = grp['edition_id'].values
            
            gt_dict[uid] = dict(zip(eids, scores))
            gt_lists[uid] = list(scores) # Сохраняем все скоры для IDCG
        
        ndcg_scores = []
        div_scores = []
        total_scores = []
        
        # Группируем предсказания
        for uid, preds in tqdm(predictions_df.groupby('user_id'), desc="Calculating Metrics"):
            # Сортируем по рангу
            preds = preds.sort_values('rank').head(20)
            rec_editions = preds['edition_id'].tolist()
            rec_books = [ed_to_book.get(ed, -1) for ed in rec_editions]
            
            user_gt_map = gt_dict.get(uid, {})
            user_gt_all_values = gt_lists.get(uid, [])
            
            # Вектор релевантности для предсказанных
            rel_vector_scores = [user_gt_map.get(ed, 0) for ed in rec_editions]
            rel_mask = [r > 0 for r in rel_vector_scores]
            
            # 1. NDCG (Fix: передаем все скоры юзера для IDCG)
            ndcg = self._get_ndcg(rel_vector_scores, user_gt_all_values, k=20)
            
            # 2. Diversity
            cov, ild = self.calculate_diversity_components(rec_books, rel_mask)
            diversity = self.beta * cov + (1 - self.beta) * ild
            
            # 3. Total
            total = self.alpha * ndcg + (1 - self.alpha) * diversity
            
            ndcg_scores.append(ndcg)
            div_scores.append(diversity)
            total_scores.append(total)
            
        metrics = {
            'NDCG@20': np.mean(ndcg_scores) if ndcg_scores else 0,
            'Diversity@20': np.mean(div_scores) if div_scores else 0,
            'Final_Score': np.mean(total_scores) if total_scores else 0
        }
        return metrics

import numpy as np
import pandas as pd
from tqdm import tqdm

class GreedyReranker:
    """
    Жадный алгоритм, оптимизирующий Final Score.
    """
    def __init__(self, book_genres_map: dict, items_df: pd.DataFrame):
        # FIX: Конвертируем списки жанров в сеты сразу при инициализации
        self.book_genres = {k: set(v) for k, v in book_genres_map.items()}
        
        self.ed_to_book = dict(zip(items_df['edition_id'], items_df['book_id']))
        self.alpha = 0.7
        self.beta = 0.5
        
        # Кэш дисконтов
        self.discounts = [1 / np.log2(i + 2) for i in range(20)]
        self.norm_factor = 1.0 / sum(self.discounts)

    def _jaccard_dist(self, s1, s2):
        if not s1 and not s2: return 0.0
        u = len(s1.union(s2))
        if u == 0: return 1.0
        return 1.0 - len(s1.intersection(s2)) / u

    def rerank_single_user(self, user_id, candidates_df, model_scores, k=20):
        """
        user_id: ID
        candidates_df: DataFrame с edition_id
        model_scores: массив скоров (вероятностей или raw) от модели
        """
        # Подготовка кандидатов
        pool = []
        for ed_id, score in zip(candidates_df['edition_id'], model_scores):
            bid = self.ed_to_book.get(ed_id)
            # Теперь здесь гарантированно set
            genres = self.book_genres.get(bid, set())
            pool.append({
                'id': ed_id,
                'genres': genres,
                'score': float(score)
            })
            
        # Сортируем пул по скору
        pool.sort(key=lambda x: x['score'], reverse=True)
        # Ограничим пул для скорости (топ-100 кандидатов модели обычно достаточно)
        pool = pool[:100] 

        selected = []
        selected_genres_history = set() # Для Coverage
        
        # Жадный цикл
        for step in range(k):
            best_candidate = None
            best_marginal_gain = -np.inf
            
            # Дисконт текущей позиции
            pos_discount = self.discounts[step]
            
            # Пробуем каждого кандидата
            for cand in pool:
                if cand in selected:
                    continue
                
                # --- 1. Relevance Gain ---
                rel_gain = cand['score'] * pos_discount
                
                # --- 2. Diversity Gain ---
                # Coverage Component
                # Теперь операция вычитания (-) сработает, так как оба операнда sets
                g_new = len(cand['genres'] - selected_genres_history)
                g_total = len(cand['genres'])
                
                cov_contrib = 0
                if g_total > 0:
                    cov_contrib = (g_new / g_total) * pos_discount * self.norm_factor
                
                # ILD Component
                ild_contrib = 0
                if selected:
                    dists = [self._jaccard_dist(cand['genres'], s['genres']) for s in selected]
                    avg_dist = sum(dists) / len(dists)
                    ild_contrib = avg_dist 
                
                # Взвешиваем Diversity на "вероятность"
                diversity_gain = (self.beta * cov_contrib + (1 - self.beta) * ild_contrib) * cand['score']
                
                # --- Total Marginal Gain ---
                total_gain = self.alpha * rel_gain + (1 - self.alpha) * diversity_gain
                
                if total_gain > best_marginal_gain:
                    best_marginal_gain = total_gain
                    best_candidate = cand
            
            if best_candidate:
                selected.append(best_candidate)
                selected_genres_history.update(best_candidate['genres'])
            else:
                break
                
        # Возвращаем список
        result = []
        for i, item in enumerate(selected):
            result.append({
                'user_id': user_id,
                'edition_id': item['id'],
                'rank': i + 1
            })
            
        return result

    def rerank_batch(self, candidates_with_scores):
        """
        Принимает DataFrame c колонками [user_id, edition_id, pred_score].
        """
        results = []
        print("Reranking users (Greedy Strategy)...")
        
        # Группируем по юзерам
        for uid, group in tqdm(candidates_with_scores.groupby('user_id')):
            recs = self.rerank_single_user(
                uid, 
                group, 
                group['pred_score'].values
            )
            results.extend(recs)
            
        return pd.DataFrame(results)

In [ ]:
import pandas as pd
import numpy as np
import os
from scipy.stats import rankdata
from catboost import CatBoostRanker, Pool
import lightgbm as lgb
import xgboost as xgb

class MetaRanker:
    """
    Оркестратор v10 (Advanced Ensemble + Analytics + Explicit Refit).
    Адаптирован для работы с внешним DataOrchestrator (v9).
    """
    def __init__(self, config, metric_calc, greedy_reranker):
        self.cfg = config
        self.mc = metric_calc
        self.reranker = greedy_reranker
        
        # Зоопарк моделей
        self.models = {}
        self.weights = {}
        self.best_iterations = {} # Храним лучшее число деревьев с валидации
        
        # Данные (кэш для аналитики)
        self.cached_train_data = None 
        self.cached_val_data = None
        self.feature_names = None
        
        # Параметры реранкера
        self.best_reranker_params = {
            'alpha': self.cfg.default_alpha, 
            'beta': self.cfg.default_beta
        }

    def _prepare_group_data(self, df):
        """
        Подготовка данных для LTR.
        FIX v10.2: Строгая сортировка и генерация Integer Group ID.
        Это устраняет баг с расхождением метрик.
        """
        # 1. СОРТИРОВКА (Критически важно для LTR)
        # Мы должны гарантировать, что все кандидаты одного юзера идут подряд.
        # Используем mergesort, так как он устойчив (stable).
        if 'fold_id' not in df.columns:
            df['fold_id'] = 0
            
        # Сортируем сначала по фолду, потом по юзеру. 
        # reset_index(drop=True) обязателен, чтобы индексы шли 0..N
        df = df.sort_values(by=['fold_id', 'user_id'], kind='mergesort').reset_index(drop=True)
        
        # 2. Генерация Integer Group ID (0, 1, 2, ...)
        # ngroup() создает уникальный int для каждой уникальной комбинации ключей,
        # сохраняя порядок появления (так как мы уже сделали sort=False в groupby, но сам df отсортирован).
        # Это создает идеальную последовательность 0,0,0, 1,1,1, 2,2...
        df['group_id_int'] = df.groupby(['fold_id', 'user_id'], sort=False).ngroup().astype(int)
        
        # 3. Список колонок на удаление
        cols_to_drop = [
            'user_id', 'edition_id', 'target_score', 'strategy', 'fold_id', 
            'split_date', 'rank', 'pred_score', 'event_type',
            'book_id', 'author_id', 
            'query_id', # Старый строковый ID удаляем
            'group_id_int', # Сам ID в фичи не идет, мы его вернем отдельно
            
            # Сырые метаданные
            'publication_year', 'age_restriction', 'pages', 
            'title', 'description', 'genres_text', 'sex', 'sex_ohe'
        ]
        
        existing_drop_cols = [c for c in cols_to_drop if c in df.columns]
        
        # 4. Формируем X, y и groups
        X = df.drop(columns=existing_drop_cols)
        y = df['target_score']
        
        # ВАЖНО: CatBoost любит принимать именно массив идентификаторов групп, 
        # соответствующий строкам X.
        group_ids = df['group_id_int'].values 
        
        # Для проверки consistency имен фичей
        current_features = X.columns.tolist()
        if self.feature_names is None:
            self.feature_names = current_features
            print(f"[MetaRanker] Feature set fixed ({len(self.feature_names)} features).")
        else:
            missing = set(self.feature_names) - set(current_features)
            if missing:
                for c in missing: X[c] = 0
            X = X[self.feature_names]
            
        # Возвращаем X, y и массив integer ID групп
        # group_sizes нам больше не нужны для Pool(group_id=...), но если 
        # ты используешь sklearn-API XGBoost/LGBM через group=[sizes], можно посчитать:
        group_sizes = df.groupby('group_id_int', sort=True).size().values
        
        return X, y, group_sizes, group_ids

    def _train_single_model(self, model_key, conf, X_train, y_train, g_sizes_train, g_ids_train, 
                           eval_set=None, use_best_iter=False):
        """
        Единый метод обучения/рефита для всех типов моделей.
        """
        model_type = conf['type']
        params = conf['params'].copy()
        model = None
        
        # --- REFIT LOGIC ---
        # Если это финальное переобучение (Refit), фиксируем итерации и отключаем Early Stopping
        if use_best_iter and model_key in self.best_iterations:
            best_iter = self.best_iterations[model_key]
            print(f"[{model_key}] Refitting with fixed iterations: {best_iter}")
            
            if model_type == 'catboost':
                params['iterations'] = best_iter
                params['early_stopping_rounds'] = None
            elif model_type in ['lgbm', 'xgboost']:
                params['n_estimators'] = best_iter
                # Для XGB/LGBM early_stopping часто передается в fit, там его тоже надо отключить
        
        print(f"Training {model_key} ({model_type})...")
        
        # --- CATBOOST ---
        if model_type == 'catboost':
            train_pool = Pool(data=X_train, label=y_train, group_id=g_ids_train)
            
            eval_pool = None
            if eval_set and not use_best_iter:
                X_val, y_val, _, g_ids_val = eval_set
                eval_pool = Pool(data=X_val, label=y_val, group_id=g_ids_val)
            
            model = CatBoostRanker(**params)
            model.fit(train_pool, eval_set=eval_pool, use_best_model=True if eval_pool else False)
            
            if eval_set and not use_best_iter:
                self.best_iterations[model_key] = model.get_best_iteration()

        # --- LGBM ---
        elif model_type == 'lgbm':
            if 'eval_at' in params: del params['eval_at']
            if 'verbose' in params: del params['verbose']
            
            model = lgb.LGBMRanker(**params)
            
            if eval_set and not use_best_iter:
                X_val, y_val, g_sizes_val, _ = eval_set
                callbacks = [lgb.early_stopping(stopping_rounds=50, verbose=True), lgb.log_evaluation(period=50)]
                
                model.fit(
                    X_train, y_train, group=g_sizes_train,
                    eval_set=[(X_val, y_val)], eval_group=[g_sizes_val],
                    eval_at=[20], eval_metric=['ndcg'], callbacks=callbacks
                )
                self.best_iterations[model_key] = model.best_iteration_
            else:
                model.fit(X_train, y_train, group=g_sizes_train)

        # --- XGBOOST ---
        elif model_type == 'xgboost':
            if eval_set and not use_best_iter:
                params['early_stopping_rounds'] = 50
            
            model = xgb.XGBRanker(**params)
            
            if eval_set and not use_best_iter:
                X_val, y_val, g_sizes_val, _ = eval_set
                model.fit(X_train, y_train, group=g_sizes_train,
                         eval_set=[(X_val, y_val)], eval_group=[g_sizes_val], verbose=False)
                if hasattr(model, 'best_iteration'):
                    self.best_iterations[model_key] = model.best_iteration
            else:
                model.fit(X_train, y_train, group=g_sizes_train, verbose=False)
            
        return model

    def _print_candidate_stats(self, df, stage_name="Data"):
        """Аналитика распределения кандидатов (из v7)"""
        if df is None or df.empty: return
        counts = df.groupby('user_id').size()
        print(f"\n[{stage_name}] Candidate Counts per User:")
        print(f"  Mean: {counts.mean():.1f} | Median: {counts.median():.1f}")
        print(f"  Min:  {counts.min()}   | Max:    {counts.max()}")
        
    def _predict_ensemble(self, X):
        """Предсказание ансамбля с Rank Normalization"""
        if isinstance(X, pd.DataFrame):
            X = X[self.feature_names]
            
        final_scores = np.zeros(len(X))
        total_weight = sum(self.weights.values())
        if total_weight == 0: total_weight = 1.0
        
        for name, model in self.models.items():
            raw_scores = model.predict(X)
            # Rank Normalization (0..1)
            ranks = rankdata(raw_scores, method='average') 
            normalized_scores = ranks / len(ranks)
            final_scores += normalized_scores * self.weights[name]
            
        return final_scores / total_weight

    def fit(self, train_df, val_df=None):
        """
        Stage 1: Обучение с валидацией.
        Принимает готовые DataFrame из DataOrchestrator.
        """
        print("\n=== ORCHESTRATOR: Training Ensemble (Validation Phase) ===")
        
        self.cached_train_data = train_df
        self.cached_val_data = val_df
        self._print_candidate_stats(train_df, "TRAIN")
        
        # 1. Подготовка Train
        X_train, y_train, g_sizes_train, g_ids_train = self._prepare_group_data(train_df)
        print(f"Features ({len(self.feature_names)}): {self.feature_names}")
        
        # 2. Подготовка Val
        eval_set_data = None
        if val_df is not None:
            self._print_candidate_stats(val_df, "VAL")
            X_val, y_val, g_sizes_val, g_ids_val = self._prepare_group_data(val_df)
            eval_set_data = (X_val, y_val, g_sizes_val, g_ids_val)

        
        # 2.5 Auto Feature Selection
        if self.cfg.feature_selection['enabled'] and val_df is not None:
            # Запускаем отбор
            selected_features = self._auto_select_features(
                X_train, y_train, g_ids_train,
                X_val, y_val, g_ids_val
            )
            
            # Обновляем глобальный список фичей
            self.feature_names = selected_features
            
            # ФИЗИЧЕСКИ ОБРЕЗАЕМ ДАТАСЕТЫ, чтобы модели не видели лишнего
            X_train = X_train[self.feature_names]
            X_val = X_val[self.feature_names]

            if eval_set_data is not None:
                # Распаковываем старый, подменяем X_val на новый обрезанный
                _, y_val, g_sizes_val, g_ids_val = eval_set_data
                eval_set_data = (X_val, y_val, g_sizes_val, g_ids_val)
            
            
            print(f"Datasets updated. New shape: {X_train.shape}")
            print(f"Selected Features ({len(self.feature_names)}): {self.feature_names}")
        # --- [END INSERTION] ---
        
        # 3. Обучение
        active_models = 0
        for name, conf in self.cfg.models_config.items():
            if not conf.get('enabled', True): continue
            
            # Передаем весь пакет eval_set, метод сам разберет что кому нужно
            model = self._train_single_model(
                name, conf, X_train, y_train, g_sizes_train, g_ids_train, 
                eval_set=eval_set_data, use_best_iter=False
            )
            
            if model:
                self.models[name] = model
                self.weights[name] = conf.get('weight', 1.0)
                active_models += 1
                
        if active_models == 0:
            raise ValueError("No models enabled!")
            
        print(f"Ensemble training completed. Best Iterations: {self.best_iterations}")

    def deep_evaluation(self, candidates_file_path=None):
        """
        Глубокая аналитика + Сохранение сырых скоров валидации в CSV.
        """
        print("\n=== ORCHESTRATOR: Deep Evaluation & Analytics ===")
        
        if self.cached_val_data is None:
            print("No validation data cached. Skipping.")
            return

        val_df = self.cached_val_data.copy()
        
        if candidates_file_path and os.path.exists(candidates_file_path):
            self.analyze_generation_quality(val_df, candidates_file_path)

        # [NEW] 1. COLLECT & SAVE RAW SCORES TO CSV
        print("\n>>> Collecting raw model scores for validation...")
        raw_scores_df = self._collect_model_scores(val_df)
        
        # Собираем метаданные + скоры
        meta_cols = ['user_id', 'edition_id', 'target_score']
        if 'fold_id' in val_df.columns: meta_cols.append('fold_id')
            
        full_val_log = pd.concat([val_df[meta_cols], raw_scores_df], axis=1)
        
        if self.cfg.save_raw_scores:
            print(f"Saving validation raw scores to: {self.cfg.val_scores_path}")
            os.makedirs(os.path.dirname(self.cfg.val_scores_path), exist_ok=True)
            # float_format='%.6f' значительно уменьшает размер CSV
            full_val_log.to_csv(self.cfg.val_scores_path, index=False, float_format='%.6f')

        # 2. Score Individual Models
        target_scored = val_df[['user_id', 'edition_id', 'target_score']].copy()
        
        # Заглушка для items (нужен book_id для метрик разнообразия)
        if 'book_id' in val_df.columns:
            eval_items = val_df[['edition_id', 'book_id']].drop_duplicates()
        else:
            eval_items = pd.DataFrame({'edition_id': val_df['edition_id'].unique(), 'book_id': val_df['edition_id'].unique()})

        print("\n--- Individual Model Performance ---")
        for name in self.models.keys():
            # Используем уже посчитанные скоры
            temp_df = val_df[['user_id', 'edition_id']].copy()
            temp_df['pred_score'] = raw_scores_df[name]
            
            temp_recs = temp_df.sort_values(['user_id', 'pred_score'], ascending=[True, False])
            temp_recs['rank'] = temp_recs.groupby('user_id').cumcount() + 1
            temp_recs = temp_recs[temp_recs['rank'] <= 20]
            
            m = self.mc.evaluate(temp_recs, target_scored, eval_items)
            print(f"Model: {name} | Score: {m['Final_Score']:.5f} | NDCG: {m['NDCG@20']:.4f}")

        # 3. Ensemble Performance
        # Считаем ансамбль из сырых скоров
        val_df['pred_score'] = self._predict_ensemble_from_scores(raw_scores_df)
        
        base_preds = val_df.sort_values(['user_id', 'pred_score'], ascending=[True, False])
        base_preds['rank'] = base_preds.groupby('user_id').cumcount() + 1
        base_preds = base_preds[base_preds['rank'] <= 20]
        
        metrics_base = self.mc.evaluate(base_preds, target_scored, eval_items)
        print(f"\n>>> Ensemble Score (RankNorm): {metrics_base['Final_Score']:.5f}")

        # 4. Reranker Tuning
        if self.cfg.enable_reranker and self.cfg.auto_tune_reranker:
            print("\n>>> Tuning Reranker...")
            best_score = -1
            best_p = None
            tune_input = val_df[['user_id', 'edition_id', 'pred_score']].copy()
            
            for alpha in self.cfg.reranker_grid['alpha']:
                for beta in self.cfg.reranker_grid['beta']:
                    self.reranker.alpha = alpha
                    self.reranker.beta = beta
                    reranked = self.reranker.rerank_batch(tune_input)
                    m = self.mc.evaluate(reranked, target_scored, eval_items)
                    if m['Final_Score'] > best_score:
                        best_score = m['Final_Score']
                        best_p = {'alpha': alpha, 'beta': beta}
            
            print(f"Best Params: {best_p}, Score: {best_score:.5f}")
            self.best_reranker_params = best_p
            
        return metrics_base

    def analyze_generation_quality(self, val_df, candidates_file_path):
        """Анализ Recall и стратегий (v7 Port)"""
        print("\n>>> Analyzing Candidate Generation Quality...")
        if not os.path.exists(candidates_file_path): return
        
        provided_cands = pd.read_csv(candidates_file_path)
        gen_pairs = set(zip(val_df['user_id'], val_df['edition_id']))
        prov_pairs = set(zip(provided_cands['user_id'], provided_cands['edition_id']))
        
        intersection = gen_pairs.intersection(prov_pairs)
        recall = len(intersection) / len(prov_pairs) if len(prov_pairs) > 0 else 0
        print(f"Recall vs Provided File: {recall:.2%} ({len(intersection)}/{len(prov_pairs)})")
        
        if 'strategy' in val_df.columns:
            hits_df = val_df[val_df.set_index(['user_id', 'edition_id']).index.isin(prov_pairs)]
            print("\nStrategy contribution to Recall (Hits):")
            print(hits_df['strategy'].value_counts().head())

    def predict_submission(self, test_df, full_train_df=None):
        """
        Stage 2 & 3: Refit и Predict + Saving Raw Scores.
        """
        print("\n=== ORCHESTRATOR: Prediction Phase ===")
        
        # --- REFIT LOGIC ---
        if full_train_df is not None and self.cfg.refit_on_full_data:
            print(f">>> REFIT ENABLED: Retraining on FULL data ({len(full_train_df)} rows)...")
            X_full, y_full, g_sizes_full, g_ids_full = self._prepare_group_data(full_train_df)
            
            for name, conf in self.cfg.models_config.items():
                if name not in self.models: continue
                
                # Используем найденные лучшие итерации или дефолт
                best_iter = self.best_iterations.get(name, conf['params'].get('iterations', 1000))
                
                # Переобучаем
                self.models[name] = self._train_single_model(
                    name, conf, X_full, y_full, g_sizes_full, g_ids_full,
                    eval_set=None, use_best_iter=True
                )

        # --- [NEW] PREDICT & CSV LOGGING ---
        print(f"Predicting for {len(test_df)} candidates...")
        
        # 1. Собираем сырые скоры
        raw_scores_df = self._collect_model_scores(test_df)
        
        # 2. Сохраняем в CSV
        if self.cfg.save_raw_scores:
            print(f"Saving TEST raw scores to: {self.cfg.test_scores_path}")
            full_test_log = pd.concat([test_df[['user_id', 'edition_id']], raw_scores_df], axis=1)
            os.makedirs(os.path.dirname(self.cfg.test_scores_path), exist_ok=True)
            full_test_log.to_csv(self.cfg.test_scores_path, index=False, float_format='%.6f')
        
        # 3. Делаем ансамбль для сабмита
        test_df['pred_score'] = self._predict_ensemble_from_scores(raw_scores_df)
        meta_test = test_df[['user_id', 'edition_id', 'pred_score']].copy()
        
        # --- RERANK ---
        if self.cfg.enable_reranker:
            print(f"Applying Reranker: {self.best_reranker_params}")
            self.reranker.alpha = self.best_reranker_params['alpha']
            self.reranker.beta = self.best_reranker_params['beta']
            submission = self.reranker.rerank_batch(meta_test)
        else:
            submission = meta_test.sort_values(['user_id', 'pred_score'], ascending=[True, False])
            submission['rank'] = submission.groupby('user_id').cumcount() + 1
            submission = submission[submission['rank'] <= 20][['user_id', 'edition_id', 'rank']]
            
        return submission
    

    def _auto_select_features(self, X_train, y_train, g_ids_train, X_val, y_val, g_ids_val):
        """
        [NEW] Автоматический отбор фичей через CatBoost Recursive Feature Elimination.
        Возвращает список имен отобранных фичей.
        """
        cfg = self.cfg.feature_selection
        print("\n" + "!"*60)
        print(f"   RUNNING AUTO FEATURE SELECTION ({cfg['algorithm']})")
        print("!"*60)
        
        # 1. Создаем пулы данных
        train_pool = Pool(data=X_train, label=y_train, group_id=g_ids_train)
        eval_pool = Pool(data=X_val, label=y_val, group_id=g_ids_val)
        
        # 2. Инициализируем черновую модель
        model = CatBoostRanker(**cfg['selection_params'])  
        
        # 3. Выбираем алгоритм
        algo = EFeaturesSelectionAlgorithm.RecursiveByShapValues
        if cfg['algorithm'] == 'RecursiveByLossFunctionChange':
            algo = EFeaturesSelectionAlgorithm.RecursiveByLossFunctionChange
            
        # 4. Запускаем процесс отбора
        print(f"Selecting {cfg['num_features_to_select']} best features from {X_train.shape[1]}...")
        summary = model.select_features(
            train_pool,
            eval_set=eval_pool,
            features_for_select=list(range(X_train.shape[1])), # Индексы всех фичей
            num_features_to_select=cfg['num_features_to_select'],
            steps=cfg['steps'],
            algorithm=algo,
            shap_calc_type=EShapCalcType.Regular,
            train_final_model=False, # Нам нужны только имена, переобучим нормально потом
            logging_level='Silent',
            plot=False
        )
        
        # 5. Логируем результаты
        selected_indices = summary['selected_features']
        selected_names = [X_train.columns[i] for i in selected_indices]
        
        dropped_names = list(set(X_train.columns) - set(selected_names))
        dropped_names.sort()
        
        print(f"\n[Feature Selection] DROPPED {len(dropped_names)} features:")
        print("-" * 40)
        for name in dropped_names:
            print(f" [x] {name}")
        print("-" * 40)
        print(f"kept: {len(selected_names)} features.\n")
        
        return selected_names

    def _print_feature_importance(self):
        """Вывод важности фичей (v7 Port)"""
        print("\n=== FEATURE IMPORTANCE ===")
        for name, model in self.models.items():
            print(f"\nModel: {name}")
            fi = None
            try:
                if 'catboost' in name:
                    fi = model.get_feature_importance(type='PredictionValuesChange')
                elif hasattr(model, 'feature_importances_'):
                    fi = model.feature_importances_
                
                if fi is not None and len(fi) == len(self.feature_names):
                    fi = fi / np.sum(fi) * 100
                    fi_df = pd.DataFrame({'feature': self.feature_names, 'importance': fi})
                    print(fi_df.sort_values('importance', ascending=False).head(len(fi_df['importance'])).to_string(index=False))
            except Exception as e:
                print(f"  Error: {e}")



    
    def _collect_model_scores(self, X):
        """
        [NEW] Прогоняет данные через все модели и возвращает DataFrame с колонками-скорами.
        """
        if isinstance(X, pd.DataFrame):
            # Гарантируем правильный порядок и набор фичей
            X = X[self.feature_names]
            
        scores_df = pd.DataFrame(index=X.index)
        
        for name, model in self.models.items():
            # Получаем сырой скор
            raw_scores = model.predict(X)
            # float32 экономит память
            scores_df[name] = raw_scores.astype(np.float32)
            
        return scores_df

    def _predict_ensemble_from_scores(self, scores_df):
        """
        [NEW] Считает ансамбль (RankNorm) на основе уже готовых сырых скоров.
        """
        final_scores = np.zeros(len(scores_df))
        total_weight = sum(self.weights.values())
        if total_weight == 0: total_weight = 1.0
        
        for name in self.models.keys():
            if name not in scores_df.columns: continue
            
            raw_scores = scores_df[name].values
            weight = self.weights[name]
            
            # Rank Normalization
            ranks = rankdata(raw_scores, method='average')
            normalized_scores = ranks / len(ranks)
            
            final_scores += normalized_scores * weight
            
        return final_scores / total_weight

    # Обновите существующий метод _predict_ensemble, чтобы он использовал новую логику
    def _predict_ensemble(self, X):
        scores = self._collect_model_scores(X)
        return self._predict_ensemble_from_scores(scores)

In [ ]:
def run_pipeline():
    print("="*60)
    print("   STARTING HYBRID PIPELINE v18 (Orchestrator Refit)")
    print("="*60)
    
    # 1. SETUP & LOAD DATA
    cfg = PipelineConfig()
    dh = DataHandler(cfg)
    dh.load_data()
    
    print(f"\n[Data Info] Interactions: {len(dh.interactions)}, Users: {len(dh.users)}, Items: {len(dh.items)}")

    # 2. PHASE 1: ORCHESTRATION & VALIDATION
    # Первый прогон: обычная валидация
    orchestrator = DataOrchestrator(cfg)
    
    print("\n>>> Running Data Orchestrator (Validation Mode)...")
    train_df, dirty_df, val_df = orchestrator.run(
        full_interactions=dh.interactions,
        editions_meta=dh.items, 
        book_genres_map=dh.book_to_genres,
        users_meta=dh.users
    )
    
    if train_df.empty or val_df.empty:
        print("CRITICAL ERROR: Train or Val dataset is empty.")
        return

    # 3. PHASE 1.1: MODEL TRAINING
    print("\n" + "="*40)
    print("   PHASE 1: VALIDATION & TUNING")
    print("="*40)
    
    mc = MetricCalculator(dh.book_to_genres)
    gr = GreedyReranker(dh.book_to_genres, dh.items)
    ranker = MetaRanker(cfg, mc, gr)
    
    # Обучаем, отбираем фичи, валидируем
    ranker.fit(train_df, val_df)
    ranker.deep_evaluation()

    # 4. PHASE 2: SMART REFIT (via Orchestrator)
    print("\n" + "="*40)
    print("   PHASE 2: FULL DATA REFIT PREPARATION")
    print("="*40)
    
    full_train_df = None
    
    if cfg.refit_on_full_data:
        # А. Модифицируем конфиг
        # Увеличиваем окно, чтобы захватить "хвост" истории
        cfg.window_days += cfg.target_days
        
        # Сдвигаем точку "валидации" в самый конец доступных данных.
        # Теперь DataOrchestrator создаст один фолд, где таргетом будут последние 30 дней.
        last_date = dh.interactions['event_ts'].max()
        refit_date = last_date - timedelta(days=cfg.target_days)
        cfg.force_validation_date = str(refit_date)
        
        print(f">>> Refit Config: Window={cfg.window_days} days | Split Date={cfg.force_validation_date}")
        
        # Б. Запускаем новый Оркестратор
        refit_orch = DataOrchestrator(cfg)
        
        # В. Нам нужен только 3-й аргумент (val_df), так как для оркестратора 
        # этот единственный фолд будет считаться "валидационным"
        _, _, full_train_df = refit_orch.run(
            full_interactions=dh.interactions,
            editions_meta=dh.items, 
            book_genres_map=dh.book_to_genres,
            users_meta=dh.users
        )
        
        # Г. Добавляем fold_id для совместимости
        full_train_df['fold_id'] = 999
        print(f"Refit Dataset Generated via Orchestrator: {len(full_train_df)} rows.")

    print("\n" + "="*40)
    print("   PHASE 3: GENERATING TEST FEATURES")
    print("="*40)
    
    # А. Готовим данные для генераторов
    last_date = dh.interactions['event_ts'].max()
    window_start = last_date - timedelta(days=cfg.window_days)
    
    print(f"Inference Window ({cfg.window_days} days): {window_start} -> {last_date}")
    
    current_window_data = dh.interactions[dh.interactions['event_ts'] >= window_start].copy()
    
    # Б. Обучаем генераторы на ПОЛНОЙ истории
    print("Fitting Generators on Full History...")
    
    const_gen_test = ConstantFeatureGenerator(cfg).fit(dh.interactions, dh.items, dh.book_to_genres)
    cand_gen_test = CandidateGenerator(cfg).fit(dh.interactions, current_window_data)
    dyn_gen_test = DynamicFeatureGenerator(cfg)

    # В. Генерируем кандидатов для теста
    candidates_path = cfg.hack_candidates_path
    if os.path.exists(candidates_path):
        print(f"Loading candidates from file: {candidates_path}")
        test_cands = pd.read_csv(candidates_path)
    else:
        print("Generating fallback candidates...")
        target_users = current_window_data['user_id'].unique().tolist()
        test_cands = cand_gen_test.generate(target_users)
    
    # Г. Считаем фичи
    print(f"Calculating features for {len(test_cands)} pairs...")
    test_df = const_gen_test.transform(test_cands)
    test_df = dyn_gen_test.transform(
        current_window_data, 
        test_df, 
        dh.items, 
        dh.book_to_genres, 
        dh.users 
    )

    print("[Test] Adding Rank Features (Virtual Ranking) to Test Candidates...")
    
    # 1. Инициализируем RankGen
    rank_gen_test = RankFeatureGenerator(cfg)
    
    # 2. [FIX] ОБЯЗАТЕЛЬНО вызываем FIT, чтобы загрузить Linear Ruler (candidates.csv)
    # Нам нужна популярность текущего окна инференса
    current_pop_map = current_window_data['edition_id'].value_counts().to_dict()
    
    rank_gen_test.fit(
        reference_path=cfg.ref_rank_file_path,
        history_df=dh.interactions,         # <--- Вся история для теста
        editions_meta=dh.items,
        const_gen_wrapper=const_gen_test,   # <--- Тестовый генератор (уже обучен выше)
        als_model_wrapper=cand_gen_test
    )
    
    # 3. Применяем transform
    test_df = rank_gen_test.transform(test_df)

    # 6. PHASE 4: FINAL PREDICTION
    print("\n" + "="*40) 
    print("   PHASE 4: FINAL PREDICTION")
    print("="*40)
    
    submission = ranker.predict_submission(
        test_df=test_df, 
        full_train_df=full_train_df 
    )
    
    out_file = os.path.join(cfg.output_path, 'submission_v18.csv')
    os.makedirs(cfg.output_path, exist_ok=True)
    submission.to_csv(out_file, index=False)
    
    print(f"\nSUCCESS! Submission saved to: {out_file}")
    ranker._print_feature_importance()

if __name__ == "__main__":
    run_pipeline()

In [ ]:

import pandas as pd
import numpy as np
import os
import random
import warnings
from dataclasses import dataclass, field
from typing import Dict, Optional, List, Set, Tuple
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors
import implicit
from tqdm import tqdm
from catboost import CatBoostRanker, Pool, EFeaturesSelectionAlgorithm, EShapCalcType

# Models
from catboost import CatBoostRanker
import lightgbm as lgb
import xgboost as xgb

from dataclasses import dataclass, field
from typing import Dict, Optional, Any, List

@dataclass
class PipelineConfig:
    # =========================================================================
    # 1. PATHS & FILES (Пути к данным)
    # =========================================================================
    data_path: str = './data'
    submit_path: str = './submit'
    output_path: str = './output'


    save_raw_scores: bool = True
    val_scores_path: str = './ansamb/val_raw_scores_1.csv'   # CSV формат
    test_scores_path: str = './ansamb/test_raw_scores_1.csv' # CSV формат

    files: Dict[str, str] = field(default_factory=lambda: {
        'users': 'users.csv',
        'items': 'editions.csv',
        'interactions': 'interactions.csv',
        'authors': 'authors.csv',
        'genres': 'genres.csv',
        'book_genres': 'book_genres.csv',
        'targets': 'targets.csv',
        'candidates': 'candidates.csv'
    })
    
    # Файл для стратегии "from_file" (Hack Strategy)
    #hack_candidates_path: str = "output/smart_hard_negatives_v1.csv"
    hack_candidates_path: str = "submit/candidates.csv"

    ref_rank_file_path: str = "submit/candidates.csv"

    # =========================================================================
    # 2. VALIDATION STRATEGY (Валидация и Окна)
    # =========================================================================
    window_days: int = 125      # Размер окна истории
    target_days: int = 22        # Горизонт предсказания
    step_days: int = 22     # Шаг сдвига


    # 'union' - (было раньше) берем всех из окна И всех из таргета (много мусора)
    # 'intersection' - (то, что ты просил) только те, кто был активен И в прошлом, И в будущем
    # 'window_only' - (строгая симуляция) только те, кого видели в прошлом
    training_user_strategy: str = 'intersection'
    
    # Если None, валидация считается от конца данных. Формат: 'YYYY-MM-DD'
    force_validation_date: Optional[str] = None 
    
    # Делать ли финальный рефит на ВСЕХ данных перед сабмитом?
    refit_on_full_data: bool = True

    # =========================================================================
    # 3. CANDIDATE GENERATION (Генерация кандидатов)
    # =========================================================================
    # Сколько кандидатов брать из каждого источника
    candidates_strategy: Dict[str, int] = field(default_factory=lambda: {
        'als': 0,             # Коллаборативная фильтрация
        'top_pop_window': 0,  # Популярное в окне
        'top_pop_global': 0,  # Популярное за все время
        'wishlist': 0,        # вишлист
        'random': 0,           # Random negative sampling
        'from_file': 200,       # Из внешнего файла
        'als_plus_toppop': 60
    })
    hybrid_strategy_config: Dict[str, Any] = field(default_factory=lambda: {
        'alpha': 0.5,        # Вес персонализации (ALS Z-Score)
        'beta': 0.5,         # Вес популярности (LogPop Z-Score)
        'pool_factor': 30     # Во сколько раз больше кандидатов брать для просеивания (N * 5)
    })

    # Настройки "Умной фильтрации" внешнего файла
    hack_strategy_config: Dict[str, Any] = field(default_factory=lambda: {
        'filter_mode': 'per_user_bottom_k', 
        #'filter_mode': 'none', 
        'quantile': 0.65,
        'fallback_n': 0
    })

    # --- INTERNAL ALS PARAMS (Для генерации кандидатов) ---
    als_factors: int = 64
    als_regularization: float = 0.05
    als_iterations: int = 15
    als_random_state: int = 42
    
    # Веса событий для обучения ALS (Read > Wishlist)
    als_event_weights: Dict[int, float] = field(default_factory=lambda: {
        2: 3.0,  # Read весит в 3 раза больше
        1: 1.0   # Wishlist стандарт
    })

    # =========================================================================
    # 4. FEATURE ENGINEERING (Фичи)
    # =========================================================================
    # --- Text / Content ---
    tfidf_max_features: int = 3000
    svd_components: int = 32
    genre_svd_components: int = 16
    
    # --- Флаги генерации статических фичей ---
    calc_user_stats: bool = True           
    calc_item_stats: bool = True           
    calc_user_author_affinity: bool = True 
    calc_user_genre_affinity: bool = True  
    calc_user_meta: bool = True             # Пол, возраст
    calc_detailed_stats: bool = True        # Кол-во оценок и т.д.
    use_author_reads_metric: bool = True 

    # --- Флаги динамических фичей (DynamicFeatureGenerator) ---
    dyn_features: Dict[str, bool] = field(default_factory=lambda: {
        'split_read_wishlist': True,  # Считать отдельно cnt_read, cnt_wish
        'calc_ratios': True,          # Считать доли
        'calc_deltas': True,          # Дни с последнего события
        'author_split': True,         # Разделять автора на read/wishlist
        'genre_split': True,          # Разделять жанры на read/wishlist
        'rating_details': True,       # Доп. статы по рейтингу
        'calc_time_decay': True,      # Включить генерацию фич с затуханием (tau=7, 30)
        'calc_diffs_and_z': True,  
        'calc_global_recency': True,      # Жива ли книга (глобально)
        'calc_user_pop_zscore': True,     # Z-score популярности под юзера
        'calc_user_year_zscore': True,    # Z-score года издания под юзера
        'calc_rating_confidence': True    
    })
    
    

    # --- Флаги константных фичей (ConstantFeatureGenerator) ---
    const_features: Dict[str, bool] = field(default_factory=lambda: {
        'calc_svd_cosine': True,      # Косинусное расстояние SVD User-Item
        'calc_bayes_mean': True,      # Включить байесовское сглаживание рейтинга
        'calc_genre_cosine': True,
        'calc_loyalty_fractions': True
    })


    rank_features: Dict[str, bool] = field(default_factory=lambda: {
        'enabled': True,                  # Включить генерацию рангов
        'rank_popularity': True,          # Ранг по популярности в батче
        'rank_als_score': True,           # Ранг по скору ALS (если есть)
        'rank_year': True,                # Ранг по новизне
        'rank_tfidf_sim': False,             # Ранг по текстовой похожести
        'rank_qwen_sim': False,        # Rank по косинусу Qwen (User-Item)
        'rank_book_pop': True,        # Rank по популярности Книги (Work)
        'rank_author_pop': True,      # Rank по популярности Автора
        'rank_publisher_pop': True    # Rank по популярности Издателя
    })
    
    embeddings_config: Dict[str, Any] = field(default_factory=lambda: {
        'enabled': True,
        'external_path': "data/edition_qwen3_embedding.feather", # Путь к внешним эмбеддингам
        'use_pca': True,
        'pca_components': 64,
        'calculate_pca_features': True,   
        'calculate_cosines': True,        
        'use_hierarchical': True,         
        'event_weights': {2: 3.0, 1: 1.0} 
    })

    # --- Настройки ALS-фичей (ConstantFeatureGenerator) ---
    als_ensemble_config: Dict[str, Any] = field(default_factory=lambda: {
        # Какие фичи генерировать для КАЖДОГО включенного уровня
        'features_to_calculate': [
                           # Raw Dot Product
            'norms',       # User/Item Euclidean Norms (Confidence)
            'global_z'     # Z-Score относительно среднего скора юзера по всей базе (Global Context)
                  
        ],
        
        # Настройки уровней
        'levels': {
            'edition': {
                'enabled': True,
                'col': 'edition_id',
                'factors': 64,
                'regularization': 0.05,
                'iterations': 15,
                'use_weights': True  # Использовать 3.0/1.0 веса
            },
            'book': {
                'enabled': True,
                'col': 'book_id',     # Агрегация по книге
                'factors': 64,
                'regularization': 0.1,
                'iterations': 15,
                'use_weights': True
            },
            'publisher': {
                'enabled': True,       
                'col': 'publisher_id', # Убедись, что колонка так называется в items.csv
                'factors': 32,         # Издателей обычно не супер много, 32 хватит
                'regularization': 0.1,
                'iterations': 15,
                'use_weights': True
            },
            'author': {
                'enabled': True,       # Включить User-Author ALS
                'col': 'author_id',
                'factors': 32,         # Факторов меньше, так как авторов меньше
                'regularization': 0.1,
                'iterations': 15,
                'use_weights': True
            }
            
        },
        
        # Настройки для "Is Read" (бинарная история)
        'history_check': ['book', 'author', 'genre'], # Для каких уровней считать флаг "читал ранее"
        
        # Общие настройки
        'global_sample_size': 10000,   # Для Global Z-Score
        'random_state': 42
    })


    lightfm_config: Dict[str, Any] = field(default_factory=lambda: {
        'enabled': True,
        'random_state': 42,
        
        # --- Гиперпараметры модели ---
        'dims': 64,             # Размер вектора (можно 128, если памяти хватает)
        'learning_rate': 0.05,
        'loss': 'warp',         # WARP лучше всего для ранжирования
        'epochs': 20,           # Чуть увеличил эпохи, так как фич много
        'num_threads': 4,
        
        # --- Входные фичи (Input Mixture) ---
        'use_content_svd': True,  # Подмешивает SVD (Text для Edition, Genre для Book)
        'use_global_stats': True, # Подмешивает log(pop) и rating (только для Edition)
        
        # --- Уровни агрегации (Levels) ---
        'levels': {
            # 1. Уровень издания (Самый мощный, гибридный)
            'edition': {
                'enabled': True,
                'col': 'edition_id',
                'sample_size_for_z': 10000
            },
            # 2. Уровень книги (Гибридный с жанрами)
            'book': {
                'enabled': True,
                'col': 'book_id',
                'sample_size_for_z': 10000
            },
            # 3. Уровень автора (Pure CF - ловит стиль автора через поведение юзеров)
            'author': {
                'enabled': True,
                'col': 'author_id',
                'sample_size_for_z': 5000  # Авторов меньше, семпл можно меньше
            },
            # 4. Уровень издательства (Pure CF - ловит "формат" издательства)
            'publisher': {
                'enabled': True,
                'col': 'publisher_id',
                'sample_size_for_z': 5000
            }
        },
        
        # --- Выходные фичи (Output Columns) ---
        # Включаем ВСЁ. Для каждого уровня сгенерируется 6 колонок.
        # Итого: 4 уровня * 6 колонок = 24 новые фичи.
        'features_to_calculate': [
            'biases',      # lfm_{level}_u_bias, lfm_{level}_i_bias
            'norms',       # lfm_{level}_u_norm, lfm_{level}_i_norm
            'global_z'     # lfm_{level}_global_z (Z-Score)
        ]
    })


    
    # =========================================================================
    # 5. MODEL CONFIG (CatBoost / LGBM)
    # =========================================================================
    models_config: Dict = field(default_factory=lambda: {
        'catboost_yeti': {
            'type': 'catboost',
            'enabled': True,
            'weight': 0.4,
            'params': {
                'iterations': 10000,
                'learning_rate': 0.01,
                'depth': 12,
                'loss_function': 'YetiRank',
                'eval_metric': 'NDCG:top=20',
                'task_type': 'CPU',
                'random_seed': 42,
                'verbose': 100,
                'early_stopping_rounds': 50
            }
        },

        'lgbm_lambdarank': {
            'type': 'lgbm',
            'enabled': True,
            'weight': 0.3,
            'params': {
                'objective': 'lambdarank',
                'metric': 'ndcg',
                'n_estimators': 10000,
                'learning_rate': 0.01,
                
                'max_depth': 10,
                'min_data_in_leaf': 31,
               
                'random_state': 42,
                'n_jobs': -1,
                'verbose': -1,
                'eval_at': 20
            }
        },
        'xgboost_ranker': {
            'type': 'xgboost',
            'enabled': True,
            'weight': 0.3,
            'params': {
                'objective': 'rank:ndcg',
                'eval_metric': 'ndcg@20',
                'n_estimators': 10000,
                'learning_rate': 0.01,
                'max_depth': 10,
                'random_state': 42,
                'min_child_weight': 1,
                'n_jobs': -1,
                # 'tree_method': 'hist' # Если есть GPU или для ускорения
            }
        }
    })


    feature_selection: Dict[str, Any] = field(default_factory=lambda: {
        'enabled': True,                  # Включить/Выключить
        'num_features_to_select': 220,     # Сколько лучших фичей оставить 
        'steps': 1,                       # Сколько фичей выкидывать за одну итерацию (выше = быстрее)
        'algorithm': 'ByShapValues', # Самый точный метод для ранжирования
        # Параметры для черновой модели (должна быть быстрой)
        'selection_params': {
            'iterations': 1000,            # Меньше итераций для скорости
            'learning_rate': 0.05,         # Чуть выше LR для быстрого схождения
            'depth': 10,
            'loss_function': 'YetiRank',
            'task_type': 'CPU',
            'random_seed': 42,
            
            'allow_writing_files': False
        }
    })

    

    target_scores: Dict[float, float] = field(default_factory=lambda: {
        2: 3.0,  # За чтение даем 3 (можно поменять на 8, 10 и т.д.)
        1: 1,  # За вишлист даем 1
        0: 0.0   # За отсутствие взаимодействия
    })

    # =========================================================================
    # 6. RERANKER (Greedy)
    # =========================================================================
    enable_reranker: bool = False
    auto_tune_reranker: bool = False
    reranker_grid: Dict = field(default_factory=lambda: {'alpha': [1.0], 'beta': [0.5]})
    default_alpha: float = 1.0
    default_beta: float = 0.5

    # =========================================================================
    # 7. SYSTEM
    # =========================================================================
    n_cpu: int = 4
    random_seed: int = 42
    verbose: bool = True

In [ ]:
def run_pipeline():
    print("="*60)
    print("   STARTING HYBRID PIPELINE v18 (Orchestrator Refit)")
    print("="*60)
    
    # 1. SETUP & LOAD DATA
    cfg = PipelineConfig()
    dh = DataHandler(cfg)
    dh.load_data()
    
    print(f"\n[Data Info] Interactions: {len(dh.interactions)}, Users: {len(dh.users)}, Items: {len(dh.items)}")

    # 2. PHASE 1: ORCHESTRATION & VALIDATION
    # Первый прогон: обычная валидация
    orchestrator = DataOrchestrator(cfg)
    
    print("\n>>> Running Data Orchestrator (Validation Mode)...")
    train_df, dirty_df, val_df = orchestrator.run(
        full_interactions=dh.interactions,
        editions_meta=dh.items, 
        book_genres_map=dh.book_to_genres,
        users_meta=dh.users
    )
    
    if train_df.empty or val_df.empty:
        print("CRITICAL ERROR: Train or Val dataset is empty.")
        return

    # 3. PHASE 1.1: MODEL TRAINING
    print("\n" + "="*40)
    print("   PHASE 1: VALIDATION & TUNING")
    print("="*40)
    
    mc = MetricCalculator(dh.book_to_genres)
    gr = GreedyReranker(dh.book_to_genres, dh.items)
    ranker = MetaRanker(cfg, mc, gr)
    
    # Обучаем, отбираем фичи, валидируем
    ranker.fit(train_df, val_df)
    ranker.deep_evaluation()

    # 4. PHASE 2: SMART REFIT (via Orchestrator)
    print("\n" + "="*40)
    print("   PHASE 2: FULL DATA REFIT PREPARATION")
    print("="*40)
    
    full_train_df = None
    
    if cfg.refit_on_full_data:
        # А. Модифицируем конфиг
        # Увеличиваем окно, чтобы захватить "хвост" истории
        cfg.window_days += cfg.target_days
        
        # Сдвигаем точку "валидации" в самый конец доступных данных.
        # Теперь DataOrchestrator создаст один фолд, где таргетом будут последние 30 дней.
        last_date = dh.interactions['event_ts'].max()
        refit_date = last_date - timedelta(days=cfg.target_days)
        cfg.force_validation_date = str(refit_date)
        
        print(f">>> Refit Config: Window={cfg.window_days} days | Split Date={cfg.force_validation_date}")
        
        # Б. Запускаем новый Оркестратор
        refit_orch = DataOrchestrator(cfg)
        
        # В. Нам нужен только 3-й аргумент (val_df), так как для оркестратора 
        # этот единственный фолд будет считаться "валидационным"
        _, _, full_train_df = refit_orch.run(
            full_interactions=dh.interactions,
            editions_meta=dh.items, 
            book_genres_map=dh.book_to_genres,
            users_meta=dh.users
        )
        
        # Г. Добавляем fold_id для совместимости
        full_train_df['fold_id'] = 999
        print(f"Refit Dataset Generated via Orchestrator: {len(full_train_df)} rows.")

    print("\n" + "="*40)
    print("   PHASE 3: GENERATING TEST FEATURES")
    print("="*40)
    
    # А. Готовим данные для генераторов
    last_date = dh.interactions['event_ts'].max()
    window_start = last_date - timedelta(days=cfg.window_days)
    
    print(f"Inference Window ({cfg.window_days} days): {window_start} -> {last_date}")
    
    current_window_data = dh.interactions[dh.interactions['event_ts'] >= window_start].copy()
    
    # Б. Обучаем генераторы на ПОЛНОЙ истории
    print("Fitting Generators on Full History...")
    
    const_gen_test = ConstantFeatureGenerator(cfg).fit(dh.interactions, dh.items, dh.book_to_genres)
    cand_gen_test = CandidateGenerator(cfg).fit(dh.interactions, current_window_data)
    dyn_gen_test = DynamicFeatureGenerator(cfg)

    # В. Генерируем кандидатов для теста
    candidates_path = cfg.hack_candidates_path
    if os.path.exists(candidates_path):
        print(f"Loading candidates from file: {candidates_path}")
        test_cands = pd.read_csv(candidates_path)
    else:
        print("Generating fallback candidates...")
        target_users = current_window_data['user_id'].unique().tolist()
        test_cands = cand_gen_test.generate(target_users)
    
    # Г. Считаем фичи
    print(f"Calculating features for {len(test_cands)} pairs...")
    test_df = const_gen_test.transform(test_cands)
    test_df = dyn_gen_test.transform(
        current_window_data, 
        test_df, 
        dh.items, 
        dh.book_to_genres, 
        dh.users 
    )

    print("[Test] Adding Rank Features (Virtual Ranking) to Test Candidates...")
    
    # 1. Инициализируем RankGen
    rank_gen_test = RankFeatureGenerator(cfg)
    
    # 2. [FIX] ОБЯЗАТЕЛЬНО вызываем FIT, чтобы загрузить Linear Ruler (candidates.csv)
    # Нам нужна популярность текущего окна инференса
    current_pop_map = current_window_data['edition_id'].value_counts().to_dict()
    
    rank_gen_test.fit(
        reference_path=cfg.ref_rank_file_path,
        history_df=dh.interactions,         # <--- Вся история для теста
        editions_meta=dh.items,
        const_gen_wrapper=const_gen_test,   # <--- Тестовый генератор (уже обучен выше)
        als_model_wrapper=cand_gen_test
    )
    
    # 3. Применяем transform
    test_df = rank_gen_test.transform(test_df)

    # 6. PHASE 4: FINAL PREDICTION
    print("\n" + "="*40) 
    print("   PHASE 4: FINAL PREDICTION")
    print("="*40)
    
    submission = ranker.predict_submission(
        test_df=test_df, 
        full_train_df=full_train_df 
    )
    
    out_file = os.path.join(cfg.output_path, 'submission_v18.csv')
    os.makedirs(cfg.output_path, exist_ok=True)
    submission.to_csv(out_file, index=False)
    
    print(f"\nSUCCESS! Submission saved to: {out_file}")
    ranker._print_feature_importance()

if __name__ == "__main__":
    run_pipeline()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import rankdata
import os
from tqdm import tqdm

SOURCES_CONFIG = [
    # --- Файл 1 (например, v18) ---
    {
        'filepath': './ansamb/test_raw_scores.csv', 
        'file_weight': 0.5, 
        'models': {
            'catboost_yeti': 0.4,
            'lgbm_lambdarank': 0.3,
            'xgboost_ranker': 0.3
        }
    },
    {
        'filepath': './ansamb/test_raw_scores_1.csv', 
         'file_weight': 0.5, 
         'models': {
            'catboost_yeti': 0.4,
            'lgbm_lambdarank': 0.3,
            'xgboost_ranker': 0.3
        }
     }
]

# Пути
EDITIONS_DB_PATH = './data/editions.csv'  
OUTPUT_SUBMISSION_PATH = './submit/final_ensemble_v1.csv'

# Настройки
FILTER_SAME_BOOKS = True
TOP_K = 20               

def get_rank_norm(series):
    """Превращает сырые скоры в нормализованный ранг [0..1]."""
    # method='average' дает средний ранг для одинаковых значений
    ranks = rankdata(series, method='average')
    return ranks / len(ranks)

def blend_multi_source(sources, filter_books=False, editions_path=None):
    all_chunks = []
    
    print(f"Starting Multi-Source Blending ({len(sources)} files)...")
    
    for src in sources:
        path = src['filepath']
        f_weight = src['file_weight']
        model_weights = src.get('models', None)
        
        if not os.path.exists(path):
            print(f"[WARNING] File not found: {path}. Skipping.")
            continue
            
        print(f" -> Processing {path} (File Weight: {f_weight})")
        
        # Читаем файл (оптимизация типов для экономии памяти)
        df = pd.read_csv(path)
        
        # Определяем, какие колонки брать
        if model_weights is None:
            # Эвристика: берем все колонки, кроме ID
            exclude = ['user_id', 'edition_id', 'fold_id']
            cols = [c for c in df.columns if c not in exclude and np.issubdtype(df[c].dtype, np.number)]
            model_weights = {c: 1.0 for c in cols}
        
        # Проходимся по каждой модели внутри файла
        for col_name, m_weight in model_weights.items():
            if col_name not in df.columns:
                print(f"    [Warn] Column '{col_name}' not found in file. Skipping.")
                continue
            
            # 1. Считаем Rank Norm для конкретной модели
            # Используем .values для скорости
            norm_score = get_rank_norm(df[col_name].values)
            
            # 2. Применяем веса: Вес_Файла * Вес_Модели
            total_weight = f_weight * m_weight
            
            chunk = pd.DataFrame({
                'user_id': df['user_id'],
                'edition_id': df['edition_id'],
                'score': norm_score * total_weight
            })
            all_chunks.append(chunk)
            print(f"    + Added model '{col_name}' with effective weight {total_weight:.3f}")

    print("Aggregating scores...")
    if not all_chunks:
        raise ValueError("No data loaded!")
        
    big_df = pd.concat(all_chunks, ignore_index=True)
    
    final_df = big_df.groupby(['user_id', 'edition_id'], as_index=False)['score'].sum()
    
    print(f"Unique candidates after blending: {len(final_df)}")

    if filter_books:
        print("Applying Unique Book Filter...")
        if not os.path.exists(editions_path):
            raise FileNotFoundError(f"Editions file needed for filter: {editions_path}")
            
        # Подгружаем book_id
        ed_meta = pd.read_csv(editions_path, usecols=['edition_id', 'book_id'])
        
        # Мержим
        merged = final_df.merge(ed_meta, on='edition_id', how='left')
        
        # Сортируем: User -> Score Desc
        merged.sort_values(['user_id', 'score'], ascending=[True, False], inplace=True)
        
        # Дропаем дубликаты по (user_id, book_id), оставляя первый (с макс скором)
        before_cnt = len(merged)
        merged.drop_duplicates(subset=['user_id', 'book_id'], keep='first', inplace=True)
        after_cnt = len(merged)
        
        print(f" -> Removed {before_cnt - after_cnt} duplicate editions (same book for same user).")
        
        final_df = merged[['user_id', 'edition_id', 'score']]

    print("Generating final ranks...")
    final_df.sort_values(['user_id', 'score'], ascending=[True, False], inplace=True)
    final_df['rank'] = final_df.groupby('user_id').cumcount() + 1
    
    submission = final_df[final_df['rank'] <= TOP_K][['user_id', 'edition_id', 'rank']]
    
    return submission

if __name__ == "__main__":
    sub = blend_multi_source(
        sources=SOURCES_CONFIG,
        filter_books=FILTER_SAME_BOOKS,
        editions_path=EDITIONS_DB_PATH
    )
    
    # Сохранение
    os.makedirs(os.path.dirname(OUTPUT_SUBMISSION_PATH), exist_ok=True)
    sub.to_csv(OUTPUT_SUBMISSION_PATH, index=False)
    print(f"\nSaved blended submission to: {OUTPUT_SUBMISSION_PATH}")
    print(sub.head())

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
import glob

WEIGHTS = {
    'submit/blended_submissionAGI.csv': 0.6,
    'submit/final_ensemble_v1.csv': 0.4,
}

def rrf_blend(submission_paths, weights, k=60):
    scores = defaultdict(lambda: defaultdict(float))
    ranks = {}

    for path, weight in weights.items():
        print(f"Processing {path} with weight {weight}...")
        try:
            df = pd.read_csv(path)
            if not {'user_id', 'edition_id', 'rank'}.issubset(df.columns):
                print(f"Error: {path} is missing required columns.")
                continue
            
            for row in df.itertuples(index=False):
                user_id = row.user_id
                edition_id = row.edition_id
                rank = row.rank
                
                score_contribution = weight * (1.0 / (k + rank))
                scores[user_id][edition_id] += score_contribution
                
        except Exception as e:
            print(f"Error reading {path}: {e}")

    final_rows = []
    
    for user_id, edition_scores in sorted(scores.items()): 
        sorted_editions = sorted(edition_scores.items(), key=lambda x: x[1], reverse=True)
        
        
        top_20 = sorted_editions[:20]
        
        for i, (edition_id, score) in enumerate(top_20):
            final_rows.append({
                'user_id': user_id,
                'edition_id': edition_id,
                'rank': i + 1
            })

    final_df = pd.DataFrame(final_rows)
    return final_df

if __name__ == "__main__":
    ensemble_df = rrf_blend(WEIGHTS.keys(), WEIGHTS)
    output_filename = 'MOGILA.csv'
    ensemble_df.to_csv(output_filename, index=False)
    print(f"Ensemble submission saved to {output_filename}")
